# Speaker-Attributed Bangla Financial Discourse Pipeline — Total Colab

This is the single, end-to-end research notebook for the CSE 4112 project:

**audio → diarization → long-form ASR → word/speaker fusion → target-aware
binary sentiment → speaker/episode profiles → factorial error-propagation
evaluation**.

It consolidates `PIPELINE_RESEARCH.md`, the proposal, the GPT/Qwen research
reports, the `.puku` semantic index, the verified stage scripts, and the gaps
identified in `RESEARCH_VERIFICATION.md`. The pipeline code is embedded inside
this `.ipynb`: no repository clone or companion Python file is required. It is
built for a Colab GPU with Google Drive persistence. Expensive or paid actions
are controlled by explicit switches in the configuration cell.

Scientific invariants:

- Cascaded, inspectable stages; no end-to-end joint model.
- Speaker IDs are opaque integers and episode-local; roles are separate.
- Overlap reference policy is first-speaker-wins (single label).
- Transcript is verbatim: Bangla numerals are words, English code-switches
  remain Latin, disfluencies remain, non-speech events are bracketed.
- Financial target is always retained separately from polarity:
  `MARKET`, `SECTOR`, `COMPANY`, `REGULATOR`, `MACRO_INDICATOR`.
- Polarity is binary: `positive` / `negative`; there is no neutral class.
  Questions, factual/non-evaluative turns, ads, and out-of-scope talk are
  explicitly excluded from sentiment instead of being forced into a class.
- Gold text and ASR text are evaluated through the same classifier.
- Synthetic/bootstrap labels are smoke-test data only and cannot pass the
  submission gate.


## Research-to-code map

| Notebook section | Consolidated decision |
|---|---|
| Corpus | Multi-programme Bangla financial talk-show corpus (5 distinct programmes/channels, not repeat episodes of one show); ~24–58 min per episode, topic/guest diversity and rights attested |
| Audio | mono 16 kHz, −16 LUFS; selective—not universal—Demucs |
| Diarization | `pyannote/speaker-diarization-community-1`, exclusive timeline for fusion, micro-turn removal, gap stitching, under-used-speaker pruning |
| ASR | BengaliAI tugstugi Whisper-medium, converted to CTranslate2, beam 5, word timestamps, `condition_on_previous_text=False`, ≤28 s windows |
| Fusion | word midpoint → exclusive speaker interval; maximum-overlap fallback; no missing speaker silently accepted |
| Sentiment data | human episode labels + optional weak labels from the 7,695-article Bangla financial corpus; label provenance preserved |
| Classifier | target × polarity joint labels on BanglaBERT; grouped split with no episode/channel leakage |
| Metrics | WER, CER, DER at 0.25 s and 0 s collars, proper-noun recovery, attribution accuracy, target/polarity macro-F1 |
| Main experiment | oracle, diarization-only corruption, ASR-only corruption, full cascade; per speaker and per target |
| Robustness | NTP curve, public/private gap, κ, missing-layer gates, hashes, deterministic seeds, resumable artifacts |

GER is retained only as a registered ablation between raw ASR and human
correction. It never becomes gold automatically.


## 0. Runtime and repository

Before running:

1. In Colab choose a GPU runtime.
2. Add `HF_TOKEN` and `GOOGLE_API_KEY` to Colab Secrets.
3. Accept the user conditions for the gated pyannote community model.
4. Put research-authorized audio in `MyDrive/ML_Project/datasets`, or use the
   upload prompt. Results are saved beside it under `MyDrive/ML_Project/data`,
   `reports`, `artifacts`, and `models`.
5. This run is scoped to the 5 audios named in `Config.included_audio_filenames`
   (configuration cell below). Any other audio file placed in the same Drive
   folder is moved untouched to a sibling `datasets_excluded/` folder instead of
   entering the pipeline -- set `included_audio_filenames = ()` to fall back to
   "ingest every recognized audio file found".
6. Every stage writes its durable output straight to Drive as it completes and
   is resumed automatically (`resume_completed_stages=True`): reconnecting after
   a runtime disconnect re-runs only what is still missing, down to the
   individual episode for diarization and ASR.


In [1]:
from pathlib import Path
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = (
    Path("/content/drive/MyDrive/ML_Project")
    if IN_COLAB else Path.cwd() / "standalone_colab_runtime"
).resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "notebook"))
print("PROJECT_ROOT =", PROJECT_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT = /content/drive/MyDrive/ML_Project


In [2]:
# Standalone payload: reconstruct the complete stage library from this notebook.
import base64, zlib
EMBEDDED_STAGE_FILES = {"notebook/00_bootstrap_ep001_annotation.py": "eNqdWFtvG8cVfheg/zBYo8CuIVKyEyMuDQagbSZVK0uGzBRxCWI14g7JjZa7m51ZyYoqwE0fAqPIU1sESFsUQVPKcBLHyEPgAIX9V4j8kn5ndmcvpCy7FQxzZ+ZcvnOZM2fGsqzVlZtRpKRKeMw4k8ehmgjlD9k4CryG8kXCeBhGiis/CtkoSpiINzaurK60/4+/1ZXVld7Elwz/OLt8WYlpHCU8Ob58mU1EEEOZitiUK5H4PPClYHt7Hld8PYcgvHWtvfmRjMK9vdWVURJNGQAz8cCXyg/HzPN54n+SoY1SFaeKyQgkXLF7io+FZNd+fvjnXzL7aOIPJywRH6d+IlZXSnMDvi+CQHgsVcDBw6GQDpvwQwFBU6EmpCURPCCow4k4YtDkh4RidYXvSwEGFo1g3ySd8rBRQGed1PPDaL271dkGXhiumuSQ7qFIjjOxR4kPnSF8kQjy0TAQPAmO4ZHkAAL29orwuPsmaG2VpGJvDxBWV8gTv763s82k74khT24YFCaGCN80lYpF0EnKoEWxfYGwChan+4E/1J7TuHZTxNv4NxFxxBLobLVojeEvBmiYDrkCYA7WNzZKUK4Ok1tmTjM+Zv1GQ8S+jDyR5dAgk/M//0EOIsvqiQFhFiVznhOuO0pVmgjXZT7lmKpksSQqM5uMY55IUUxQZhUDRN+LpsVQHstcfMzVJPD3jey7GJLQX3V3u6ythzYQ+AH0O81EyCg4FLbThCoRqtUVCGqSiKYfSpEoe2ONwW02sTtOoQ4K3NiPReCHgnHJgpixS/D3x7zFum9vXCWNl1gPJgjVGEah55N1yJM4CrAJ1DELsC2wwGxsKaSqzLYAu8bW86932IgHwT4fHkgHQe91dt/v9tyt7oebt3a2W9hMQ9UHsrXKF7al/hoMBjD1JIuhdaez+5tuz2qZCT0ZRxKQDgWm+9Z89u189mI++7v+/xmbz36cz/46P/t8Pvt6Pnuph1j4YX720FpjIH8+P/t0PvsSRMzwfjWf/Wc++9v87LOM5Bm+SAh9QM5XIHxELDT4UssF0/dgIvJxEh2pCYM/UChSIWlObyzF0tgarFWAh2LMK8BrSJ5ouY8zAIsmzeZnfzBosPYDUUlUk0Y0GlX0edFRWGg8zX+te91bvZ3di334PLeYFPwzc4DGRONnpPPsjxoFhmfwaobyO+PZPzFtBgY/aZfP8J1BHFJpwK4KsJMpF6Nw/DqXnIPjDMHTOL7IgpqpX9TIdMieagkmljmCNPRQgWPsGInNu+yiWzt37na271/so6c6CI+11m8yfd9qZTp9qlo11icmB19oppfsfDfGSTTydapggLIcomBL1E6uLnQUeV3b+zxTkKk9B+KSS3KFR5kqmgkiKXUlTlDwlp2z233/g63Oa1PosQ5Stuee5pmhETzXQcp3DACZbTj7VMf7c40xT/TPjIRsB9DeuNANr1T6k3H9j2YrPS5UVzf910Vk4M5/UN4u2X+nc2t3x93cvr156/VeyCIwMxvnhTHxmwIoy9LBuONxniY5w7+qKfSyqAF6+bvzCtZF3nkjMBfl8DkAsl33pKq68NROD0fNxf45J8BMh874gfDg4/sLDXtkrMrAf1EEuF604E2dBTWkp3S+ra54YsTKxsLOOwjX91pMn0dSCHz6ocJpdOUdhzXe1YdVKxNE7aBLRy1Wg7h5u9PruLc3O7ubv+v0NtEorbORdVLKPNWdpZXx+iNqbkoRTd1kSttplQYnnLrU93DSb0fqvQj1q5skUWLXWxvoKKScsqkvJXV7Pz/8C0vSagf1tivpWHavuJU2llqnkZ9IZZVCndI6GEaYm0HEPWmXYNGieq4SD5SNjjRC5zluW6kaNa5bTs4txXiKdkRCArE10UbYlplEavUHTs0RZmnJ/N/yIBWZ3XVLJ+hawqhgtBzTOyYwv523V81d/WNTIA2yWPADkRAyqYudfYIA27Jv5SsIlTVw9HVEUvNtNJzWBbhJFAjqUozA/sagxaxJBGeuFVr6VwZkYiBC20w57F12hYkAxtVZYx7iOEcaWKfGlkvsA5AlUTqeoFWf8GBE7T81zYWDqXlbvlXoBBAhMWpLuJFX3kWohWT2z4/+fa3Ct8YOcTnysjtOzKWkC8o+OmIdJLo/hcf6xmPkLV8CjrhUAKD8qWCeCIS+OgHytJnxDBES0iAoBH257GhyGMVDhJ6Ls9sasAaj8CieqGwCLsSGbF4bnCOxHPRb2A6hfRUd8JQ/sK+v6TCU6w5bX2dvOc6glonleiUXaxoMzn7r+sAEqvRgK+tiqU5QE9vPpSvdShM7LduoF3lD3Ot8uLO9c+d+nl15f+2LKundnS2Uld79RWJynU9lakz+E2E6BQglqjZWjMggQGyOpe+zX2iX5GOncmkq2vx2BZFmuFqhMneANqt39/1M4qBvxFR4qGyQaRlrgSEfVzF4KRWghAofNvB4ISFoopISa4hkyVqGo8njGHz2Sb1qWgUFbffWQqVupCd+a+Mt79RaW2DLY58xZRjKmcEidSkS1OVgSWhZePRpk1lbK0dLLIXlwI7ybHgqe2QJTO69GkPh0iVyeF9XgZwHw0UKiqSLy/4+6KYgofG5NGGUTFFVPhHeq6h0utCi/lhcNlkEAvO5SFKUn8yJVnGoL0WQTkRJJOVzVEF8o/J4cejz6qtKVc5pcdbE/JjOxvKy+oZhtySuzFNO3pPwMcG50tyo6rB86VKtdqlWY532SPXRCAXwWk3g8usNuHp4v6lR5QcOtVB1v5zUsxAHgUUHnIuegYaVM0+f5bSehgchXTad04WOpCjoubZyudbPleZAQ+UIqpCMRagrmudySg+UQjdVQ2h14VTbqXWHeVwX2iPr3nKY6XUtjuI0gGR9mBYvIdgODRU18NNk1oKgXREHHO9vy09aN9hNEY6R4Y2tKIJ02CGHiR/rQzRJA9TyRWF2VqsD7aebPBwHnB1FiQe/oqESDXnk06uKx7pY8uWEyLagDE7Vkp0liXj+0eeXCAFvCE566GtW+zrT/+ZtksAbVmgyuOyHpxxnJrbhYat8jmG/Z9t4/UGa04/ug1Gm8qMlxrR56mp2knFK5fAujRLbE4Uj2tabvwVb5jBscg+hz2XaVvHIhw4LWHkaqLal3/tezYGHhwo1PYWZXr2zvb3T6/S6t501po5j0aa3tVwOJNABHDe1WSRQaqdUlpuQ3JweeH5iZ29vsq23W/Ze7EYHeugsForKfUNLyS3K6SDTXCmMjuweUaWt3SRgjS5YLs3Zhn/NaMzlooyXfS/Vkny5X5aEQUFaaSMXiCtb1pDHCZ1YI6tf2DVgdbRF+ZDtkxLHaWXL03w5Oq3n9shCup0Yw05NpPP83cgyF82b64Z8So+y7TazXJfy2HWtPEnpVRRxUbZOb8f5L+g03tg=", "notebook/01_stage_0_audio_inventory.py": "eNrVW19v20p2fzfg7zDLWzRirkTbSW5617kK6tjKjXZzbcNW9u7W16UpcSTxmiK1/GNHdQ30pQ9FXwu0D31s0z4UbT9Asf0qQYH9Hv2dM0NySNHOZrEuUAeIKM6ZmTNnzt/fjCzL2tw4zbyZFNvif/7q78Re7gexGEZXMsriZCX+WHwnM8/3Mk/se8ssT+TmRv8z/jY3NjeOg6UMg0iKcR6EvliGXtQVxaSdA28ldrrim754IuaJuI6TS/Gl2KHnSbxY5pm0HRrl29gL082NHv1tbuw44nsvvBQXF8RbKrN06+JCeJEv/CCdxFcyERL/rYTHC5oGocQoTxxxnMRjKaQ3mfNLcR1kc4yyyImfCGN0srkU8zjNxNxLRRSjcTpdUq+LC1tksZDvs8SbZJsbQgg/T7wsiLGe1FssMRy+yq6YzL0okiEWkEdZV4yDTDcQg5PYlxMw89QR+2qBgqY8fbPXe/LVc5oVrMWJwJzYBS+aSLElkmA2z1LhZZlMM54SIzxzxN5yGa64Pwu3R8IVqQzlhGhEkocy3WVW6S+RP6JByGWQgolUfCN2tsUiiASmeyl+ys8vSupp6M1A8lSRQBgXF1Y6j5PM0qJ+KZ6bbWEczdAExr5yxGCBRavd2Uq8661EzoI0S1bOJL1CdxY7sS3fQ5aQSZgvwPhkLhce2Px1HiTSF+MVM0N0tLBdDKh5dwO/FHPX2AYIbZZ4iwVkDUG6iVzEmXRnOaTW5bFYN1J34SWXMnMjeZ12RRhPvNBdetkc+zj3sAt4F0xklErXkDgvbXPjJI/ENIkXzFYil7FI4jjb3aU2Ettylc0h+ggTj+P4cmt7x01Judxtl5XRDQrrcpYrsfb3hQild4XNeXf4i8HJ8PVwcPD7jNvrQQumPcW+Hvg6CfBN7O/3Xv2q98zZFmmWj4ntEVaSTpJgmYkgFYEvF0vMEmXsEhKZ5FEURDOBHSXp6WHU8vWuCgyxuXFxUZsW+8w6NF4tvTTVXRZelHshdrJVwloTNjdIv/REnvjZ6dGhSMHXxEtgA6ZetYzi/JjyUKRjmxtLmfS00ogsWBDRYpm+AC+0VMWTH0/yBdYLlbPShReGvRQaIS1BOiHwGEp/cyPOIZ5IaaN2ab1EptJL4ExoW1Loh0UedXODFcR1pzm5TNcVASSaZLAa0DGTKVEVb5PZ0ktSWb6AjZTPWKkkvssX5CDCYFx+p9WWX+K0fEyqLukq1SyR3Cah2o1qAvUK7in1gwkc1jSQoa97kAgwXUF97JFQlf6vlqwVqmGYycQbh5LW9YXo/eH+xBc0IM2bPsDIJ4PjI/fk6Ggk+jxHJ04dGV0FCfRoJrMOm551PDwevB0eDtzjk6OfDfZH3MPqqh7YZkQS17UdaEMcXsmO7WA/oU/6Y8M2mjYO9kZ7p4PRqXswPMGkFQNbwiqCGXSIyNyTve/vIrPoARZg0Rq+HZ6OTn7l7p/+AqS1nkRkuF5Q741Gg9PR3mh4dOieDg8G+3snLZ3usitL7e+BnHp5mBVukj0J2yvchrie47+6A4Kh5an0HcP5KNujwRKJSA/78yVnBxM5j0MfEZyiYGFgsDQ0reAfEqib90Lw94ksIwkNhFBCvqViKiEFZXbIaqlvnpqBFTxcRvE1BdODweu9d29H7tvh/uDwdOCejt69glw6yvlaxmKk36tW0Wkw+ELAQwbTlRhLsA9PkY/BDsvPhuzszY1X797+3FWb4I7enAxO3xy9PcBMX5OLnss8wWYFk12xoO4ZQpxyVfCbK05ZUtF7KdJ8NiO5GmypjXlVpQETeBmkKkGExIEc+SReBpAw+BuDn0UVxIgaEvhueOgevDvRmjHYB09IDx4jyKNt75fNtp8WbVjAyajZ+rRofXt0+G2z8XnRCG18dzA8cge/HJ3i/Y3lLJZPYVeWc+1d8efimcefCCQTfohnM/709PfrhWfd0kgP4nYOKPNl7/gAw/9p6Xo3N/hDDFSgOoFBJL7O26qMZ5c0Wr3UqY/xpsiB3FRCeaZh7GU6byhSIoO4mRvtCqQUoR56LUMyW6tcyRhOJU3GixbvYZJzruxSSryLiJph57dry0qbryll5gHwjgIsvURWzUO4l+NlswM75DT4C+mOV+Cg0VqKirKTaliyv/gSzo+TXHxSQku5N2fNlAFUOTT1TqSX6nUZbClZuOwFYG5Kdmh/jfJFKhLOFcx+DxUzUbWQA5zLEFnQQ2iwL6fC1SuOpxwHO0o3jjmbnszz6LKQ/o74BgXHc5v9V5Zo9Z6jRac0jhqpY6sW9uw0mhMvZdSxkrFlU6kxrQoaDhFj6OQlZWbIFpNO6C3GvgcLQMj1/A5zYKMGsyzb6MczO/mSsqsOD2AX24uMLULbXL73A/KxxA7947VyIegSZ64uGY318soog9ITIRd8hzCkCVUc8q68IKQ86YWI4XuT6wAUU2SYYuxhFagvSW17Y6QBPkdVp6grTpgzSodpCiWdS7mCIpmm3zWtqyyRkNuxDXVbTMcpmVUPyBQMQensrlgDTARpH5ntLIoTrc/y/USidBgy6SBJ4sQYQAv0pi57y+TZ2hXbzna3QWGsgwiazcXKWttorWiwrGZLbeWNrreFpOsSWEyhoXr9zmvScCgv77pt19Y/4A8qu6GkeLdL0ioEN0FARizGZqMdJQYUd4Ed5oD+/1FYTKCsQdKOW+zPOli23SbSIJrGECMyaoSEpLOYInTTO4TwwziSdt0xg5IDWKegJ1L0CGU0y+boAwnYBFrQ51pMQW84nGZfU0IYQHW362GnvWcpvGa3Qj7t3dDaPhnJG13IjiAI23HdyFtwkTglZQsIdcpYLAKzSh0h2tWjqRolEmKQ1HXDdA8GkaEhpc8wm7WSmF6kVUdKmWxtIXvc3qZFle94OYUWPVzS9qZIonXeq9G4rSoVwjPF/rTKgckQaRsegCH39fDt4HDvu4E7Ovr5ADnw8dshlZuJdAjhJI+SWGfuD70f0vMvLTPaQJNk4i40AusSs27BKLyQXLDRtYSdUgIILCvUIAgv1drnXBEQwHKHDMqQM6rjihPCYxLGFRVCl4rH8xyFyWNR8KhxlbKMgbcLV0UeKnvLGLx4BOowxKDQF408kYf0JSZYEFa8xpRIY1XmsSHE12wnkoHexTJTCFSBr8Da6kAV1A75gcKcn60FvPgSySo25CzjfIJRntZNc9JlGGQseZsUOzsvkuJr7FM5zk3m8JuOXY2nGkt/2EzA0Qv1XYdJa6OVA9xYip6KHu5CD7/9h7/57b//hp4+fvjrjx/+6eOH//744Z/xv3Vr35XN66kqG16btJ5e6dnVCDQV3JrPbBDgmGYEnNE3JG8Jv04zpFOKp3/7+OE3Hz/8I///n+Ur8Piv/P9/fPzwXx//5W8/fvh7KuFoQrsQEGevWmFpuUhzMu9SQeWUl5PMUOBmwSyPUdO/8qJZ6PUCYOe8BhTGv84lSvxiNG8KzVK9kRZSVhwB8kugoQFwh/cO5ecIAEUSxppHZkN77TQKKUiQXuvINi0UiCwKNr3wsskcFv3nP/hf/hHWrFrPts/N/NMcy0IV+2McRB1NubN7zgFDzXGn5y+HgNetYG+DoKljoGvHxAsX39QU9vdrcHnNextuSrvZZj5cZfowuH3tinlHhcYXCSKDpjHkQ5b+gvPhai8eIYMnkOaRU5qs7tRX5YEG25T7NqSlqbBD+ol2SD/+REFeBRDHnXW41dNZxgoXUD23KsRrNU7gv+cKp7FazcPUkssbItl+6t8+XKm3z1iTRqUfqNArTtb0cQOnrh0++6h2O4TXP6Nv59WmA83IkzS4ggtXWHke0VZfXFBfOiOAritgi2sadWKHUzbYAgHl1a5jIykzom6OfI+Z0o69Xmmcndc2IEVNIrFj6kyNfBn3T2ZhPO5Yjy125EsnSFX1aislcdJ8Og3el34c3SqkygjRHB3dAmHtVOgD/FWBiCt5aLt5rD8JuNPwSB0i6NYRFCoBC6igDaLU5AXQgtKrxBXQVO1JDVk6L3WUvoHVFhKKh1qSJDmocFcdh0AUkn0nlc68VNScmZdk/R1zN8h7FQZaeUsuqJAroOXT2Y1d85dj8pV3FeBl3FjDd0gU8aVlqkkdwDGwm0YNwnOd1VPsc6M0nhrEfapFGgDDGh8FkmTV6do4KkfOI8IxSI+MXjBzY/JvRBO6/cMxMq04uSmedp3t6S2dHt80571N72FyHSf+FJvqtPmuEV+KJir9f7Lsl1h2Y957lv1SrAHgn+KSz9FNfUZmRarfgrPZNW00fUp9jsqbkGswfUudjpFJl284sBBqpx44xxU3laOhFOWRl2fxo9va6lN57+RWda7N1dkiRyY3lnzZI5Q8jTo50dm6dR+HVs2uqaquObFOvW8VwPutEZ1jud0AOXRa02/JcpqkpqPoJ7j74XeqmwlPmtRlztYnF3hm5HPnDcpm1qY7rOV3zX7ryZvu2ZLqNftWOH+/gLsAqKJ2QxxHmdApzyPt5rqUmvbx0W3XA+NQoG/oRnOYCqfoa0dsAhnn7duUFrQlnrEuFCAYJRXDGk2SGqZRkNaBjmaXxoFDX4c9QFg2Prile4/Z92vfuve7p37zResGlCcQ/VGSNydnK+obtmS02zWTovzA8ZYA4H0cdU7qKLlur9IhrrjNDXb1rY3OfalGt7jbYSSSBH9VKeT3NG7zIogakko5RhEqV6U8r6TbMCWUcQQcAqnLFUosXyHx65dVykNqvg+jZymOqTnLQmqaUSjGlY0smNLNJXJhND0dvetbTOjxIzk1TwAkmVz2QtzWoSqTjoHjpIk+6HmKImZx6QdkbvQlVVsnONd140v+ahcV0AoIqU9og1HDIR3ivMzHHhAaqO+OOOVDnk1Q20ApIX+CoKGgfClmCZuAt0dF7adIib8U1p9ZNQyRb2aRTlFOjqF3zFYzoKDtnjyVyYv7ZyA9q+tlA/g2qeGkQZ84xgWwFuLKbTGxcbOrhVgZChPqa18tRA2ANXFqpy1tLKy7OcXL+vtG99uGT6HrD1ywFGZWtJxXRbipQsr6Mhh0h65qgFPcdOpoTeky0BFl/SdQqCila0leOgmCPlcdNr2EN4Q19a08m/a+NoFINXBZ5dxvy7gm5eKmSbst68bP1XZGC6kuSDl9waGcuqpUsELoikOHYR2b6zn9WhX0Z+uqR8pUIfv0WNvmromu3AupdFtxk25NEWt2pFSu264ma8cUxtnDGhjfbYLveNEIQ7WFUWSpDdMMIxrEM2MHveIoUXQ8N45mi71Up7PXoMXiCX/oo/ZcV6fm0S0rFWWENMYBNpOdfNKZdo3t7lePdrOr0vc5qiMq0eugJSWDleU0ElKzO5BkQ5dsQ+nTfIE9hSzv1fcm+A5Xj8I5WTNe1kqnkeur0vS8uDhAB62f1VnVRyUIHc0+rzsXGufmtQbpf94IZUF1fidMmQGQD+EAUdcXgjQzRhKAaowv7bqlYG1KaVU7v6iREP8mBX231zVc+uX06qt9nxVwVEqctZrQEIka5fzBz7EGEbytWAIazh4I0lvgdlgH10+vtG7DvZ+Lv1Rnj311Mkv6DQZ2S+A1ZaMt7qw6e8mM784ec0sHYZ2Pcyi5t8xfFihYLzB+WVAgPzj54V8WWLY5h+P5yGL04EYhZ9WyNtOheRM1K84fENwyRA+zlS7B9BsFYZlX1q9E03rVEUCLky6yPIIJJV1y90SjRLWa15U/eVnZEXRPhO+y5PhRQ3O8O+5AUhqblBeqq6vFYkmHYvrKc8D3fOkHC3QY6BhDF1bwu0q9qvxNsfrqImi/cW3Wenc6OEEUaHtNp2oUWAqcvVZB8jZZrxMpe5TI0GV0HPIgCiinxNcVEW6xyi2xP8QPHHi5lMreVQg41mcuVO9Sj7PZlqV+Ir1VKzhCCE0CX7bXJqxl6nrqZ7OHtU5kL4p/PwuwTvMlTmJTdfT6yLhV6icxyjz6WQWcYEAIfc3SHvG5cZNbsJgyystM8wexnbJTKUHZZUL3I6bWGf9gYfvc+NmEuCmRhNvSATTJcRuff4EAavPQpuygFtBvP6Ewe9i1gwSF05sgM81bTfv93snh8PDbXfoVUPU7IjqTBbTjWPba0cPOnQvmLuKGohCPYd8aI3ZS26o4M3abmCRpOgayZ+w2RiwpWCvcKG6lqqYVL/ui9QKy3tU1aTRKkmpFb4aHo936irQeAcTjTKKDuW5aJ7u1nTUnd0I/M1GJZf3qOFwxecUeRg8/4ZS7a6PSmQ9drF0zG3XTLqQjYn1lgbV7DQYpEqB++1FPqX2GnRmb1W/unkFmFs2KznxjWnXpeBVZ9b1ui+1VWleYPw8oTRKK1uStmZvfh+N0RcvPCOym8hjqUvN+Bdpz0zLIHSjjrVWyrhLzFZ3CN1N0+05/o4RS/ESB3U4llfvmbHdIigUMo58M9u4tP9qNSoieuAGxAW7cMkLY57e1q91/4uzQKURT04HVa2rzdvVXNM5krhqKWvK2AEHVW75bvfOMXq+NydhknQe+M/01UVcANlOUX892n2+f/yS5bTMmdpPbKmOGCpZX36iYcF1KR13X0vLCr5bomDfrcJZq2xv/CywKZ7E=", "notebook/02_stage_0b_audio_hygiene.py": "eNq1Wt1y2ziyvneV3wHDXITckWjJyWRmVKvUySR24t38uGzPTm25XDQlQTInFKkhqEhal7b2IfYJz5OcrxsACVKKJ7N7Yl+IBIFGo9Hd+LobnucdHlyW8UyK3kj877/+LV4sJ0ku3mxmicyk8PvPxMc3/xDzPMs74u3Pp5cdsYo/yWlezEV5t5yPsjhJg8OD4X/7d3hweHCeLGSaYNrRMkknYpHGWUdU3Pmv4o3od8Q/++KuEON8vliWMghp4NWd5N5CLeQ4mSZSidvb6XS+kDPRjQvRf9br9fA0Fv3bW/RcKl6LIRKXSZ6Bjh2CLokSWV6KJFNlnKZyIvIMy0XrXa7KUPwihVqOVJmUYEHE4nxT3uXZ4UEyX6RyLjNNEiPiUiwVc6PyZTaZJqkE9W/pfZwsNniGJNFPikKqmEYfTfJVNk/WhwdxNkG/xSbNl5MM8jad+VUqxbs1Ssqu/G2ZfIpTzCrKXOglPFYVwYJGHR68uLwQq7z42BFEN/6UJxNl15dkMywC6ysm3UVclBvQzeJiw7I9xZwyHt+JIl9hAFiaxGV8VMSro0LOElWi31h9AncrCEfS92XB64+maVwLk+TrFfJXOS7lxKPuckD0+6G4kPGEhQAhFWMIlHXQj7MN8T6HEBvyO7q9TZNRkasYVO6wnFQq0oPjULzSwiNBkMqK0QZLlUU8oyWO0TeTqULXJzSplg/1NVru3yWzu+5vyzhNIINFnm4WdzGWhGlLyPFTEtuNC1Uyy+I0tEKOqPPtLXHxNBQvtW5CWqWcQRbQn2rXfFa8I/H2r6eXAe9FWSxldyHjj2Ly0+klKHwXive07DT5B3N3e9sFgzSOhDllSe0jDTnny1IlE3l4IASGXXePn3REt//0xo4mnaHhJz/9LC76xz8IbP/HrrrD1qqVlCUMKC/BwrNQXEJwVjOXaSliBR3pP+tC5cT5y3filxd/E9gSWZjJWCsWRT4GK3Jy9Ge5SFQ+kVEyeR7CZdzeguz3hmwsnvS6So5zLH/XnYjz968bpAu5yItSHU2T2RLcHNkhUWOSRTbjSQ4PLpaZmBb53LC/yKG7eTlgdQNBsWBzJaWUozz/eNQ7jhR5mag3ilj3ojvt/8LF5vDAIx95eMAEo2i6LMFDFAnYOpjCDoIMq7uiXqYVBmEff1V5Zp9zZZ/UpnpcxUUG9VRmCpLjOI0VuQ07h5ok47JTfzJdF3F5B0uw3c7xar6UmwVpvPnwYUH8xanDYLacLza0p9mCWh+JS5hWNtY7XvucSs/InRlGyRcqaGdKdiHBi11AqO3EvvoebCQvpBfwnpycf4guPny4EkNm1M9VKLNPSQHXO5OlzxvjnZ+dn7w9e38SnV98+MvJyyse4XX0CAgfE0ZRQGaXp5+kH4RwV3B85ucgcD5hxtdnl1cXf49eXv7tyyZ1R2DSmuUj4ZHsPXqA49O/jvPzMPMBWH55cnl58ip6dXaBCfcPrywESvXm76/PTjDvXy4/vCcOGwTQ12hhpPU/JE3CqNOz1/tmMEbCkxhDYb29enHx+uQquqQB/WcRDsKqjV3REA7iWYhGeovefvjlhHrCc1RtP5+f67b+U2q7evPzu5/evzh7G12evETzkx610v//OArKPxZHXDBvA219tc0OBASoG7Xnj0ijnVa4MzjSdqvpq4oBqWejzbp450sZF9jqZu/qhIIPGohpmsfmQ+1Wo3Q5BRlrOtfc6caQhMeOyGNHk9FnO2XWhWOZ8DKpbp3FSYZRUbxYpAl9ciavHGB7weSoFL9D3Nobkcl2///+xCMiqLHf2QcAlFeX519hiomciggLnmg36+t1kmUGovtclEscpdfZIswmcVHEmw7tx41RG7jhtxgJh2vwATkD6pDjOGEpPjm2bo0GYxHmgKf9DK3vvyoIHVZoAlQKVeLk1zBCvDt/cvTu6Yuj07cvXvKBD/+qdNcgFB9wrmB/YFua2BReUIlRPP5I57QBJcJf3SXASwz8mNWC8M0R3OpkOWIMR7izYC9g6DDKYRBjlmoVbWNWz8ppjo6Kd7hvNbULY72G/XWEIjVR05Cm9aE0LOYAwC9dxRsVHU+Gp3GqJM4THBNy6BnhkZ+uppoyLWxEMhfPRd/hws6DObjLXMaZH68TNew7BAqJYzLTPWJF8/jYVzNTwBvrq8IMkOuxXJTihH/opOGFRWh25m3KwpGHFbt4RMeehJnzsePKpSUbMyIkTXQFpIrh+zyDYAg7Dq9g5UGTxh9a1edWlo5aK2PKcQKgCeBSJnN5UhR54Tc7sJp4L/MlAiMKTSZATxMp7on1rVgl5Z2jGP69Ft82EBQxWL2811Nvg9BrEg+0S2HzbEBafz0Qrj2qIkoydqT8DP/ML2y9db/aYt/sB9R2DuEzmoZlnbYsKZNxEVZ2AHXkmcVwaKfdUbW1MSWCPzCmO6ses/HEqsIMe49Xn2nZBZjdWi6EpS2OjsTMHBSIJ3Q7Jq+bm7rIU7phgZ26IUpXHw3LLVF3wESHpwz2aNd+U3HYeMRCJBFiz+IEMFuLkQ+1YgFZp4D5I6xPOwIOB5WjrBkvfshaXJAu+YCE/joQf6oEoyUROIPWJDewibnUIh5Lv9cRZliXonUmal2NXc6zp7uuAt80o/4aUsBbXMTZTBoe9lDoiPV+OVWqTKdzU4NZUbmnEVwxV+CfW4iI+q3gX3Zq/L4EsqTFfCv6sts/DuzaDdfHAD+QDwkgn/V7PtGzXR1Omojhd3jiWNBhah6v6SceYWAQOOQf4INouAxgQYriFkI1u0ZdG3ELytSWfFYHnAwb6Wys4wQKp4kPJciBkr26MYQS80QpxAQG1emMh7UVBYvPBWLQghWM4pAyfPgIdIjDoeIt2/X+v2MtRm7EbuVj1rBfhNxwMj1ikmSepuMUWQ0yzl7w0PC5pBTBkJkJ39FLfRI012CG6u3lYaELO000T1v9xxZR7XWFPKMyf2jDOwYbGxD6GQjWVoRzaFZXxVNZIVyT7MoR+E0MSdaRCnNZ3fBrTKwBYKeNh3ESnE1Nbo4jazGq5MsKrPNEGgAmyggArcuMoP8MT3wU3t62CA97nBtooCsSDXasZRsknqBSCe6TGLUmnQBL5KnUNMmSEt4J3/foBakLRUV8hsiU0ZKRTJdp1tEAhWVsvH8Svu56xNZseNjg+xqWTV11yyPBW5C5KaJun5NH9MipPTFOkwWnAJDOk9N4XKrf9S2bHd9CZkz9AQA1f5iGmHMXTPxtwJ/f6kJ4V/sg29Msln++pL+VW9u1MYG9TnCzF5TptTK14KuFTr/sZLG+VvikMFNUZcCq6T5r460YixR58IBrBXRapHkJzOgClro1RFjjey9ms0a40B6LxBk9sXdOyy+ALufoXXLKCklMcwZxtpJCrAoXxoJTJUjhyDRfSAe6QE/mSeZrAA5taSQpAgtJXGSefcKY+yYO9urchDdgyYWqlPNOq5dGbcpjCftZ0P4Oo4pgUTjwi+i415uraJVkwHUYcH3jdN7Wjyu2jYr9Xnjs8EohY0K5d42IgLCAXjEiaAURWNP15ye/CeELJVDdXmBxnQwSWBX63gSBKyiWAjnVSC2n02TtezoRFYSrAh4wKuW69KklnCD4Vj6YoEBoglN8eBzs4LwqB0tkTfZu/nGSUBBGL4rjrg60Bdm1KP/ohmF/bJdVBhcoCdCurweZTd1otGqwZRYwpDWRQzJDjLymMzwtQ9R3SIOVj2ZCBUM4rI54QkB0kQz7Peuf43VI/XwKiPSEHcbdq2RS3g174XcdRPtpXgy9R/3p99+Pnnr1QIW81DqNRzL1vSvEfYDoQevzxnx+AY1DtWkiW99RgkqlP/V2/Q9bz32lw1vh61THfcZrHoT96XbPdMkcLjnEUvuhs8RZkUx8vTFxihgOC3sSVGILS4R5ZZTGG2B932knX4VfHV+bfYdsNaTCh6/nj18VCICKr+SDTQ43Mt4CwdJqIChFzx52X85TFWNO6rWSthh47aX5ONYpP++mnSFFf+7j+KUba0Frk81wE2p2nsB2eiTeUR2MC4oVyt2T1SG1X7dyOZaCrZQ5sbj4ZiiqxHKLTLQT1qqiU/d2PEKVyOUQ22mnhmqAkQkyVoPPD7W8vjXwuUZ40YgMQz4I9GpcStkhZ5CpAzP6o1odvTiJGSdl/ufmQLzWuXM9oJKogaaYqsKIEKqTL3bk2XF67wP2DeHSjM6+vQQGFITUkXvJZPkfBJc1AOyFP/74Y1tfCMbxhxq82bm5zFfVOqhYqL80ahxf7PjhVgjy7KmRTL372ja2VGj8D1OnyJfyecb5QDMfpRi0gHEa6FwpKp9R/5n3QOKymdvbk9ebepRDpkgmFzyluDfzbQfinvJzniPIfcCSvpgq0BeLkIdDgLZ6RKKrUGRDhqijGhl+HmtqqXBDxasB4Q3359hKPcWwfuzsWDR5r+GOX3T6OTWhobNXyHanCEQ/kW34lY8Ngs4enzGsnna/2grSUGfBVlwn9GwrqoJ9zqn2XcJViWnYoOhWmYaNhNoRi+/YpdGqPA33OSt3ykZCyVDfSTMFrUlqHzOkkpQTj7u92qGzJm5aWxSbNSveEG763Hbo+wvUgZVU+QF7dgTlDTZR7xq2EuAeh+Iom3QbeYeBaOWy4REcRzwIj6dbz0by+9y6mb4mYpfnpM/4hLUFX59P+RTP13Tm29QIu/BSuDXkao07DuEUDuB9Xp6SZPdl+7EIl9DWJtAY1BW43tDr2ysL5sZCgpgItTDUo1EAY6TnLkk/cmqkwSCCKOQ55YoQ6xB7gBACZQVMNPSW5bT7gxeQS5vuJjho+T4q3+EriIDu7yDfNQ0ckengwBSlffM70GJruIibOiz9L04IXppbT9dL81ZfsqYqevGv9VULvwg46Co46NK83yCTUAc2RJXNM0biP9EFteDgQIfoqHFEuJGVTKLqBCQX5bcq37xwssOBzUvpRBkbN25ZpHz9iQueKdXyNlbz6UTl603m3hJO1/BAVzfpjto8/ojKI+XJzAUCwarCUEDEU8pTxrijhMiCKhBwXsgAskxJwchpjVLplN9RxCyoZFPCAWBiujNG9cwc+i4lFU0VGOVyFOebGCVpNukdbJikb4JLWAjqjRBETA4ltCuvYsIvO+Ndi+MQp/ImEJiJeYBvAvxEnNMFHnv6dNDWYt62Ci78DlpwXPU01xVWeqoLiEGbPH0OzS08eHZKLNe3MUhQ3MGeLfS5vz9Tsp9va2pzCFI7JRwitq6Rryi32XJcFkpjK7DH+H5dOHpOQyDT4tpr3ODzbgjt11f3TJCyKOiEdJIU3jWrWW90I+7pnLPTBNsjfif6wdbuv6rZ8FwiPhafIs6diHu1nPv9ffw9bvD3+IYk99jy9xhz2OfAa0Hvh/wQiePGluPtbUfL5KBRh26WoR+w+J3gLRjsVHW1IKeeQGr4ngY8rgc8vtmK2jady7h8/c7apBfsEMXVOgxZylb5mop1e6LXoN2NhGTTRXhtlcHbW1/vnl6DXISOwQrdUmGubfc5N1SoaYs1tU9xTY6REHduwaNv1OD5D1sdzegOrTs5z5/Rwc858f2kyfvoke0rOd9+p4cKsiju4UDO7W7l/KFa/26hv97tb/bt9kCcIp118grFeq7U16HA3gM1cG3RtUDr+XU/xBWNs3E/ONt6nyWnDdjMuW0GdkhaMXOlpCKQuG8e4Fk8l9sjr5moN/fFyHIi6oDrlDBhL4rIlUWRZ7MmG0WOHVUy9nDBwf8BHEfmTg==", "notebook/03_stage_0c_diversity_audit.py": "eNqlWc1y47gRvqtK79DLOQypkrjSzFZqSllOxWV7vc7u2C7L2a0tR8WBRVDmDkUoAGRbpdFWLjnnklMqh9zyCjnnUeYF8gppAPwBKdFjZ3WwKQL40N3o/rrRchyn25lIMqcwnMGnP/8NjpI7ykUi13CwihLZ7QStn26n27lIljRNMgo3qySNYJmSrA8loHtE1jDqwy+vh7BIMs+HHyhP4jXIWwoRkURQCYLIRMQJFd2Oeq2BBgoI/vOvr/wRCJrSmUxYBjOeSFxPgGQR0EUigcA7wj9E7D4DTpeMS18JNdld4cacLaDE14J6YzUZ8PMRDs1EXPLo5yNc0j+tEk6j/cM53OCJn89MLODgbLW4oRxYDHSZCBZR0SLdGxgMYPQKHpUOLjibc7JYIAxajVNBM7lHo4/wNoCv+sA4jCBBY+JxDYRcRWtYoAwfc7TjQqT7RN7iiXC6YJIO5isqJMQpmVtor9ot223CzRh6YpLNYYFHjH6S0XvREG70ebgLyge50SBacaLd4j7J0GfquqKLou1+ox0V3QXlnqNNzr/zcrhu5zRGx1sXPoUwiYCMSVhQ2deuJXBoKaGnFgv9Zk6WPfQ4CRGjZnJMklQFmnH2OSMpEIlzEUvosMEHyUCseExmtACBGxozTnF/xDA63NB5kom+AkUshgd0h8eoo0oHweUqg9LpVXAAZ0yOS59fruUtwuByesPYhy+Hr0MtQDichVFBAiFRJOAv192Oo6ii29GQYRiv5IrTMIRkocLOEkyoWfnbmbgrHpkonsRaGJQZS/M4FQXMIVuhK/J8G6XLLCVC0HJC+aoPSBlplM9cEnmbJjfFrAv8qm1wfHEeXp6fX0Gg37lM+DS7SzjL/DmVrjaEc3F6cfz96dlxeHF5/vvjwyu9wumbFahqkqKino9xwtI76nr+knA0df6v41lDuOPJ6eTq8qfwcPLD0za1V+CmlchfgqO0ddQDJ/fmPx66kHzto2Ud3FlreHkVvjvCzWpLDR0Kvap5nItIH+XVweXJ8dUEV26MTOj5YUEwzhje9Lv5e/Jgvx+9KgdwwbIkEzUESQwNfZ3J6dnJ99q6J5cH78LDg8lxOLn6w9FPqK4Tk1RQx/NTdk+560EQgCP5ijpAcQDpp9rJEEuoiQX3qklhOCJUHKHEwKGtDtmz8MfTs6PzH8PJ8SEq6mKU9zDI+yrQ1YOnZnU7vyv9qtvR/0z2u6Rilcqx2aeyzRiDTRYvK/1rr21hawOWpNb7gprwXYonfB2njMgpSqy93I1oTFCSEDlBMr4O1Bwvj2Nrf70U/eNJC9FqhgfDmYo6LQyuG5pRJJ3nAorVApVbjyFKZrJ1gRrMrY5DgHpGYeHWeP6Dt2ZTNW2aWx59SlGnHSk+fcBZwvXyKerDSYIe8w2G6xmT36BO0THnjLvVDPWJnY0NtEW+F0KlGVX9cCTN4aggQh0uLDTMirIjD6JSHL2vgsx117mvJiBb0szFU1bVUeCgq9NsxiLcKHBWMh68cTwgAmJbfoqcmmn1XYxv/whNcElJhHERe5bJdBC7nN0XB6RtpU2367VYnc2Tm5TieVxzwCQCXOVytVrZlV87heuFKm05U/giUOzxMzIzjZxp7ra4PEV9CrQd38PxnLtdhCzfI1y5ZWNtESK4Eh3HHe1OzOW7JaIe+tNdsvDq0fVZTF1biBprtKOWsalsqAPTtc0m6MxB6+9sNW2EWU2oSM2sgFGqGlVdD6fwdYCz8E99YDT1igy+G6LX0ypg4GvICf66TuxTy+UUhE+W6KrRTphkFOsfrE1SSrCQ2xRgL22wl9NtWZL24ZbcUdhkW3CaWC6JolYIGACuWWB94+0GllblraWKnYueqkopWCUrfZihfkJfBLDCfLCls7ZABffKpIKh8n6vaWwrKT7L3FjT1q1UASlTW/GmnEhVdjOWxcl8pW4kpjQXM2Se4iwaYm6dfeYtArGhQz3mfpUWNpTtMoY28+31aOVF+aL95i8DvSFzLaB/lcgW0q7EZnCgbySaSjBdVIKb0f2CV2yANJs9UcJNhiFSrtyqe6DtzHinSFNgKymSyNwYXg8Hg/wmI+6pumMvWSNpmS95yrHShrV5FaNB1rdfVx4VNBysNs0+9aD4WpthGTkovlkTSn4Myidr1JJCYKlLI0sQHzsAC0zaWCDYMjVKnqD83q8fRKD+WO/yyibY1E+mynMGT5WeKhe3WMQUq5RkYZk78IRwDVe1iquSQ6mnh3U78pCL3ZSRelYl6yvPsKEujIc7wEkLLj7XcSusKgG1YCIT7sdE0Z6Puc2frVoGb1BY34SLyOWmarF9UZc0mNryMFGVVEvC00NF+Dgv4LGWVpHW62v2vi0ld3TfKpmZqzeZE7x6S3hiu8qHinYddS1///55l+/371VLgNMYbXTrGzCvYJTWvKPqmNG43LmpbsvAoZVPVCYZQ6+nyuOUDkpw3Qky6abX+20ViFCKXrRGeN4t8/P9nmBz58UL+BarXt1VzHTjSzzjyGJnAMdFsVdd2Hq9jXEwv2K1ba/XCnGETpZkeI2xr1c2SPX+MZj9vTHIb4U2nk2WT0Lc1x6rQ1rs+iRE9GmVQKzkYcjRRm0waBtu+9GqflzlMDecEt2+fRxFFe1m/5yJTWNhh37t26AqkMoJfdAzVNGd54o63vUu1tTH6F2oBDKus+KOATfl2i/4dgwbY5lCdsWCFkJdwQGEbsYy6oXPNONR0crMFXh8ubq51BVu8YV3mJ1KBtfnLrSpX+6krZd9GHrq8pw94lrvkha05P8BwyJ9L1gjSX0WrN2qJ5j24U74e0j8sw5q6E7bWF/M2o/80z/+AgdYsZU/ThS/gEQ+TEhMFdmjS6n7iXo0uWzktziU8nPVIVaX+r3b7zXmp7//87///ivABidvH9ctrxGdP2aO/zPDYkJPs3L4ArOh6dxgBynfWPcXgmZ3pwBUYuJo1cgoC9Kyn1k0WRcfooS75osIrvBaru5uiBeyD/qr11x4r8waSvqA0I3qwtvThSn74UimEm1zrRPxcDYts0ewJ3XY1efepFC/A8eO4feghfBzEg/2k3cTqyThoI2Wd4mz4RRGW0vZk4OLybhYVvrV415VmAzwerKZtzHezlYtvu94bQdhetnIrNUhc5riwjs8Z+aWfW9v2/DZoXFStEIYZmShfq9QzZ0wVC4bhk4uJf4iofqJ0jWe7HX+B3n+2Z8=", "notebook/04_stage_1_diarization.py": "eNq1XN1u20iWvjeQd6hmsNNkIjGynaTTSisYd6wk3iSOYbt7gPEGDC1SFtsSyWZRdhSvgbncvd59hAX2FfZ6H2Afop9kv1N/LP7IcWaSNNCWyKpTp06d/3NKjuPc2Tgqw7OYbbI//vafbDcJi+RTWCZZyo4m4XSazSN2n71J8qT/cpYV7CDjZf+gyCYx50l6dmdj9Pf/u7NxZ+MgyeN5ksbsdJlgqXwepj2mMXJ3wxXb6rGHf/ztPx6zWcEm2SJfljELJ0XGOdtinNDIUu75BOx4lnBWLNM0PJ3HDJ8vkyKOWJmxchaze/eKmMdhMZn1T8PJOV5E1m55iWf37rE8Lu5sfPgQZRP+4GDvYPxmb38cHI6PxjuHz1/5i+jDB/a//73tb7IH+PvQfzSkhRn+/Yz5p1kaD5n+l6/CNM3K+AHP4/A8LvrWen1sZbFMk3LV35Tzx4vTOIpAU64hXMZ64kX2cRLP49M+dpDG5fbD/pu3jLn8Msz7SepJAIskDaJlIcAH2XQ6ZH/8+7+xAVDleOvmdHKzbMLOMImXSTmZYcB/ERF7mAuKAj8+CYtIgrvx30+A+8MjmrlMIyC45HHUV9iyHEcQY8iPjN8CVJmV4VwwXzifszidZsUEZ3O6EodmOO8WkHbmZ1mRlLMFWCdJ2dbjwZa/tT34YXCx6Un2sOCxWbwsEtBhwgRlcsPTxDi0NJ78Fk9KMF1aFsnpUrAJ4ZmUdzbUGLn7OD0DC8fEa3lYYAZ2gs/ZsiQ8PnzYeTnePz6SzBOmkZg5jy+I08pZWLJLAKZnu2/YUV4kacke+gOMCAH8NMOJMPdVyMOUxSWI5Nd3Jjj/LwLdiySCbDDaBPjfCAKtIdh002dvsjCSqMd5wjOMn4MIbFpkCyAahWX4oAgvHxTxGR4XK3/CLz58wBJbPnsBosUhuAYCe5YQYAWiR0txw+42a0MCyxktJ47vMwwdawlgirGZm0zFxA8fNHA/XEZJZtHx1Qtw0HmcsrDA3i/CZE579sR7Hl7EcrPYkwB0eHz8lvSB2qolkA9+UrsJkuhZgPF+UZYLsfdtn+3k+TxRsCoWajJOaBgQbGIY9v/+J1jEBRTaiG37P/xIQoNHkEI8gHT+oB7w+Ew8kGJFD/JzPPgRnCDFSIgIdnWJJRQqE/BICk5T21K0unFralvQtEsOXvnno3f7LFoucpZNbZACFFBaxGnJb0Ex/zeepYJaD312GKdLnGXBmVYJe7scogAwakp/nkFI6FF8RuNw1tOkADXDHDOKMJ1IjskKiABzFyG0FVFYyCVsxPecOTZsOnwDDCpcr+OANcENQOuRz/amhkdJxpepYRfmEuBylSeE1STkMYNIhhCLmM/YDMcsGBkCLbCy+c7rsSnEnTOyKUSoEIbm151d2BhO+oDHyyizNT9sDElmXChQkH9oTw7NByZZQTuXtNH7hnQTHFQZF/TwIgnZX58fQsAXwBtssGI8MxgB11KfYq7NKvZ5L/4YF5OE00bvYZmoX2ZQWZHPSCES8oS6AIPRdP7FfMVAl1hoMcie1CzFSI8F+yRSYy3iMiS+AB5QiyyVSg2QzmKwDTTnlCyD1AIhMAR1zRGACkJ3HS5TqX/kJvKMFVlWDo1hzVegfMpozmmWnT8YPAw4OQjBZmCR1c9XcvirF8Hxu9fj/dFsGnz8+PH20xm7S+cHhA2OEuJd9i6nMUA+w/4K0IIzVzsT0g1gsxXYDto/BElwavwyjnNvKAHs7u0c7v1153jv3X7w887z1z+/2x+Pvsw7sEGM3/483t3d2385ulmhtme+3dsPdn85lF/evXgxgv7pHvX8zS9Hx+PD4Gjvr+PR1uDOhkOu4p0NcVBBMF2WyyIOApYs8gwWT2xFoM5plH5anIEiPDYPSEmYLxk3H/mq+nwZFil5QWot4q7JPOQkImqEeQTRS+J5pEbmYTmbJ6d61AG+qjck2GTX5Ys9nA/JAiF6l70F+QTncZg0GgUQQSU+Yooc/Wp8OIZCJrguKJDMsX/PB72z+UXsej62CmV5ZwOb8QkXP0l5XJTuoAfnsnBpuueZbdaWCTmb58RoafZ7OGTjh4Mti4xQpuBOjEk7xtxl/a/3j929C+XdcJLjabicQ5rdLne47gx7XxMfglU51aRZVg0PiS9PF4kIAKQePY3xv4G/PWC740PY/FR6HneZ7ZEYosNTnkFSMVPqMrVRRvspIEvQ3z57pwSeoAgt1iXLUGO74xc7v7w5Drpeg2mc20q7QwuNm47Q8HOuU5TB+w9LHNf4+c7BTv94d3+fIIEyknA/h/AJQ1AoncwWYXEO2hUX8SqOqi2292cUzZoNmve0w5sxFPt6Lk0ZuMUEI+dpdsql+pd61FWYykj0kx18ehYaTU0mfan6e1uH4T1pMf365c5BcHS8d/z8Fb0AK5FgGaQ4lLiJZlSoVM7IF8hI23xNJr9ruZNmCcjazbHMVxZ6hEavxsc7wdvx4cuxdlNBkBodZIhoXMLJPOOV9MBxgMVnws+NNDgQWbu4gr4qtpPDF0sQlUOJwtUDW5awlyk5DDyeINK6iLX/wzW0o/FL7R8LaCpYZTBECD5At6K08THTDl4rL1pOk8GpBi55fg7/3dqJQE2GpRgXT2YbdOZ/Nmbnzob4ww7DyyNJDmXn4VcUJRz5yZBN51konUXyuFrP1PqB8LKGZB8It0soMRJXGVrmeBqHCzKOZbxAdJSUZSzYrwsZK3MDDQ49pnCqfHSxjFo+W+IgArJS1lPt5xl8jNZy4IgzR7t/jhyOGClQ7BBMsiWoQC64fKeiiLXv9f6bz40DhSRIlCD4HkkT7yrlHEzDSZkVqxG9VHkPQpBLnKGGHEmgry4hd2+I+ygyswT2G6yO7bNgDgSCMyAQEAJqfdccxlCE8icVV77vyXf31F9wVRnKWFQxIwhmSX5tGOLT5iDIc20IDrc5BEJaH5Kft4YcvMYQj/WftfBVLAtX8wgZF5z+SwTJjYwjc8059DoSITT/qIxzqNCQ6yglD+Gba2ec/iERciT8TqPNZkWWZvPsjALA+cqvhiLxsYOFz9Lu0LUR4baiWBnBWvCQTBgrPfjhgyG1iKjA9pXVUZrRpYhjDjIIP1PmoEa1XJhY0aBxCaezJyJgk3TyrNURnL8VuYgw+i2cYOsNW9el3TWagnFEiK+hIabeVVrYzKwpYj0VbzHRzHvss4OGGkaMHSudW2BGQlmNMCnKZCFiWGjJ7NLetYGcIxr1WSWc31dJB2NPCQLCcgHeBqIWuFeN7JE2wdhF+DFZLBfCuDCdVPU1d0oKJFMxmCSvIgmQpykn7zUz3tXMZuQUksCJRpGLLz12Hq9G83BxGiGIHiKn6xsrgrjBV8bD8yp44MhxJyvu7WL5RZjLgcK0BGUmND+pyxNoyB4Nfg8Mrq7lKHAinR2xbn0j2B2e+zVDJfaLkTboel7WenPSmk7rgqtda5C1LUswKDf2J9bmfG1PZYq4re8Anwgv1A/Olbz8AAQUbsNAU4WXUitREPGvbB/BBd7Tn88QRKbjbt5gjXx6KeI+gt8gFWYH4qiJHwiSPvZqWDznzVl0LAD53ajaSTszXsS/L0XlY1Snw/1Kt7cn2fiA/d0aTj0D06vP7EDxlpujIfJw1GpyQk+M12zf5Ecxo2+B/wln21if8vVJuowtOiqO8Ukvp5FbsUxjXQIvo3UQ1vOs5Rv8pMZW781hj+h8Kq6+rcIlP1RaK61pVVFHeNQ3srrFsUY0anSTMETuWH486W++b0j2aNQW9w7Ws48S52CBUydGB2L7GW3WqOZgD9ZJdBd4bIQrZiSOaa9dZ51eN8DWLjvGee1HbaaS62uWAlhLl8EwvrXDJRjpOYyiPC7xUQjnCZeHR0enDglEdo3WJ06vtu2xZ6PK83q/odaCMf2luwYHbwmWkUoXK5U7lwaTspyikCNM1WW2nEcKVpJOsqJA0QsJ4AghPLK0Mn0wRz0IIoDwhC2ko7RE5ShLz7SlTymjAiSzNOLSTirjatse4QhK69PkXE2V4UbDOHebEfXSR9Dptk+U1L0Hbec2Fa5NWJuVzThPfDqP87JxPObUcEC1xTuXrg4qPzcHdSCPBOWFlGp7F8LOwU8FIN/4LAMy4bB8pIa0c7QpRHdLoOX5CtqhqLPlCIdqrqflPTV8UErRy8PUlT9kNuZzBS7+iJgCKdYsD38XR1srzwx8/3V/Uy6tVrXPFf+zThVVkjX6qnnotKHhrTwOvWhD0YuH3RxCSpycDTXI88xU4NdtCGqeV8UsvRsWUmCV1wfI3yr4PDCxeb0S+q1iTfBqoFdyGzmEHnL0Fyp5QNnwz8RyVN+5sTZcJTfbJWC/2V6B5GE16IFdHZOpRCwQi3yinPb5nHUr48h5kfYfbw+2BttPlHEw0UX/yZOtRtTJxqRKaXEh2aexLhJFSGihiyFOLySQi1DIYMieZ2AeqjMVYoYoFbEyXOoaI/YA6YyjpORm+4fS/eKmvMVcQT5kiCDqk4nIYGVVcW0BEHOuQ79mzJKRiblIEPMKJeZooI4dJBdhgpgMp0fKblwUWVENFFB4XD6lHdBnu2rmKPuJ/gELnKzR1JjXlGp0Sp4qcKijwTk/S7NCGdsY2WTo47H4Q/EvAns8uxnTqdNYSioSSDgVNYfsChCuHWOyTzV3jZq0qVZxunL7To/dlPpX/KMIUjHubZcxGfY165j39YVUKxB1AVEGjcyue7sFm3l1R7rD69Lu2svyrIWVSAY8+UT0hBH5grXtnH3H2vZre23teDUO/SHkI4W/TVVscGgQLstZIMr2SLiIpgb1BX7SayGFs1hB0slOEllRHgQYVbhHnwceXmbFOWeqc2fb/6jEGk7SWewb/jc7NWWnkWF3n0QiyGE8CmGy6+6KZsneRr1bCviOKnqeVLL7vtdwY5TgHEOihFB8dWTqJL0FVm2yKA2AnC7S7PphFF8kkw5ZrHHL7vjXvefEI84kXzpea29+mbkCrC/BufKP592oVCowVGWEGpHfTkTLAEIWYq5hRT+FKYevPJFthOcJXAulYL7jUDGGN4V9vblrIEl/U3DW1kQlLOrc6LSpBjPt0MGIyNQitS9QQ0Yi+kg0IHR3aHoJ7QgVCiff9X3fI6n4i7Fm1BEzX1Wta9x0dQHKPKGaOrUEQE5WqKCEKFUUT6W/WYrOHUoUMFfFQyKd1tOzBfeo2oUHp5Cyt7UKiC5B+112BSZthua4EppCbwU8YW2mZtTa81t8Y9OhO4a8Wt+M6Bgd7wwrfd+7YULlwWDGldPUoHjYfHR9Ezibwgag3RaqACrzsA7W9Wej4qYEdZAUjTNzijyMV0J5dMEmXHITOS55WbUsTucoFFBc4d/yfNacxBedwm1IftPU9cTtmHXt3dapucv2RS1wih412RpFfVIInkItxqiZhudP2W9AEwEbIiqbbLfSXzYtO1SXtqtWr0IA/xQt2NDNGoZLNloHBEq71jsusrTLLsu6rXa743cSbnZKGhDmeK/kJjSlk6CSsE7CUUY1Xxak4PrUN14KuhSUfZgk80Q2dAv3eOfokAy1zm8QAGx4kXOsIHlTdXpwHcPBId8xzUzYucyG+E06gACwSELrVGarRabK9jlmF4GOIK3hTjVQQ+0AZloe61M71vUkPG99abCKxy39aQEiFQrmhzMwOefQoJUbT40KIFSPBSZFQRG63UtXzXRXVL2VwfLouFjGXj2Epz7bdixO8H2VlRWfrbSsWNDTVpyyzx2odeP1VZdW7AIo3z7kr9pJjVfqhvNLsrRVx/O3zARchFGgl3a/KPi/qRMWiUsSPHSz5tRM3t3/qvsJTJiPkHqB5gLoH2rLcSezGDUiFMM/rhCL74vqnXGyRBQsyvPxR6CoriMI06Pc+ii7THWfBalIroN41TRLbfzUCdzqAa5ud7w8+MU3AXbDN1HtjehyiKhhkLQ8n/5DYW0FS1dkWsGs0LFgWuqH4FOfUgR1JQ2VQWH2yBGR4faWDtgpt0lz/TRKFuwZ27SQkYp7pAbQCbiokfLRZjUXC6Iutfl4MBjU7Bj6UUJxxIi4Nh+z81efpKiC/GAPKlnjFMAGn8SFolq+AI7kTFPxbBI13vJJkq984VTM9ahCLYYeifmqO6Eg1Cwp8AnogmyWwNhrb7UGylVUFYPZgwfsTFCYPng+ij9YxU1zX1HUgiaOQcyyap0DtqCuN7j9qgVpUzyZZTn7s6KRKu7QmACJTBVJA9o9yjFva4RpSv3V5qA6kmo6Fcqoi4fG68+UHxXb8nBbxwy9qZqdBgppbIkS7BaEfgXBI7JgIV2vFYI9QvuqHxZFuHJPrHMUuQm84b8X4q/gLPF9iX5aCfwkwc4Ab6j+YmWzFlKw+LoZ9ze37HodMVgii7goUrgabzXivadPYicKc9FuZloAYCuzyz7CMqrWkZy5uhcDuKIJEatB5UToUVcpRvU6EK9NugVbqGC4kgZoWBloJEm7fWbwj4/04KqTYdRY7j5OfGsAsrgCYL/+umM6lc6sdgfwy+CRR5QIT0WZXW1T0+fXjOJbm1Nn1I6G+Qm6FeA5JVQByki/lqpbThaEYoo5VeIzk9G8YoRnFj730B3xsCpjWie22RPsKeZ6Xj3eo1yeeHGSvBe1EfUF29+UD+RS9Lq+2sB/0ohTDBz0JxVLE9EqS0Zdd6LBeCgrgGfLDNdmLiyiKB5QdVzlaJVLqI0Tcba66mVXbBEhyNs+L5BJiu3ewYR6FSx69NiFrOsuFxRgx4oedXJciC3LBKdoymrUbfVq1f6MbjKLJnbXgSaxgHszTGsHBqiovkjAUlwfQAk2gkdacu1brB8TK1MdbYArlqI0KRQ49NSjAanKzpYDcQTaqXPRBxV7lSps7eNzeBKORnEQa3mdw5rItro8urEyjKZcVeEig3+Up1N1BohrU9voAk2ppIjriqYuKP0q3RZzlwLDyhmim0DKRfoUoxItbsyS/0Rs5Gn3x/iT2t3hGtgCCZfElJNFRYJ8B175WLC50pkz/W3rIw2rJYkH4OygITCqglhraQChiPfNyHrrSM0fORH2D8fD0eZHn2P5+X2rf0S6LyI1/ZOy3w8/2z/yaVLT69pEQWPSnyiZToXFAou6Zg3ECdA9A6/ewkKQ6O7s5qCrcapOGs0yztHBeOc1kt6DgeM1xFTD2/q74G3W4bU6eW4FxPYeOyYIHUKaHNUit+O9R77H1tDupGnDAGfUMasRRUg3+ydUyEfk2IhM45pNdhga05HWxKuaZXGllN+exJKAfEpyndXkvS7k6yW1rpiTQKqpetUquvxm7cy7RSIu632TqFEphkBVjnUFWTYNiJDx8/3q5DHILydO9dRRh6GDGerJy/3dHXQWHxy+ez4+OhrvQkNPnatqzrWP0fUKqJ7uy8DQbRc+X8CRgqp7QfGWDL4abORcaSDXTHgJqhOc8nTVbcMtddtwcCr1QjBbnSVwT+jCoejYcCqwxizoTnxqaG933d+oY0U3vGqE70yWr6/9ClE1Vf/ApBbcZgJdNrGu6xGo2gMagmfvytSKb1RAXSVok0y27u86t86qavJU9eF23rMmr3qjnSkQc2D6IvZoXZc+INlGn7p/CLi4uH2/fo/7vriUXd2xoAvbFpdbNbAWn5t76079GsYXgOiaTtc2bzudxjp6p4G4qy6Wd/VWesxmFUmX5lgb7/p49aZJ+DWmAhh3txfqeylJNGpQEG+ukuHgYXTtNNL31aiRhVGXQ42moVFBeqPezb3d7ElUDUZmrGk4ao3UiV+sLVydehuSPboZA/dk/1oVQVQUtEwbCCXJT8fn2ufes6tdth4esk4qOFrKMUB/rL02N3yoOlXfpWN1juKtfTVFv0PLnHlT3UdpAMAZmkF0I8U8z8+r5+IaiinL2ChqXsLYExBG8wqavsl6udyr2gKb7Kev3FwbBlWWvGXuLIb8HGNZt7VGdh4PmT3Uy6ikgFI3ED0cH7wLDt+9O/ZshjC39ztOo3WFa0TeEMmkNajzLpcYqFnJxtW+2CUGXVXMCpu+lnLXNhShpUfi/1VjC/0nMtJdFopcChhc3bPcmYZtFKEaucHqavWLh4PNVg6sCp5vKHyqsSooNghbmq1KnPdYq62uCor0HXXbyIs9WhcJBAvIS+f+4hyVKld+4aLQ0pMp7yA7l3UX5TVRBkdMzOCFus4lykdIF2VUHh05y3Laf0J+AHKTw7oeUR2bHcGYqHwtC5VaQj6pd1OnbdMXl2qno84+1R48s9Uzso5XNXhDf3t6za6AgPzkdEH6aX/nGRP/gwa4anVxXot3/5I25lost0BLjItfM7iww9Xa5RFxOIm5CUo5Wf3rB/5OcbYkqh3QN1TZYj4pEsE6I0f/1JXxRuwa4/31dw61x4MoNIqCUK3gOv2+ohanwqDsmxg5Tnf9ehbP85FDpdoQneqkmekGueWBX85wOOIHgvQFzKH8lSb1C0DeejRkZ4yFxBf08nRjO5lR7ouPXDGIxi6jcN1gszXxe2GqTafaBCA8RS6H5D6S9aDLWZzWfh4G0X4E3aZ3iL2Rc5H74kxpq1xwhKwG2q1PHRt7L9mBq1akDVW8VD+jJDwr/U2HTAhJTE2GZurHluRdonIvfM6rGAIBlnKlZRJpk9ok8DtadFynB+GmfJUefl1LJ2hsTgpZTxd+g36MaUU9DqPXEoX3tdBKTxk2mxEcqwuh9SNSvCN22tSxkwqdkvQC3IXrvwie7Kheqd3NKnySvQ9mvSsyRIZpr/WaZMqRXFPBqu47G11ZR3Wt1xGEVY3zFl2sTbbbigph6UHRNeFwKz5KOu/BUBNHn11JaL6tDSvjrt+aO9zPnnC871SHsO9mfPsS9zO4vqTMlIE2I7uvdIvR3cvoPKWBUPMN1s3CylcdgcZ6X+f6QQdl7ChmTf8SMbScJy+Qd3Qc1VpoZOfMlT3nupY/+2zsWYf5nTFsJ99XmH///hoOyM7eG2QxXFHX9RwrcDxEjwtu24ofVZIt3uLuUJgmUzS2GF++EMMCMcJiKGqFHTldfYN904Jvmwr8ghJ+I2+0tnv3H+yKFodiO5rAJykDdKPM6iZLuoPNhc2NgNv+tgrsaYMlnC/9RbqnLQjNDjD6SSX+tNVAht9Q8ltzX+jsO5QQVcLbLRLSLJmue2Scydtz2vSr5w0H0nUBiwcBHTp+qAmpUScIyJEJAmdoeYP695b8Cf3IWqC/1pI+Zozs1ZCXoVxHOs62ENBvH1Ep0BUek+dt/D9dQife", "notebook/05_stage_2_asr.py": "eNrtXP+S2zaS/t9VfgeEqSuTCUVrZmyfV4lSO7HHiTeO7fLYt7eZmqIpEZKYoUiFoGasTLS173D3Dvse+yj7JPc1fpCgSGrG8e0fV3W+24wIAg2g0ej+uhug4zh375yW0ZyzQ/bPv/03Oz59w9xZJEpeDK4WiVjxgt1n3/JsHqXJ4EWer9h0wacXqzzJSh+/19kFj727d8af+u/unbt3XicrniYZZ5N1ksZslUaZz8zw3KfRhh3982//9dBnj/Hn4JAtCjbNl6t1yVk0LXIh2AO8eMQEFyLJM+EFRPVJkZTJNErZkzyNJoNZwVE9jlZlVFIl5tIkqS9vdPcOw78BO12lScl4NF0wvkpEHnOG+ebs4WCZZGrWIjCV36wzlmPQspgRMd1/VeNJxTL2p9NXL1m+LjFowUAxTsQFi2bgN+OXvNgoKlXLl1TICi7KqCjZrMiXTEyLqJwuRlS6XoJXGyYuktUqyeZ6ZCxKCx7FG4xK0q+ovROcvX9/GmU/J/Fi+Ifhfb3EgyWPk/VyMMnev2euXm02w0oMynXGPRYJtiqSZVRsKlrPojRlk2h6QbN4/75Msg0a38fPSSQ4fiYz9t3rd2zJlzmmlWC2yXxRyhV5pedPvDL8dcHY9+/jqIzuR6K4/7UuD5P4m+BnkWNkZnmu1R/659S1nBE9DYcHjm+9X+JlSq/2TbrRQgtUWG5WnBrO0jwqDx457DfmYAEfNypDaOZriCdV3KEj+HzJs1Lg1VldvDP+ujatL6rK3nzMJIurp47qJf9QhpCMCUR4iYqiLHqrZXmxxHL+yuP+itM8myUxz6Z8X69XeRHL+Vyb8QZBYAarflIV83tV5BP5e3veJLatH/WbLUkF7SMp4eWCQ7hXOSvyvByN6B1VWm3KBSQ6y0s+yfOL+8OHoSDdEB6GkJhgtWFng4FccLZvtc/bM+v5B3Ixv0ym2NrrODqnZy0dA5IOpkXjowjKHToQfMqOhkNJUouwYFJ6ffrvIUg6pJrv3pEMCcPZulwXPAxZslzl0ARRBjYo/UW1TGkxX0WF4FUBbZzqIRfVT7Gpf0OCuO6Hdt80jQT0l+moKvKhD3ga65qrqFykycTUeo1HGsf3J29O2Fg+uhh0kmLIXgBFlaeX3PUCjA574u4d9B8QiSDJBC9Kd+iTZLrU3POqkaGDcGVMAlRQumLsc6z/L9GInTwYHlKPT0+eHb978Tb88dXTkxfo2pko7XX8/H65ngv8Lwl1UZQMICdhUyDA5GfHL158e/zkh5qGwIZJHeqMRPHP2g4K7CEGgWNulrNvsfPTyNKR9VCefP/u5Q/h6ckTkDp8XJe/OH753bvj707kKDO5up+zN1zwqJguBhjgPOMxi/k0j0mXX2T5BMbp9fPXJy+evzwJ35ycnhy/efJ9sIzZP/5+FBxC2f7j7w+CB6QXP4d0TXi0DGmQ44eWzI1k+UD1wnIyJw/ZNMriBEuLdSarL644h1CscpiYvFDU+v4ZLJASFgCBEqTLr2Qn4wMWF/lKsD+fvCGz9NcjFk2EIof9zMuEBDZc8SxKy814GDweMfmQwC4RPUHmfIk9DluguT7QjHbVlvYUNeirWBEjegU2ab4WIam7MewSqI1YnJOmYJMEgqMs88svD2hU+mH/LF3d/TGtCjt8NDwKhg8eD/+ACaVy21nr/e3J8Y/h6fOfaGEf1sVvTl6fvH3+9vmrl+Hrk5fHL97+Be8xZ0tQXr18qipQnTcn//H81bvT8O3Jf75FTTkPJSOnPOXTMrnkTOTrAupIcGwkOYw98nHkA3fkV/FiDfOLGRwGhwf//uDAJ4p/ihZRVs8lALd5RuAvfHd6Ej49+fHdk9NxtIZlv+JAESnsd3aZX3D2lC/XU0G2HTuDKIFJ07KI0sEsXX9gC74uEgG0Bd0YzQVbrkUyHcQ5UBOELWbROk7yABMqSxJx0Hfy2cxhSuicKL2KNsLBKq2UEqLtN0fLgJhGg5J7NNQMBJdyEfDsMikAtua8dJ3mFBxYIZqF4wVpfsUL1zMMJQEaUCdqv2FTuIXeiSPWyVJBSwCG08ZzFoAyJPpELJpO10U03ThSKYKXx5kRsqtFDllccvCaDOOALGI0SYAtN2wGFSOwbTAwovLi1Z9JHp6Fb79Hr9+/evGUIFPBB2p8MbtKygWLFIjCcK+SLM6vYAfAVCZSKiUyJV9CaiMyFrIHAmg+eplGa1FrsgXerKe0JhIAY7eVcqS/AsYCPRZyZad5UfBULpvqm9YOFSMA4w9TzmO1PIePBwIEi1jPeRqtAiBtUY7YXx8yLDzRAt/ePiDdwh4ODeITpDQyPktQc0pokLRGnkmikt7RkP0by2fMICkIQQeXxsoMu12S0K5OEjEMHg0dMjNQ0diB4evj09OG0u4n2NmCaB62KdpqAcDxRnpVfaL3WJKj//tjZYHv3pF/sKvBZ9iNdVpqRCw5H0Ii+IcR9WWXSqQWQno1sLPfAbi13mSh4bdFSsgNI6Bq8yl8Gx43mkjMJbElmTZHjTvmMxaqbqTkuFfRZUg7ZCThgW+GV3XvscE3MPmiPCvXq5SfaSgs/5yf65kCFf2FYAhzq3n5TE/DwwZMCrIhMHJSvUCSsAVTbkQukKCK6BgclK+zmGAKwQsxM0gzyWY5piJmAf1yCZqY0dOyUJV4rRWwEReqGcyKaMmFB8tsFYpoifmgOtdt4f6N+mcKimcaUkKxiLyQZmOoSgBeUm7Kv65GMaohKJiBBth3rnnpm/pf1jz36gYYTRCtYIdj11UVJUc9q0o1DpSr0oJDxWTU1lptcjpDYpTsprXgu5JYLZwRgHqN3yjyrmKdCOX7o0PQkAWhZKbUJ8pX+GUNbQxFBUU4vcUqW++y9RKOA8qzlVl92nDoqVASQJNqSoDPYoL/Y0ePytFTG9MurybJvmAHj4bDoef3ugdKWmQzV7MCjnVFwatI6KWA1aWxBcA+S/YNO7BWncoxXvma7I0bfUjE+MBrrJaZmLVkxnyHZL5duVGphtz6vhpguFSagNDsUG5SOe/WYkkz14QDtECEM6TlkGsD7EwM18J+dAjdkeUqTEPUEKCICaPRqgqDegYadBAQYG4XpmEChYJRn98Mg4PHml0wZZg6gP8ESw/V9WEjR9QJSsjyaKi5yuMpImAyekR0nilLrUMd2Iq0EGKarDb3tfTA2C4ThHyyeVBxpVoyRR6DeUlBIvSf8kxx2sMOhpjdv88OraXUi1Vt+bLYWG+7xFZue5jkVclO5J+mRthPUMeUMJtAAOsiTGZ2TTkrpQsEWYcAzBFH4B/dV6aNH6b5RSVOUI8HJNYVizL29ZgdtHiToc0+zizggYxZ1jkrn0F+f6ItjGkYwZ6JsYB2y4CBYOTG0IwZ2Yo0Wo3hgU5ITSDENaaF8j56qpjHT4H0EcdQ2HsqLuFCYtirAM6R+5OxJ8lspkrpl4s6PmvsYbmhZAXxCzxmVzb4AgzyAoQB1X6HplC7v7nvlS0iAp619ZfRZsJDaP50E8Zyh7X2Pxl2FTOS3duGWlqutsKuKGD0oJwAKSBWk/oyajMCpXqvE3Fi1T2F+u8xIhuVhGBLy8PZqQx34Z7kKipnMjh6YxPyAO7pJtKXMS2uyOmRfDWAtlYeakIK4iYZYj5snU3hNM2hMFyNxWlOntTK2isSe9SAGY5xeJhblUgXRUJ5OZiv5TC8tgRZzJXeIVbF0V3LCDANjYiOr+m/W7Vo42u5dMHRbOs5vQpAkQEOSMz2P1XsxEBvpQNIF6HsowZN/nl0GSUpKegRu0b7z8TW+Ri1p8ApqYFqvJIF6dhZlKobmGcVxRs709XasWDN59Yk+QeyXGQx8IRASckzQJ6vyP+94OiA5OD49XMCVoiDp3K1fl6L0qYm/clyUeTr+UK7yOiakBUiqxs2L6LVgoREQEoFomlxwJ6ha53gsCldHgQH1HUB3A1PyLjB5KwmQnIb3dt+ORVBir6ktQ4stlly1ylloyY4+QhB65EtCVKjS63gRFQU0cZohR3Y1FiJWgCDelH0liaflB0OnhJhDgu+/EpNLJPhiRiaCOkdGSSw5o26Cihh7gc706wGCKw1vXDP8OxT4bnWucMmPCYrwlFZjyZUskGA8IxMxblHE3AnlJXxdYhGkH+DIC1P8YsirI3Jvk4IS2BRL3PKSwEw4dmV/hs7InfcyC7CWMV6CRITGYHN0QZrIlsJi6IqoHDIujyDJTs6b7BCve6CjY3GuloNH202aMnQdQCPsJRuvZRKWN4Wa1tWtAW4nVL631AoM+gS8k1rTVJbO8wqpCAM+bATF+YfvE2mZQvQyiSHXPG58tNV1sMjlW38MiJBdc6uVBWZ6fDJwiun5IrWUBEC+9VrWXTesMtktyUpchgJ+egHtFE9cAplStzQM48wupy7tb8ufUqa1XlrWtWQW5xQQ5ZRIENpd5orWWVFVVQZjRCiNPz0+VgRglDFtNyb/MTfGTpQG7nlWle9dTrYpuf97jJ527pmp7Pd4Ut/Oa7f73Op11mo0vEmb+HqcTadayuqI/bEFnS9L3wrcBNmQOV2ZlBZS7vEzona5Sb9aZdVWQjjNLbC5L6Z725GQK+m1aYdQ69G1JcAQL4DgNOi0R9lr+ZLyiOswK5s2wo2+3YkLCRTX/Uk1Z7mSH4VUjI1BAzg8ErTuJ5UOxrZphnuijYhm66gY0fTFuc7o4toWAP4s1pj+Arn4z9WrE2mZJunQST2LfZnrb6SsefKZZbpNIIor179WMH/H2R6iyjenOGytp+V5PJ7kkr+jekhjRRk0uP9e4uHdGxByPVkLvZfhBCr55tTFbvR/FpJdsT0QbctCyBPsYI6qq/38p7QfoqcLspo2naKok5AEAGTgtAZiB3nowPuN1WKQdc6PfCjTKF/Guo/O/ct7UImekeMtKOkbLWy1LKD2UwFfUKVyB9bVIzOEWXICxNqllOCPq0ymmSkXLvrZnrXdquajJErAbjYYINb0a28CPXHbyjFsf3gNWm2p1TRbFacINR4YQVyb+T1DjtmznVFebvrTpHpvc0y0U5V44T2iS1IZTqq1wpVwy7sYVlZrVrJUTZVSMFYNWwFtqeaXOLER0QYaMCjNebwBWVQ3VXb017YCiKCQl9UGq+JlLUxVIClLyTqdZD3zQQ10vXtCROtjhhLc/napIj5CiTXNa2+bY4aHDJzVOrteppsd+G1bjC+bo50a3tf7X2AZQV3Qp0HuVoGYEiGg2fJhLvtcLaaRLvcwIOx+dFR5xKLg9A8VMNYo3koHnh15gjbAi4fgHpUJL9asZ3GniUoSy4WkNxyJcaWNW7srspwVL86anVYlHZRR7s9Fqf/1Q6dHb0hUTmfa1wuRt0zl/i8O8Nw3X8uqXnezL0K5LMH0Cr8Pa2sU2loQ0mim1qo82DsKqAf+yqqw2IVbcuq9iVQtt3Flf9FflGgWCRdN69dv+PkVuNknXYDqQwcKpKV20HEVoNmR/awvmJ7QaFmV02WerDYz476JqzZv9u2WoY9LXfPCzae9zZqnB5MV0H1LCU4jAucXXUb5HoH0ThkuDsLOLJhms9p3Wm5yJX22YNeWuYoovzbVym08uKoC/vRrrj1PsniWor3WoyCg9l2cM3l32YUQsV9BoOBNnhMORAVEiTHYVAzqIKcncgQZKqwsmU/FWK0jfIO6pvmaapOsdTH+xTIJbtdVwb4UayjjGJdwSXDbBvMWkH19NogdoYKZ801OT83GwbvLMoVctZsaKm4aVIhAhHSsGRowvQUJIjvi929Cma5neETSYDyTW3A7rVOqtZghLrcGeiuFUUojg+Z9vTF2TTZUThiPamn2BkKkRT8bg/R243d0ql5O8bRb9+r5SMMJCFzPZRR926iCgat4OdN2EoS9m5JClAD/38rwHQDnQaIqjJuu//6xgUze6gGchPkaQ+hv85tYFAvHOqvekvc08Y/nTGDPS1/Lya6BTbaN0ev375bsOhwtJ9De/DRLXDSLfDSPvhzA3C6VdPbIqhPQFI3IKrfg6z2IKzfj7R2NN0NYOsjQNc+7HRL9HUrEr8Hhv2r4Ni/ApZ9FDz7KJjWbGBpMDS4QfFtb8Ds/QqC1EwPyFHQq4Vn2Gcw9kmHowER0VLbftkKgRjB7lGANugsuDkf7F5TxkV34m21Cy+P5V93anyk4hTKaL6v4MVW2DndHq18O8j8qbMU1nntdo6vZ4ifM6lI9AkOWFQ626SIqkwrrrNdyNAq1pjriBhCknXAaT+4pe2xylduUwR8Zp0eariGyPqU7gXfAAYsJ3HEcJzG/XCmtRPicPhNKua8OmGqoAySanTvjsKdXzEn+Dmnk51WEOhs9Oi8PklFMmC/9ZClezRqhY8M0S/Jh2G4oqXFp9F0i9h9GaVGCuqBmHwrFqFJb+t0+SU6rmdxrtH7b/bygmQ9zRaNs9GDc89ecj3jVkWa9oNdNN4z53bjrQbsZuYm8Wytpr8b1vUN/Tp5R5eN9MFjLeT1bUU7Y9bIxtUZKOteUzsxR+JAp1n6EnT03r6t2JFWal1X+v8E3v+pBB655M1MHV3eztVdZHOrVgYQ6hvb0gzRreH6jLTx2dAd8MTT47fH4PGrJyenpydP6Ry7c11L7TZAbadKp++2o/5bLegKYq0W6MiX6THgOGRSwk+38xIR3QV7BhX/Mi+fERg5KQqctQFR02xrckfy6hoS5NaN0EN9I3SovbJwsZknuGRCl0NnuBtQOp6JxnT63ZYHa7vZxqC8UTeuxXpFwROTEqMLT5k+TgJ6142rv+caVMdYFdtQUKIDF+fOsCjnEvyWbq3FDW+7ONR25s0IQIaYHcjEjVvRkD46bSwcUVBJ2rGzLmcDdcXEJtQeI02nbWDlgV5zPKZp/AYHntcTalDG1AzWbgyAiaYtP2LbH5nr50A39/fOUPO+WmR5nT/d6KsFyyjJrKv1rv7kAQleRqlToXaUPGOOk+u414SEqcnBKhAGU2GMhIob9h/rsHekSTxb2LbO1Y01wdsmJS0iVTxi92Kq35Vm74oQfGxE4FOzI5aCH9sRHhuMVWrUjm/4HTHFSrGP20XdFGudPu4s7Wl1u4hLJXmraEM7t7nlmh84qB+6vnFQi5n1UgkG3moJ6f/SQY+02N842CMwTsUNItXBGaeauDNiXULltEWIHNJ9cuX0S4+c0C1Ey7HkSXKpS7qcTMdGQ4mIyfUGcNQgsbMeMTPlpXLTCVi3FI/X87EIqTJafCUV2f6QhNP0nBENd6rjbERKfZtBP/FtO7+4P6NuBb/tEV1F0k+iwUKd2W/mMLVEIA6jUsUnwnU5zbDNILeumbEeB95e4cssPCSjVZkr3+yD5hUGXViDa1LLLs7GXNrnBX5Td2zGygcjkARTZQ5Aoth8HyE4LuZrYvhreirg9lCgV1qWsWO+N0NoRil5kyvHgeYYU9NtXUd/bUIeMJdZknEDtfe20nuybtZ1NfQpcO4TeRdUnlrvJWZ/keIGkk9e/fj63VtJU/oGe4iar1SgrjQl+hjh7jwrNNtLSX18gHa9piQvlhg6ffdi7buwLffD6x93rS0GRoV0T6D/im/bY7HG0H7p7VsZrX8GNBytfwZSOelB6UgAZPjSXFCX8VXngNaohFdCfzdc0AHr/Svb5yI56ssaN/tSnnfTEPbIMylNuRlukMDG1fwOH00etsuxOcT4TF1Y8KuvAvjq6sx57zCyfGBFEuRIRDl2srwRL8QR/6na6AJXDGDu5BQ7g1ALnq7GDr6gJMGdHYSq87a4n96dutVf0fKh/ik1K0/o4bpQvy4xZAYVGumR3V43tHP4za8P4BYTfewAoHNKiWUdDhN2HFF3M2J0T37PeC1+3Kgv9nnAnaNWA5O36cwtXytxbg3ycChuOcb9mqjP1e4doTyReYsBPt4zPPMlBGvXOPitWYBPoUQDc+kkrj9EFuN8Ld1fo1AcHdfDDdd5Agk1/aAHcmtWgbR11KGQlrJCmqaBct3Nk4lVCcsRpbaBKR/ZLkpGQwJU5SZdI9kgU8mNRoGgT6m5ju/IewBVddu5q4dzVkgyhYQjphjNijMbCJ/LuxRyCOfVlX0VWzCNrLHiw2XE7TMZGTg8R7WadPXtpXYo4cCEEnQkARetsGr4lBliCU77YsxBBeNld9ZhgbpnGZ+hK4sy7mgGgXCjiRIKD7FPu6m6ynYtOSp/b43Dp8rUw9Zgd11qI/ltk6DcV6ZaHel3Kl/EIMMiv7IXwQ5A0IkGyjsH9B937znCQn6lguS0FQttVMuvmgvcsfEUK2pOdFTRrLE403lSz/KQW+zqalA5f02u7T1jKKt+5EFD2eZTTxvqGd3yyOGuc61Z1+UDdXna8gInNWna2K6joG3vWzbc64Lvd8Vl+5v88f1+eYtE34rtZr1wY1xI/WftA3y6odw5b0K331VWjrbBWe3m7Ya6dpVGvWHpU4vXtD/u1fvj3vnW7IdrTfqefJQvnC4ypufxdbY1sc+6cdO97aei5z2+1j/kQTchL1ZXH27sGKwdA+7hKOl4ORh95864mF5HJlExi1ijTJGqisyRmY4pQtfORx/vq6l/1sX4Ee4TPH+B0LgrLy54dUAZplRd11VBGGs1ZbzOMd+da34ncjEL8cmsbr1GCCApw0UkFoQLLIKUphrvyEvrigXMzVefZC72mgydWm4quo7mUGikyXTNtnrbMrX9oKWq8fRprw7y5nj7ruays4Se3zBv5nsMKpgAyQtlRBUfV6Rr0mFIoYUwdLRc0KcKEV8uXRlx8Lz/AcDcqLg=", "notebook/06_stage_3_ger.py": "eNrNXP9z28Zy/10z+h/u0dMx4BAQJctuwgRpFVuO3cqWK+t1JuXTwBBxJGGRAAOAkvVY5m/vZ/fugAMIKn5p3KnzxiYOe3t7e/t9D6/X6+3vfSijqRRPhSfOzt5611EhY/GzTGUelcmtFKd5nuXiRZbnclwmWSqcn08v3P294A/92d/b33ufLOU8SaW4XiXzWCznUdoXhgrnZXQvnnnP++KpdyxmuRhni+WqlK5PUy/vMpAyj65FnmGwGO7vCfxxIhckL5I0EYf+M/FqHhUzMcmlFCfv3+D1Mk8WUX7fF3czmYqfz89/PjsN8Sr899NfRFKIQpauwkR//lrIQjy5jsrxTMZPRFHmq3G5ymXsYU3QIsbRfF4IJ5fFMksLGS6ShQzL+6UMouVynowj4tPBpyJLLaxFJp4NhFwmRRYD/yQpRTmTisoykblwfntqNqEWWGJQw4skLUoZxTW6bCJ++24wGLi+uJxhC/gfoZsl05ksSi8aj1d5NL4H+9JJMl3lTJOY4Ch/kuk0mifi5MNFjY7Wev/m/enZm3en4cXph9OTixev/UUM1qhTf+o/9TWzr11xloFE8R93Mj3yn3kD/9lP3ptUMUo42WTCx1tEE1nei5SZi2OXucdMFcscZwrAf/tw/s5bYHvucEgQhP2XbCXukvlcQNwkyd87QWBCpmWegG9OEvdFKT+X2PiFxLGkIlIQUZ5DdLKJwjNWAgtZBllJOgWHUubQh5O3p+L84uXphYgUz5IUh8ri9YZ+FUMhPn6MozI6iIr84Ad9BGES/+jTmX78uL93zoIASA04lZ2AAkpVRAuwYhYtJa0HpkPgV0XNeaCg7YRAgQlRGmNAxkkZ4sgkRuhkCjldgAFM4ws60FimY+lFd1EOCUrmpaQtDkWWzu8NcAE2ljN1/gwuruU8u9vf+/jR8+pRr5xBjmfZPMZaTiwn0WpeioH/7TNXEPZidb1ISuJjmTG3YCR88Rpi5u3vWcirVZdRQWyFfk5nYpWOZ1E6lTGpXoKjh5heZ0TWTEa0M9pvdJslMTiS4IizeDXGVnD6U5BVQPIK7EpE8xzSf+/pU2U2KmVUPImAD0BzAdukX2Che+HUZ9I3JIZJGsvPSojCGSyFS7qZy3yVgohyFkExsxVopd2q0zN7izORZiW2eM8vIdl3WX4DHheYc5eMJVNzsYKiQcQZJpdLIM+yshbxpv0JfN8Xy/tyhm0CubzOspuDwfOwIIMYPiW58Jf3wlO64xXJ32Emj/CcLxfi8JiQvoIN8diGLKJyNmzYmSfityOhbFlxIJcwlbdT8d1zAVHwuo7vwJJM8+f4W7GASEapQiSIBlcE4nAwaFgsX5yUIElcvH8rmI9J0YHtt38WmELWm4yY4WEJGwru9cgj7e8x+8JwsiK7G4YiWSyzvIS0gENsyQqCMqP5dBnlhawG6FDnyXX1TMpYPWRF9TOvpxT39TCRokkg5R7Dm5BH0G+rIQ2yBMuxmHn9Ho9E2+vTi1OwiB4dbCSZYxuuD5nO5rfScX1QDH7v72Fhn1D4sPAyL51BnwyWQ9NdtyIJC4RL4zZhRuZLIR5BXH6NhuL0eHBEK/50cvnidfj+4vzt+0us7CjW98igkiJH4ido4jzy2ARBYj2yMlnui1fJZ7K22TRN2NRL8vhQH3i6JQiWOaxwT2MbQ6i8ApaFfKM4BcKkmPnbVnudbnbbbYOs237/w6a7XyHMwJ1xHamwA8X695WrOH939gtPrFfsg40UZJD4I0bw/5b+LTX4tD/4W7omEQqh9/Msijd4zR7t9OWby/Di5PI0fHV2fn4Bpg/8wTNBR2PbobtZVkhB3BZk09kGkikGIfjJJnweTWEkbYwvTt+cMcLjgWCM1XxMuc7AYp5drDAR1gcDuVdvfX/v5emrk7+eXYZKKj68+S+SRtgNYKpNNfhj6S9HNDLKxzAhi+hzHUOwIwHBtBB0262xk6YHpPLmzyMOaJQxopfjaMnaXohDDj5UTFAjeHH+7tWbl6fvXpyGl68Rd7w+P3vJ2/72GZN6kywVL9OabO0MFAOMDSvpQPb3/tVST/6HZiO2LE2omIYGzxDSU1aDxs3xaLWZGU5pEaW2U4UmIRoEu2gwLS23WOEC0UsZhxSLhUSfxvmIfSN50rZ3hDyMo1UheTfiDsJNcw2+Sh8sihcySsMqTBhCgrJIv9LCFJJIWDPM8AreL2/sXXsHPUb/PYJ7+dP+iEeE8BS0eiz+IL2AVV/wmf3pKyGCETVjHNhHGJG+uOZ/XeH9qFilpQHu5kzeyrSYlTJJdfJjaWoqRjDIh1c++yWakUw4BIg4cqFf18Pax+XKysAKqLEohHG6DktIdOQXSA1Kx8WA+dnEGJbbmJhWZy5TB1hc8aMY6EnImBbASm8wEUgNjHodL/FyNBpcwf87C/GNOHRZ/ULaUk4y56Q8eqWFA++S9juLnHg5Sq4IXSCSesanesaiY8bgavSJZnzqXOMQhrc9qYX0kDbZAlGxfUEsHRDrsPtRgjD7ECsFxOvRJ/Uk59Cnw+ZEtQ0mCkbJoUcGpiGs0zcACkM1omHMKC2v+ayPCUDp1WhxJQ7IcDp0OK7SJBZHE3jW8ZaDoaGIk3HZJZIvVMoLIdMzvTmEdG7F8r54n8sJxc+3Uy+beAih4oLZcRslSI/nsq+wMRc4kN2BCemDnMM1K3b5A+HAHRbJNI3mbi31Cn9AWPypLJ0eD/RcgQMbXdUwvEUCHN0pOKR619F1AnG/7/VJMZQc3tEhV0Q3oPjNVaUaNdZt7ShWC6d+74L7pAXWSIWlV2+YFyD271A2vBrZ4FfNoz4k3f5KNvJ9nt0mbJ23ihmejuFxUlXW/LWs55QXV27BOB8dTSalXMBRIOgrRyS9V1rMomUS3sh7ZWy1hwKFcx6AOPQUTg8b8ia0oZ6Bij6H4CyFiMpLBuIYr1greBXMv6o14wO7aVIMxIVzqfMRxIx1cACzzNlAFeZ5KrA01RrfpGFc5vmCGo6KW6oikA7rWE/m2TTRbhSJIqalUw4nC5zUIkJuI2MZu0yT/Ix1QGGCusl1NL5BkYSCmuOj7wq/2p5msor5V/kcYT9Shl9XCLwM1ToGRRJLWrbuJXEPjCtH9OMKkTCF2HqEf15tlOEtSez5+LRu6SJMIOy8wQcsckgnDUiT6rXgYOwIOKAHP14tloUFBCOSkmcPo2KcJMGrCCbFdQ3h2E6dlbCx783KclkMD1A9MdVGlAKnK+S9/jTLcL4Qq8JHbH5we3gtUWZhmSoOejaSNQ9uhhqJRH2kBJ//BeIYrLVcbvQMrcrXGaLHQFibWNcYSfcJQQEugr+QJfNTs1YxbnO1McLPs8wespQLNFMArpvOB/MXS4IBi/AWtrDfAoAyqMLSZXYDVgLoePDd8zbUtsgCsNeW2p41baN/b1wfJg3scnqrcuJ926sOBxyMiCWHJnKBjpYhEkGMvYPc1h48KmkbpeX5aw22/TSyrpbXhiADW1Oq/Qv1r7NdJwBgn5PtgI4LHlWiRBIHvffnHy57/W14FJRgOotg3dMS4F3uYs2mNd1tPnK606ITj9kSKoHnPhcIYAWC5/BmkbIhw22C8ugO+6WXPuUryPlj2WJ+IzbBVo1QkkIVDhC0YEgCAUOg8FEwK0lMCWGPQrOREVw8aalVw8oMNBE9+iLDd5dHS5VnKxOKvUY6J2+jowIi1e1uUVmO2ex+r81wUvrt6M2k943dcm2gCakj46Sg6ndEYVM1t8/egeMPslTVuCv+oqJiNnVu57EkoOo/o/lKcmujQ/SUYdE+WPl9ULtuLQTqHqKMA6rHaZZ69Px4I3q7FqrT9oJVbF3Tv1EVjqJjrtvWLg5PyF86YxVkjUlJK7osCZCfxxIqfMr/UKEEBytbrLIMgNw6FWMEfrD9N4XHHfzmwl4xl3LpsJVxt0GU9XkSiCMdavERoYxKc9UhVcehPP4EUS6OJJqg8i3WFhEbUUUTa7OFTc/9PwjZ5l/QEnG/Vsj2K5atQjXSpDrdxb9WslsRKUyXBvWR78k2rxKYBGpGwKwXFAbAmOqGBFkA9rIUscRJcVPFUJfUwUIZg6UIXYS89MZJPl4liMRQGOHCDIJ6khlCEkvYUAqr76kItkgKCuNQfVfICIJqPVz5xt9cT0RRnjBRF3AueRnu4fEWYAEfIwGXgiyIjNtxVNMFqYq8tTkTZ52syowdLqraqBTQ41va7Kssf4GiTDQ/e0sWjkwk4qppmuVaIdpqtJ1R0El00VJmN9CrxsI+0RcuSXojHEvs9GiLB53yZLuORTzXqNpE/1GMVWDYsoz/u0pyy4L1qsryjoqyCoTqgm3tNYipdpW2NqSXLPhrAtgQwIuqcGaBWjtNOKfFYThqz319biHcaAHCg96ywRn4fKocxHPfBJvOkydA0mczmMq7sOTALTh6hv51nIWcFEgdCVu1EY4DYrW2iQqAHO66zyXPsFjKMTIFg+8yX9nzbQ+qcemSUs/asjvyULgid71E7KHfg2sultmS1BolHEcttl8q4lWpgw1RM3Pcyhk7kzvjv5qmrE5j3I405ivZ9Jc5MpH8a5nqqu24005rVugulo/e8SHDtoN315/Jz3FC9X/HHQ0Pj66sg8Ag7jyUMqR6fEj1+JDqVg6qxro1yPl2nypIzZJ4X/nYkLp8NgzJpVVsZqLpPGvvgpoUDVgtAK7UXUscnSRTnsLc10WlurUc0DKOTQnVLaka5TXXdt1W0Zr0ERU3FOo8x6sRHhxYm3CbFRw716u5gUShfrATuwppiOQNEACshmw4TVANVdFoQ5XoX87Ni8aa2HHnjOogcRMjS+MijMrw8BjNX8xGGyGNHacbCdXD0Jl5Ip5TCds1OWAtI1OmlCdqU193y+1KzpO+qRhQybFV0jGp2Jha8OGS+8/U9BT/zYmjzh81UEuw8HK7TaVBscE2DPpJZrGqPBdW1xe0WFrwXb0lU1uyBBeXPJhuKqcv/ZcnlychObYDqi3UDNnwlY5WE0DP9OVnWLPCsdMNFcO+Qpj6Litf0TmZQHZtpm2qCMgj5bC6/8909/8IpRTu/k+SvKhcEcaaqVNFB+WZbGAcNhVAHVTGwoRsfFCq9jzi1ggZYeBbb6q91WfJ4V/92LXR7USf4ZsUWigepnFXirJjifXGbEtXejCWFTCUt0mOxbkE3bxuoWrWWzCnb9+8e1PDGFaluhgKtDiZuaNXUSU9pQ/UdjC6UFXWOCcIVwXbNnWlQeXJ7AowEVVQdPhg/rlJmUzumxVMfU9A7bMQnd6ETaickgkkKeF9GCSotY9M4brMdHdT++ByhZhkxHad3TEd/uiKqXNaV2T0o2tJDvqKuelwailiVJUUWR1mhYbUrQ4vlCigv1m1UKiCoUljDWZ6UB2j3hVlwrRHd2dvlfpAW41SNWi1nXgrFDrIdLVQkRujtcQKLh/mKVnYXQ721eYFeNpT0tPr2bEY07Cjx2NJtBLOpk0ZDtcJ/WUFBWY1d2MtAp2k6ZTOs/Z2KIN9LuhhYSkeHmFeq/Zj8+mboN0ioyIS+vWysThv8seg2+5+ETEVd3UX3NM9cC6XV73zBhIm3Zra3sW2KHzJbipt8FHukuQ9lXDUBlIrqYIinptQwyG5QTUHw1T/+aKQj1O0JIUmoFKB4hj28xyyrRuFcHLkmrdjmkFli2vtbbZJtSvFlRWrFjtQfeBqjmvHcltd1NkqveEcREOPLJxDG/83FpaWMHEobnci6hZE8Ts6tKlUkwWbqGkhhzbRtZRA1ZDoL6ddEN1yPpULM6Z72F12s21Fd8eLt9Y3jsXtRrNl67caXdvzqDL4JVTtyqW+kJTmM7H81zr1v/YSk/o3sf2O49V3fOhYqItFpRkuxRzUtR46Vb6aw6RvT/+je/xz9ldr2ba1IMpxTw2Oj8w5lIdIIuH8O1JoFtB+TX1XbbnD+MWf2RZj1jb4w5a45ZTCf8QrNQ3pttOJP+92O930PhK19UNGV95JWVnGlgTN0TVQ1Y1abRHiKm1uwnJdOCS+qzyOmmJNe+kZfNu9gXoy7sU8XHmuQNtWni5kLXBdwnR1/2Bmr6tDViJdlzs4vNq+mkXBgH0lq7b6Xz1iqfVvS2AZCxav5MHOaEoKua3LVQbI1opG3MBTfhDte5IU4SaqtYOKD4RRbZkVj+96Go90NWwLoR071CFzKm+52ICOAfUyNF8F89WraGv1oZZ8SbCW2p30tohonlpTb2ssP4rmXc4dSFgSmjg4IFZRE/1slB8rATPRy/rJEwxpp0t3xeF4rVZUrzqsqlpAD31x7G4qVWhkfBadVtKmbiv7i5s4yR31oKqS6PlTShhmN1tFynr2XQ6brnI+q+fOAF13BvqC8oa0DI5cet1KEg3VBV3qBI/M/U6rWl3f8QyqBKLfeG0KOEErXGoAtePLoGOsMaNifGD9tiCalzYDdR50iYmP3NVBIZGkBrhsg6OyUNhyE9gPHTAsoEHjqUGutt5Bu+zkNm6aUFJnl6MqRwKJsr6usECMY1Z3JVo+2oQJvdYMlN8Bb99F6TfvZHTfqagDU7ysH7buTKhyGcLt1ouujIZVaHu440KFKhCS9FG8S//6YUhGPwwbECYrHzZ0uK7IGe1ulaGoGfZQGQpwSrVozDEI+ubkmmVPPViX/xboCjn4duJ2WBfjm1U7rpTBE2ubQHdKzbcW/kk+XdEu3tNTjtZuMUajgSLGoGe+5UNexw09Z/vummtqLEs/iuMw0ticnuepsgp813iW4ZOaInCqUqMK8XpQCv2hUmBe7cRW1XetObuh629sAM+XIlSdRM/cWa3cuosyXwa93ffsNb4hLuQ/wAgS2oeoqOuhnctvfUuovrqxVj887lu1erqa/wA1XZ+MGfK4btNNS5voHUXZ3Uzs+gKg/WUbShMVTdb++Cu2h3ZErUtdXsdWorES3wK9TOg93FnvIcrwUWm61euAJfukvz4pqU8N71jXbXMu1i191iAipWD9sx0xDfqEqWrgWA7ZDHEUtqO/0852jPwHz9rXzex2S7B1zay2pQHTtNO2NvozQfNWW6OrTC2e3ogL20+v2nyrtkaRoPX92tDuv9I7pGC3HBbrCT6ni047G1PLTXpCrG82cBu3GxtR3VNR/vz/NUOPv5CjrBTHg3/ib0fsb/vsmPl3+VklAn+Er+b7ByPQCIenCWVX7NPMU8VC6wsIZkl16DXGuyhVmcpamlY2k83fZjQmmQ53v8dXsyrwTSM6N+SMckbD2X01TLH7yI5wrvgyOpNw1ej8mCnD3ceBumaFuKKxg1mHjcZMdwjQC/ml7fvNQVTrsY2kbIKvj5mlcYPMtPgQYDZvYUx6ytXimiwxUj1stKTqwVpKN9yT08P4tdnCRiJXh0sasiuSqmRGX+OyEmQ6k+zOPhW71TTYVQvsum1a0IfMQUeXswGW3TVPvN9VeGIuWUzq76i08OkF9c8OuC82A7oJGhh+d63ZwdtgJ9cfvvRaqLuq4NlIR7RXXZq/TQTZAk+siZGPa0Y+xr3zKvoP1hrzYzPy+Gr449FR0ZYhg5EytmBdjB7XmRzNOI431q0BA2CeK4hulMqseZVxNNPbGd3DaEzOpCbrJ5rylKa0EjwCag4B0n862YVcJWw0q5G0ES8537Pf0AAzuROTrpkFa0tbvHIw9A8nm/ZlUvd3LoZ+HrdrF6y1Vf9kW4X4C2bMc113l/f4S5fEDMWrkzdnpy+Fs8bsjduzIyO16pbBtSwgGz4F1jR7eqpYqx+V+ZnTzewxfS7EtxvtCgI++6fcIveqZNZSn9kkXOVJ8NB9ur5db1tAQLnMitTDzsDR76fsRv+/kwxFu3fQlTj1Ow58YqXB23a737Lb7vcdKHr6OuhQ7Lgf2KwN2I4eH6njvlFILMOX+tQND0PKLcOwp0+LvnGnQNjhlNN1/we97p2k", "notebook/07_stage_4_annotation.py": "eNrFWuFy28YR/q8ZvcMF+mFAIWHJddIOG2bK2LSr1pY8MtNpyuFAR+JIogIB9A6wzDLs5B3ad+h79FHyJP32gAMOJCXbk3rqZGzgbm9vb/fbvd0FHcc5Pnqb84VgT9nPP/2LveZJwWM2kjxRMxlleZQm7FkqpZjpxy/ZW5Hk0Qp/sUGSpDnXw26+FGyWJrnks9w7Pur/b/8cHx0fvYkyEUeJYNMiikOWxTzpMCO6+5yv2dc///TP87MO+8fTM7aUbFmseMLuUnnr+cRgtIwUw/8k6unpNErCKFmQ0ApSR0l+esoyIdnNzeDl8HL01l+FNzfMdXh5SsHmkVR55/holYYiZjGGpOP5bLSUQjDe6CLjSgmleYksUqDuMB7HNKAilYuQccX+8PbqUkv1nUgWPI66r9I0Y5lM83SWxkwk81TOQOoSm1ogbBMSUZYqGOk///7a6x0fMfw599llsRKSx4rdySjPRUK7fMeTRcxJB6HyS8onPswZiq66i/LZEjsMQRKpJbsVWc6ihL3CKRJWGr9a8yufPY/UPC5EMotwNJJiHsUxVme8oMPmFV6mIqzWPIVEadJVmRCzJRPvABjFpoDHrSAVuMJf+ND1eFWoaDa5uenQS8yLxRJ6xbtX8fnKZ29hp1h0Yz6F3tN3QsY8Y1LMhYQ8QsNW24ZhM34Lfd1FiSpNzuVC5Czn79MkXa2ZC6tzuWbKYLiDOSLp8jsuhdFmOaZ6eHw9uP7jcASgDZ+Nrq477NnV6zeDyx867Hr48vtXAz32evDs+iq4uHx+8awcuBr9fnhdssrSmMMe6x49qiiP3gn2I0vEgutHxtwkxWsBBcb6JLWtPTrAdRoLiLNQJRAqPHXjdAZyYFYsMFodO4hCc4BlCqTCNomIYVs8SrFKcxEsCkFvvAgjUl2HJWkSlCbqsCK5TdK7pHYV4LzA7ng6BbZPYXUb5XMJb+8KIEFLDY44yGM2fDW4ZFAlednxkUWfp2msfHaR1y4449g9ooMoAHHFO+wdHCHU5B3C2PGREjLC2N9LFjFfC3mKxTzXHBoRwATAhz5SvQM8JCxmQh0f3dyAH39sfDh8/E2lQijrW/+vKk3g44AyQZjYAqeARwg1ICgIvmKKAszxEUUJ+FeJqutCH3+lhZAiS5lM07zXozlt83W+hLjYUkzT9Pbx2a8DzSZ4GjQK8bM1G3e7lTgKpj07O5/QUCjXXVkkk+Mjh6Lz8ZHeKwjmRV5IEQQsWmWpzC1jKKIyo3KRcalEPTBT7+pnOnD9otaq4k06msVl2Kom66EO1CPisKLMeL6Mo6mheoPXaiZfZxROq4kLODGfxoIEgy8MWV/TujgFlB0Eni+FSuN3wvV8iAtPhLHXyif+PrxXyNxFJIcRXFruebXU2D3IzFWAEBdnjJ1A13/jPTZ8evaEdjw++l0tP2xH/5jLSoTf5yQb0F/5SmHeAYke7VgOK7GgANEebMBjU9bu1yMIViFEvM8DgGkKA60sYj2epHJFsBY2mzLo6AFoy9ExxNkNItVkNV4hIJVGIGuO0KfswcNqeVsKT3Gm98BxJOYDYLhmWAUL54PqHpYq631YgZCWgtXYEmmCrTT83FDMeRHnwRz5RSrXfaL0dgxoGOyb+oN86L8T1v3f/WEnxPBPdUD7DOxxFBMxRVArwS16B7Duse63pXKg9Ykxhqx1TqPQ0XhSzkRzVvglIAlIlBbEmT8aXL8cjoLR4M9Xl1evf6i4GE4+zzLEYnfuVAs3hsUXcmu4bPbZbB3P2tRAXecY1msjxZurV4Pri9EPH5ajXrxpGO3Issds27qQca2kdyL0LBFpMQ7Wcu/DAjhilWHzFqlhJAVieaLJS/C1jFl5iSuy3p4jfbolSWKR+bWX3SMtjm1IWCjgxVKERlwko5gkpR1mhF1U5psgYRnr+urV8JCh9o1lcreNqndAlNjWkUdPmJcdK7Z22dpCF5XMVoxohCACCEEkh/zIe1DempDQZd8h2x7bgLQW44S9EdLc80zxRAdy5Bqx4MhZv2I6r6VcuhES2RIKkwI1FleGi66uommR0zWLeUo+FikqoTwSskqWa1YAQHGPAnb87B6nauDTXkxuGYvENTt57Bv21b0emCYxvK9Fvz14XjcRGPi2D3WQ1PW5vHsd5jNE6++QxVEtmOlMRmc1g7fXKHnDCLqqktAv2UvK7T9LNJ8aAQLaPEBu7+5cmNr777lbkS2+FAk0ilqVs1Dyeb5H2uStRVZluFzmEd2GZW5LjEaYLpfXub9Ot1E7xXYJQIk/TZSFdp2MMJR0VdowKxsHzIQ/HQyR2ysVLZIq4+mUmNLFnUGl38hQBbj3iG2KqRL2zVaoIZgII4oEqKx5yNK5yUT1SVGTcpS5fq2h8oEMGlCuCUcB+p8PRoPg+QU84C+D0cXVJeqYubNpVL/VlYLTiqg1C7+UzbXDBfoJSrAXSHUv0/wFHDkcSplKeIQNpVUEPZCkOIi9m1YsKoCqu3FeVrfGE4gDxCaJ/DjloXIbUWDQMCAduyjvUupv9BGo5t3fOJ5nrAtU7R4diH7wyFztLSHH+ICWzEaHFERla+sMNfHDRzDrq+xcgQlW+oCR65gxp4Pbr6JFmTCvxT8kCObagtTEnywIVj4kiBLWtidojrBSh6T8NdIlMnrZ2jLOSxW/jZeKb+UFNrPGIVSOvgxbcvKVlciX1WWB+hO1E7mEf1D2cfuq27Rfte9oFk6PzaGn3FXjciBQYuZMvM6BBbgCbHK8PkDczpJ6cNV7qZrq6RDdtv2qMxe65EiR95uH/kyMh5yYqkgnH6ppMHXpYuPlrU8tFrYUhQSkolmn7ByU+iZbN92UD1c4JldTQiSI8iIfo+6iCTy6dgYmFg+dxIKXym6xHFxckIydJp9y7APrrO3WJFJ697byaMjnYeiCzGtPkWaoDiQ1OCYm1vmj1oBVI7aYVkQmS7CU4TaC9rFlp07++vTQhLATtKmzQ66hEydyKwIKmyL5Rpevbhm6pRGrTbrUJWxiKrC52PEE9zDUD0K6VPaOqr12rvkROJx8UklrgEPMo47Bh0h0JzgXrtmhBY1gFVF+aA6n8aFde+IhubFGyX9xBoSoJ/6ZjS1zQnDpnrePGKJfRP1uwlV5WK3XHVwBLqFi3/QrYfAQit6+u7d2Asd9iimC9O0OoJs1JJ5OLu4R44S9QIE3RUu6h4oP9Q48OhHRYjlNC7kLWnKoVZS4hlcHDXO0D/hqGnL2vqfn+FS55ZG67P34jGDRHjqHPr3xk8lBjzhwzjpvoliwKGGjY+CCvj2wMiZYw02ZSXOO5XkNoozf7UPKbYtlFzX99k0fFJuod/Y03O5G36Zr1q8la8aA9B0+mDOcdq+FhqzfPHbu01y/edyhaSmmbx4OETUXSx9pTv2mk4BAX8muWW4L28SlqkjZTbrdw0epI6GRHZhqzNRvHj9ff+pN9UkKu3ymksacN0+DMJrl97c0aLbXUqOVhDiN2nDnoyI9iAgTfBVoxsoP9I7ol9ex12peTOx1jappZdFeuV9Bm7XbpnVD39w+om9jHZIqWbj1gZ5PdRWkRb6XdF9eXo0Go+Fzk3r792TfGV9TNouln67E5iNMIKhoIZ2QsP9HReu1SdBaTU2FFrHXJjZdBpDiy417/ou6IS3ekQqoPxFQfwLcXS0JFOTRjXOmLxzasR1gfuH+ujXSkqLICDZhwCknxxpoZ4ZsK4BJXa8BqO4KZX6JT0KIa3DVMShpd1iqwQbaOMuKEi0R4LJ093w1vTM5Co3uZiUawJRc4tONCN09IPuLOJ26zmkJXs9OVOiTQrs0+4SyjMQyF91OMdN2A9plbA9Ndm6GXdyVC1qDB5ZY6DML6qE98h1AlQtag3tLDjkp4bBcuz/bKrq2bYtvzPGJCSmug0ILsSve9SHZPoY2sO6e0iLPioYrTsmQXLyze9I/0u8BqGagfzSMkDJX9s6ofK4+XPoDuSgoYXhDbxJfbOofovQd83sPqwulZnw+h6ZMfySjiiXgFRPXab6xOpSb6s8/fed+6urzK4j5rNxUobDGDSYLcagopc/tIs76zhU1O+sWXlnDq9/iY7KujbQH6sZml0xafnQ2UmB/ugwyX2uAxFFaf3VmgT7GIprqkgs+ZN7MpaHc5vsErTWR3U5473iS68bwRvgQMMrgyroBTgZsLULojiOoooMskvq/htyqrC1xxrLBQT2MZbLtVTRdilCX2ebTRLXIkjWTVE05Y/3V/OkEZA3rWkrHdvcSyef1F3jNYN5wmLZau7ojbTiiI23uXoRwq+tLjZlW79f+sADI2ye2O+BZmciDoq2ChiSX651SpGz49A93gOmHHW3yhHBA7k7L/Fam2CIL6orcpjeD3l5VpnEABwjgAAcKMqNWhmJmo+XaGtHrXHbTbLq1yw5M4N2tvKtT+wSST8/ZkWSnWVZvX2c17XRLS+Dt02s/Q4+CnE73KCoGu8G1aldoNs5Hnvojz+scjBeEIccE0v7GiPWoia6PJttS/v6G/n6Y016472uA12z35h9NvK2tc/F+Rj/+Gup/KKaifYixHRM0evjC6KHHXgwuXiEfdTeg32pDVr0snTWsqXBt5w/3OChfLCT9KEr0Su+s1o8fGX8nmS1H7bQVgmy4XtG+u0iTB7447XxXOisvL2AkCBK+ol/ZIJlzgoCusiBwzG8j8BMVNJFzV99wnvdfKkiBSA==", "notebook/08_stage_5_sentiment.py": "eNq1W91u5EZ2vh9g3qHCwWJIuZv6mZndcW9orOyRJwZkz0CjRbIQBA7VXd3NFZvkskhp2oqCBLkKgtwleYJANjabH+Rqc5O8ipAnyXdOVZHF7pYsr726kMjiqVPnnDr/VfI87/Gjd3Uyk+KF+L+//kfxTuZ1usAvcZicySyTlfA/TfJZlnx6cHQ8PEuUDB4/ir7fz+NHjx+9TUuZpbkUZ02aTUSZJflA2JX9V8lS7O4MxF/tiXklxsWibGoZhDTxixzPaiTE+/eTpE62kzwv6qSWk+0/lWWqiomM08kn4a9Vkb9/L4Q/K4C+TkF4kWdLEPumqTUGgyAzjG3LiySLK1kWVd1Ov0zruUjG46ZKxkuxLUpZDcdZotTjR8L+LGRdpWM1ABONEokYz+X4vCxSCE0lF3IimnyC5d+/X4C4TG2fsfzOZFXH0zRP8nGaZNvv3zN3R3IqK5mPIReIlgWUNwvAKuF/CT4XRS3KBFQMxN7znZ+Fux+/2HsZjDQ13cYI5+fly3B35yca4vXb4+Gz8IXYfTFUc6DSPx8/D3daCLlI81TsAuhtVYivZVVo0I9/Gj579hOicb8az9NajuumkmbloVhRCgEeCuLaHyt51sg6z0qH8UCAczmsmxzi0TK2At3T8hVzmUyEXxYqrdMLCdHncpbwI+llXuC9qaskI63QJBwX5zIfZvJCZlCZfAylyDGjyEUxhfRPPjt8dypq+aEWJ+8O3uIxqWbSvGCvVSHqubRkmI9QSMI1SQmPIgBRFllSpfVSlJWcpGP60JLwDhTV6TQFVy93QPPejlBlltaax62t8RzqKrPhJFW/JgXZ2gJBqhY++CkuZJUlpSXgTNaXUuYCGNNcJPlEQxoMiqVQyd80KagQZ0smrbMlbStHTS6mVbHgj6TZoiqKejSib7RGuaznEA8MSJ4Vxfn2zstYkQnGL2JlDT8sl+JkOGTdFRs389SxhQf9AB1oGc+VeHZKL1kl9uTwBfB45H8eP2Ka43jakIrFsUgXZJTCmDrtBEHZ0WpWJpWS7QDZbvtSqPaxghCLRfuqlsqsRG6AlU4qu1Q7ZEDKpJ5n6Zn9/BavRMKfHRwdiIhffdCbZqA2CCupiuxC+kEIwiDCx4+wVkgowjRXkJgP16bqyqfpQdBShAXi0vrFRImsFOIJduc3yUgcPN/ZoxVfHXy+/8vD4/jLN68ODrG0t3FLIMXD/U8PDve+eAWYK8+akTcSuwPhWVPC6841YX386BcOx9oAD+AOj9gbGiNnFRgR4fo9j1k3RyIlFs0IdNQZsJ5zJKZZkZjB6W5s6Vkdt4T1xkk9Fa9L/Hqa3omcCoBMYvLvMfl3PxDDTyBDVZ+QWZ4aqqFSh4ATcAvVUpgQ8VQJmjfkuGACAPx0XUsoyViqkBWRplfFJdbusIKEE6Pw06JivQC7cB4VQpCfleGr/eP9eP+rr94c7x8fvApnWXHme1scULzAemr6kSVw0XBIfCif9aOC14vJR/nsQNN8FnlNPR2+9EhP7NR0SkIBhhA+yvdS1UnBc5egH3ivOs0b2Y0+EYcwdtGU7BY6iRjXwnxZB6XdFxDMuvmVnEEcECY0qzrxuqjrnY5ExdMrkgmEwTtk4f3gukNCUA1BWSY64XsDiHiVDbDcaEDrf72AhaAXevvmcP/oi+Nfxcf7f/Hmqzdf/mq07pTWJaG3IUYAT8CMpZOXkWWfs4G4ug76M0k1wqQsZT7xr9aXc6ePxBq+DRPMBjA0E6UZtsMw3CY/z4tLKNKm6aQ2mNuc8FOcF9UiydKvJVYTJG0zDjs4g5EtNtOg4x6h0Yubd6z95hjuavPKbEF66XZ3VrFb6VUSTj1n4XWWbFiMrdrFrHb+qvENOAbGU7gU4yGwbTvhHryplBP2OxjY/VnQ2f47Dr/sqLY5gKpmPIfiYypirlV52kXEEFKmswIWDb9d65DveIJ8RkrCQSQ84j8+LWv4agNzZJ0BGYfdu9POLIipVhj5LFTzZjrNpG8RWHxNXUynwLZIPvhw2uAN8kAq6Wcy74DFVieTwPoIHnEJwk7a15ORxnxqYTm/gFer+iSSxfUYMMbWw33arfcQFHdMNyrBlOgtHvThOkVRbYKlVSSmxCrNW8gfQ2XeUo5vFEO5Cd22TedC8UtFiesc+VkiVpM6jcjkfUWDjCyTqANQf0ih0glc7qJE+ujXyzIdJ9rfJmJ3aNwD/FRVNip4sOaZKLwxRul4fGf4Gs9jvVG56GnrCFArGhsiHWikQpTFrn6eZJTj06+TU8fVlpQ4IMAgBR3LOWISwivyakVZ2xj5W5ItFdUyoJ6DD8TbcB2hUTwRr6HipVYe5LSGHv3xbGl3eSSIlxNMHjisEW9X1x1vLekOfR2OEDYBhUqaDGblcM6xx3r1KnBFNRAzpo6cRIcHldACUnGFYJIbtWISZjbbhPaYIBkZTZucOYm0TYQeiqPN6BwcrnNpaQo2f2/XC9w84Tg5l5396FJCJvCfnCH+HFReUP4kFVVGSS52qT7WOg4SKaMU8gM2yKmWc8r9YuMwNrq2jtQV30Y4O9my8u24iMHDvYg7Hjcg7gS+ipggQwROUoh2+ZNRx8hpsBm4RUnAlrgeMBnuOuoO8+hO6A53h7qF3uRPKQQEjh/9ceOrLuHXPCGVyKafMRDTJENhM0NjY3wu6sJxrRqZ9uRdHGXvSk5iDSsCmHGsZVVMGjRLEnKuQ1I8g6wN4J0XvTu8gKd7EhBHKFoORsomCdfxE/bJL7QNbqbMG/GgoHXXMg8Mjk/E8Mf7EU8I4TEtSTvm8/bOQJmcNpmYSDxOdGcFQgBluYKDWpBjT5VYpEphVvBHoIk0lwURF1WsztPS12/rerw10OVqnCcLybXjQOieA2sx4kZlNNzR5WPdanH7aOzoaOuqn9ud8DXiSVoNOLA1Crm6UmibBK6yLR090D2Ynpx0xe+Tl0UeABLTGRL2ldpkH5kad7WQxWM1ev2SFv+8qN6h9UN29ZkhcMz7sZJzM0M0027lfjVrqKkDbXsF7/xZkWVJXVR/jv19m0yo3HQwuAWnJhegcP4bSZYfxrKsxQH/Ic1A/wJj68bwFSwVzsBzxbGtETd5cpGkCGsZkF9h9p+oa8/2qmqUrFFfIiHJNYaPYS1Axt1tuaHdFtFk4kDgd9ScTD3dEETe85Tqoqen172+4Eh/4ef2m9dF4k7z3EoH+UyutyI6rhowWmqxmjfEpRihaFbPo70XPzXzAsui7rvEtDPkrlkoYVOnmQoplobmg5trkEXEhDKOkRJm09Wy2Qidwl9HcLtghwFMUiJjsCByrtXf+EoJ1vlIa0GIMIQ6x784SU8Dlsr5QFxwQW9lblOj63VMJnlRlL308dm21UlHLlZok53TYCN/hNMypTuV0QOs5R79GVDbPdZERntm0aKpyeyBGx0HbsC9i199cYQ+r7epp29cATSI1G/NBB1lLPhIgnBH1BY067ilNlGjJaK9WKT/OBCoMSrCHyPWyCirnE9IyuKJvEjH0uBA/Y/MX8GGopcuimI2IwyqlqWKcP7SfaKDjJjjmJwtIy8vvIFr2eQd4rqgD7n0Or1u466srAjQqOtmssAj/j1gQUX0a7CS9MQTrfhRaxrBgMxZe4EIT84Mgo3HxrFFd3g52NZ5sJHMkP/6K4PMPpPpuxsUtK5JQziqtAHM6KoZRUfFwDodTTqCaiBjnyP8PWENCEbcdhZ/yR6Ve58E1EWzA4PLPb76fBeJkmiRh1TIIf+xKVkCb4AuHJ0AKRRshuwmk3ymMxEKpSl2kjMzdE+0pSHkG0aEr89QjELQF+jDkMvdrpBaPahoY6ZNqFrW12LHlWd5oc51iC665zST3TG3wx3ubOpZ6V41faeGOLWY8ezp0lw3itD2urZOBbS1gtd81Sx3Pphpv4S62ulVg/1M4N5s4HuH+/uzh+8O2i2ND4itpNAto8GKG15M/kCP+11YQ7IIf2WcDt6sYSCZc5sarStjz30/DGe1OvjkRUw5rR9saBt3EX+DYvakSEeLkfjDEovNKCFBk7kQpoGxg1jHShV5JXVm13KOfpYRbEYNZ5/WFJggZX9rCysFoR7bDM9Ct40RKqy15MAGFdt65gAuaBENdwMO/X4Q3LU2745F1ob86s4wD79T4cCZWpnNglcvqe2x1JlHORBL2qCvURAwmQOzQrA5W4jNiX1stdEc46r+7IFOm+yuu8i+O9v943itvufqk+e4salHZoOKGweSky6Vbr3ZE3HUOXWqyykAjOyBttDjFChYSYftAThGz+Xysqgm7Sm4KbcNM1C7D3zgeHvzb7c3/3N78190ctC+3H7z9/r9d7ff/O3tzc3tN/9we/PN7c3/6tG/u735loe+vb35l3b09zz3X3k6vv0HjaINdlnP+Ql+hP6iEzBN+ZwCLqXIZ3xaUtLvNB/jVE1Jz+ShVsgOrf/MS/wn//6tXvWfbm/+XT/9Ny+rP/5eD/2W6ftWv4CLv6HvBPU7w3FWKDrG8ki29HdSFUzLRI7pcJcf6RxHD/XIu8e53efU7vVSxjNRo5MPifg8yPPI4C/p4NTJ8yjCGzPbZaSXhLS3v4iFl3qlD7jIMVyF7cm3B+tkpK4z2SUovfAnYsf03xzgB3uLH2begwdeY2Aji7xq3YL4GofNhTiqBZ7Tc3swXdrAB3al9ki3Lns7c7fb404pOchdzk+W/GiT1fz7ItnpkNhtmf4QSlokP4QSyw4gx6mCFMmhUqwskeZSRMLDR6Bz0EICEPu0GSwfbEAIRSbI3ELmK5AG4UYwZ13t6wGyh7Zzn9wtlyg9vQ/wkQMAjHL4ceDelVhDSmNbLmGrSGnsIwegh7QtFOAqWC51HhgMViV3+6bmHHu7QU733O3AQDx3jWsl+GlYPbYB0gmJLSTT3Yfsia2PuvdpdV4n3v4kV+x3rbRGWu/THSutTXL3oj+jjfNG+O4nE+j5Lztz43bMPS0K/7YMv+480IJKW8T0CxNDECpPTQGJPW/rSIQW43HIzO39qtB2Lt7SW+VPpBpXKac/kWdvbrbXxuytmsqzdhWi9I4Tg8P3zJUyhEBzDBf1rjbdOUs3PijtRTkTcS/XInh256SsshO459tNoXtnQa9PU4bMLU1XLKu2W8anktHaraNe8boSeMuKclXvhG/VvaAz9PvvHYkl7hyi120vtZI/1KJ9jhuTFbShdz73Dr3BTFwiD6MjEPfQ1VTgqMcpz8A+y2QhmArc88FJHdKlM2CvQyfMliFjiulOUneT6fXro4PXdJWJ2lx2U2NSMH2naSBW7r54el9HLNDQ9HY8c1nM1ttd8b0y+UdJlbt6vidwag25fSv3Qo9xaDttu/m+06KNh0MRDsraGELbPu32nUCjK8o3ONW41uj1CD/SEBCZETxhoFcTomHuEhFdmUsmvdHg2gt6bVDdaNJNtIXiOLV+fOIelUTOpn2Pa52mKcmTTWcSJyz6Pavukoo+OtNnKKhSWjKvrZabhAlk9xpjThMs6poH9uBvSQZKib3jLzeoZN/RGt3s9mOjI243x/3a2wEAbdwYZ8LWlr0wvkogseHxYZXTE+m3njg55iMUZ/JMokdJV+DjhKksw7ipx7goFuNCiR90ceAHmbmR7F2baQ03ujLsnTy1Q09PR+Hz6bVY1Wht2s4Ex9bvmQPT78+xvsDM8e60Qn2bVej/G5jNKpqHfxzYwG7QCgvxGfWuaf46DXkyFqftP+za/s7WzKdxU6WbjQr/1bDAWfA8UXPk+G6vn9N+1PDaqK4cq7omq7oyZnWt/VC0ena90n93PRuuGtNpDxGPa9V0cSSOKSmIY89ELbqpjC5m7XOuEAT/D3g6bHM=", "notebook/09_stage_6_aggregation.py": "eNqtWuty27gV/u8ZvwPK/AjZ0rSd3WZapdoZde2kntm1PY52pluNhoZESOKaIrkEaFvrutOH6BP2SXoObgQvctxu8yMJSeDgXD585+BAnucdHnwWdM3Ie/Lvf/6LnKV8WdQVZ6SsilWaMULX64qtqUiL/PBg/Lo/hweHB9dpybI0Z2RRp1lCyozmITFL+Wd0R05PQvKPU7KpyLLYlrVgQYQTL3L4Px8RcnubUEGPaZ4XggqWHP+JlSkvEhanyTfRT7zIb28J8dcFiBcpq0BltmJVxZIPhwfE/OFpdg/f5AAu0iyD1XJRpQtYkBNeMnoHX46Jlk2WsDgRGypIUuRvhSMpZwwWotWaiSOwYw1PnOUi3cJfJKMLlnG0wI7X+icprdJfpAOHLVgVoB6YmClt0nxNUCiIuqqF8oVxhY4F+CIxgYp1oLiV95CKDVhQMUbuU/bAR45O8OeIlKw6MuYSvxaCVTRfgulFnYtQm0hgBeUmUDwkZZGBGWLXlkU6owbMCIZWN24n/pbRnDQqZCxfi82goLC7NArSuhrtyDZ9HFwQnLSu6HbLiM8pyCKNKwldVgXnhAI0tFc4KVaEdtdbbgCKLAOsWGmteF9nNYdIVawsKsGPV+m6rhg/1rbGP9dgU9xCQJmvIWDpimypKLNCZOnCkZeCTvc0BWRlTO6MM8bTdU5gOzAT1CMyVR7ggu4AzqykFRhFVlWxteD2J5/OL6efo20SjNz9TO7YDlYxS97e+o1yoZkt/6/cHNzeRmbZa+Nx0BI2VJKiQAgZSFWDI3IBPrQ6wLCy4DDqHihlUdR2Y3179f315PJHQvMEdhgqZkaQm/NPP3w3mV7dkBSkbhjB0JkYhWRRiI2RktfbBavAYRUyF+OsupfbU7kj21m9P0s6OJJ0YGHHiZ8XZFNvAYsGSoHDEwBHtTk4qmnWbKETaE5gZHCoRaO7ObgM4U2dq9igOYgUUhWFGI3wm8T0TmwKFeJFUdwdn/wx5siZ8fvYCVxU7sjs6ChFpiRtkpx3UfulPyCnkDRDOhQDkjxMD4cHUuE4XtUCAB3HJN0iwoleE23DUeZttQanc2ZfIC3ZB77jWt6yyDK2lLONwG/RxawKScJWtM5Eki6FHl1SsYHdYUZeUwz94cFfzm/OyVg++qAg0GAcBxHEv4Ao+0EEmgA5Hx7AshGKiNIcoCF8yDwQGR+nB4FVDhaIS5O2KCdZScgbiMXPdETOvz55hyseHoByJDaxjzH2zrYZodyAHH1DUPlZiny6ygoq5nrDgkunfW5DjmptOIsRJ3sQFadIRgVlVUJsYzQLXJCV0dlkOonPLiY3F3+bTC+uLoGoVt5To9uzzBB6LpAOotWKiNgjgJX7gZMtKgYBz8nTs3qFqsBKKCQCkxLuN7MrRpNYsEfhs3xZJGDX2KvF6ugPXqD5GJQf9ZwC4pxY+/KlHo9ZkbM1bn1cOAJK8T14gdmWeyGZzV1VkxpV29JH/yQ60dJ9GD3zWJ7EnC29OUTFfQ/bqhL6i5MyQE1UUI+xEYFBc/K7Ma6jfa98I9WGOYGDDbP9Y0hGvn5IgbEBX1zMAB/zBiDw1AfIx4ouVcBXpJmvihJJcIZIjy1jRuSygIdaVDRTchzSj8iN1JYTti0lLS0Fedgw5BlVuGRAlg0bNgjTKHGM2IsOzY9js4sdy7V7VU4fE15vfTU6uqdZzQB0QcupT+UIaLHOE38JJsppIfk6kJAoQ7JETGgBqWBbmP/cuN8SmG85MU7SaiRJIpRYap5tILRdJv+PnOjg/xCoxtDFLrb5/6VxoAXS/w5fVYBDuw+9OZgnjanQEti5uJtiM15bgyLekMlyWW/rjIqi0nka145Vio2VE1wltPO7G0u/DhwRFqW/RkiDmZGZJQHuwCAwxpjNNKS7qMuMqcVh881fqYKROGzKr5M5ZFlHYsdKd7YqY/dqIonAiHB1wQ/WYRLuyO4AEg75iSVtSEfrrFj43m8VrQcuHbKyTdSv5Gg1FSAKs1nZxmxrxJYJCmMMZCU5y4khwN8R1pTdYzNNEbn9AEzu1fldXjyACc1EU2t3punXeybhSTCWld0YisMiA5XUtJTH9ps0tmVMB5A6pP3g6uE9tH1pgoWSmgHjTlrfNWhAEmcKFkiSoHdbjOTB+KGoEq5ENF8RKTXCxBjcoLeXKfcoBbnttD1IH63GpFYy1bMXEFjMu5pC7eS1J0Cy0HMwZyhmm05uPp1P4+nkr1eXV9//OOpXp3aZQZm2lLZqmDeIAa9z1mvyNYzHLK7nOHk8JEenQdBzh4lBRJPEb4Z3BubW/bDBfSMdt1ScF9WWZukvLFEecr/BgWMBWXqrfecFES+zVPgDerhBhpDoBd1YD4F2pp7mA1Hsp4uZ3XvzL81rYORM0oN7sbehUpG/vvoOKtHpjy/Fvr+bZuZ5SKd9ucu16Evz9+SMmaEvJ/Z9YcNoa3t3UNB+P/fzzaCEF2bqXDM4LaJlCQWwr3HUYr435PO+I0gziJd38ogDkO+dedT+cAVeO60dbD2p1tYGzv46k7iC1YaLcdxLZZThN58p4+6CELYFuIvLvNhznykJRz2AggDym7FKcANgxON+mtes/UV7Fxm552+j0HyIOLH6x+yOaX0fUqyAYR5TzkG3+HIUeqWvt8tuIxQXDoxxEsJIuy/cJ8qEWB6OTCVugCBpDRchcNCCULwLhuRgV89JMeAyBcBGHJwBtCMDqPHx4IaUal6F5HSvbO1Et7cCYhunD82xu70zq31WG26dzEpViQHc9vRWZvWsSUvzL2fjfS0awOhLSYuMEYR3+zo8SMH7uHdgTte3z+2qSJ2DZnKzDADPLQ1Hakt15NlabWSKue6ApgYcNYVid1CrdBs1JV53XAvg/fqmO9zsMNwxqrR2ioCuazwnKyvpzove2EF4Shropu3eOv8rTP8PAAT0tIW8DknBPsdK6pJ85DwPDu7QzZOku8AQhUAaUM0gaEMrytdMZM7+jtTnTj6yxw/aO0d3j/IRVNz6KOY3aOzh/lWg1YCX6MI6PhzePPh9Nn8RySevg5c9eLwWUUMT3IOb6xm3vGpMm8tSWT++ZqY12lYl7RLiDfmYwg1Cqu4eddwQNU2/Qb77cskg6RqGh4ChB4RMK879+sBtTO2tL+fdZtWAUmrgAFt2UfMSYBQdgeaus1+CkDujcXLwIq6sqW6JP7eJDlpw8PE5+C/Ibc9BY/5qSA6UNq0+INYIuheIxcHXAyzmNghfCKRGQCf/DdGIaUg2nzy+3LAtxTMdV3p7p9GJ50z21ixnlWzSUAHfgUXBy0toVsQQm9aO86DmjnXIZGJRweuOGOScHqN68l4QFdJH6jtWiuEbwQ/NYW347s7Y4/RV4SIlAV3at5lwgenr+2cFgxAb6PJKwGmwYi+m6W7/GW4RANhw34LtbefE0LmI5h+0+zn5SDNghYF7UjhZ1HlzSWr71tCTcva3vt5xJzcfm7dRzZnvTdZrt63Qmwt3b/g/vB8qM32LyR6X6Otz+Q/4st8llxa0eoR4pNCua9V7OnM6jTTdhLdtopeEp7I7oRnBNjXgnhfwHuJ97zij20UC97IjWWtyw2a2fT8zs2YcSNshDcUO8tQFq+g6AC654S4dDzrgjIjXC/QN9+E1h17I2H8fkq+i35tF6GO0oJUP00O9IP6bFdXYe/PV4v1yceI1IyFzxo/ydsL3PmuIXJwRc9l2lBVLmgWdCTs94YdG7/YAkYqM+SvvehB4EtXyhzBPOjazt02l+3b+bKSBgZFI1xsRZ3QHkPed95zeM/jXNzsBElSZjk/fnZjOOXhqmRUANhjVvvmYVnj+NFtuS9Pchw1571wdkb/DTU+O53H8R24viKPGBHZ9zf1rNKnWNV6WXeNT5QOvLKtUwnPsmd/h2F+RuD8LMDaWmOFjqsX4nr5w9uz97BhLNXPxOLm8vJpOpudngDOxK9lYbv99ktRV5j5Rnz7B1f8eWSBFbp1IWolCufSRrSPwTaSvSrd30Cf31S0wH6N3QyIvOuPiTj4aBekOG+Xovub6COVIi8Oh29XmWtNcwToLQ7ry9vxCRzMUSHwAAmYxvnOgojVpLgFgJ8U4UsB1ne0c407U7R7soemfamHWU9NnrcwyH6h50hcYvREpLf948QkMv5HXye3BT1KHZ/wdi9dtuLQUd9tnZYXEs/Jm8ocN7+fuT3GesAlgbHjr2PB2Hjw3+ZG0usRwyd2bZtOknNgkzQ/dqQ8VZE3yZAJgNzj4x7Vg5NZ8Hf21CHf4s2HrI8UoMt8B2UK2ApeT7k+EvDYLnCgKABXiOIefvMBPLuDo78UxEkIce1oZ/FEDgFn4kieC4D/P2iaL", "notebook/10_stage_7_evaluation.py": "eNrNWl9v3MYRfzfg77ClHkravJPOiWtHyQVQKtkx4FiGrCBN1QPNI/d4jHgkyyUlXVwVBQoULYq+FG0f+1QoQdAWRZ/al/arCPkk/c0ul1zyTrKT2EBlwEcuZ2dn5//MrmVZN288K/2Is3vs61/8gb09CLI0jMs4SxkviqwY5EWW+5GvRk78pJKPN2+Mv/HfzRs3bzyNc57EKWfTKk5Clid+6jJNgL3rL9lo5LKfv8XmBQuyRV6V3BnSxMM5Z3Puh3JywUWVlCybsRLDuZ/zYpvNs1O2qII5Ezwt4wX+Y34QVIUfLFksWJIJDAh2ym/eENVUlHEJ5KzKRVlwf8EEEYHPcTknrHHB0iwWS7bwgznWHCTcL9I4jdgJLwQ4ICRZP9TcEszmgNQs4iED0+IUaxKJwl9wFmXYseCls33zBsOfHTv4fzf2i/hzydRBliZLKYZ2CyLICiAD80/nQIvFWVn4qQiKOC+xL4VK/fllCRIAXWZy1VCi5rRoRMhAYzYrOeS4yMsl8VWRQXTsPDtol8cqU1C0oFGWVSXE4IId7OHeQTsJs+jzbRpdnfX1r37PHj/+SH3Ec5oVCz+R1JT8rGzQnGDth8QYwjCvFn46AJ54FhNgs1GXVYJYKuS+qhwCZ9OsSkMphI94WcSBqNk6YJ9gTfuTrAjZHqkwOwDjHEZzWuU+EUMlEIOaAdulmYZEOgiIQsmiBfdJE2ZVwmb4CjluEhM3FVNOxFALRUk85/4xic2PhMuyRVySdqiJJ06z9LNVrd1kD0aNGklceZaAtnLJEn/KE6WC+1I+evfPn4d+6W/6UVTwiPRwU5qxZ5jx8DORpc+fa/iC51lRis1ZHFUwrDXweRo9fw5hLfwyT7Iyiads6oOZc78oHSLhoErZrMgWUjqEjxVZVm5v0zdaJF+WczAzzUo+zbLjzdGWJ83Nu+e1LmWYL9nRYMDzWGQhLJHnW1ujianfr/IHDEphWY8PwGSRt7t5Q1LqebOqxH49j8ULYgDzU9AnKREEpUeLKPcLwZsBYl7zIpaixpf75ZwYU394ilfCssE+idMwOxWkeiJLOJvFZ9twNmFGJPKZD0cmyGCDfHTn7h0Wp+xpdsqLZ3OeJC6MPoZTCfz0+yUh42kA3rCP01j++hAVUG+yD3gawbpYlCzzOTT7gA9OCz8nvB8fPhjcZyJjeRGnpe1In7EBjzQjUUV+ESZcCO1L5+QlBa0N3SqLZa1V2OdQ0TwsOHaidMWW5MASxlZVzgb3LdjyBiuXOd9mcQSL50dwScUA24QHDScdXFCz74aLnwUcLtDewXA8hS+Xtuqy/WfyQXvZ3BdSnh/uHeyxsRSMDeHHCUTvgAII5YTbzhBShvEhNoA8EuYQVseL0t5yIa3CpumO0wgeovZyHcvglpKcyE2zn/rbbO/trTu0IrF58Pr+2AYh3IP/gmOH+aQBZ/ZjRIRUzEsOvdl96ryBJcFw5nGs6ulVbewxwfMR+DJx2dR4c9jgfahwWfM+npHNM3+7teKCw+pSlvDUnjodqOl6KL+Gygt+AvnRUjbiQsRthQMBaOTUMORUY5edkRXxtFrwApZv+y4gDOTwrkB0FE8w9Whrwm51qNF4PnPZsotn2sNT4zr6bAJ0izi1V32V/AznPqK1kNi0fxtM6ZfMpfrTaK+EtjdHTgt5wq+bpVezz9j3xmzpuHJWk/GszDS2XbMYRKuxWgoS74BcMf0jfYB/sgs+47CYgJM3g9nNl3kGDyJiIQekJsySzNe6AOcrQ7IML4wYStw9gnl9/es/O+8i2FdpIBMnOK3RcEsKQceTohxK562omoHIdn0GOMtyhiJPYvg3BQRqCKglai1UrXlAtap7WyCh/k7IEG05kdXhDAkdY27fQIBQMsSBaybdwrvjvDGX0OSgDCxMkZ2+KS9AKYhHEceug7QXh62wpRMI46CcNJ4XqTQsNh/u7hzueDtPnuwf7hzu7YIpM+tFi+JcJiRWRyTSA/MzoBS2syqcozqWcJIyzR5C00Jhy2lI50OP0rrVmNKVO8+HES9tKxZqa2XMC+ua1WSwHTevpKCVdBE1ngqZXUEKICwXUCYmrFgpIJ3CIcARDZgOBj3df7xz8OjwU+9w50f7T/Y/+rTvZjIkh2nFDdJAy9BHIpyG9osucEsHmGtts+qoOzJxe/B1edBAG++rsCqbVbCUUNTbMsZdNoBH7k8kgRB6BU5vXlsTgBnETOObLibUF8vqo2vYKClu3lboLZHAcXNh9a6w7h8irJuoz52OfROTW6/nhW1hUOvXGjMgCziSHpEC4npT2H0Eaf945/DR/pPXYAwvztUQkfetzEHjEV3RbzNbXCMRp8tpsgZB6kxk1DpRl53KGs4NTvqi+G4cpBLz9XEO5PwfMA5UvIxvEf+OfKNy/PXxDeS8Qb5F5I2vZxlAXsayIEEBEM+WNuGscxXlBFrO4Zf9jD3JUt4mK2hKpeGA3DOV2gl2KlsQ06VMSqhsZXUTRmRGnkKtphStHNliUciopUTVnk8NDSpe2AfIXxP/g72DQyrVgFf1NYoq4YOpT8vM/CSZ+sGxKqtVg+zusCGulm0mEPdPuJfwM8jhhXV58bfLi/9cXvwTbGhfLr/4rXr/6+UXv7y8uLj84neXF19cXvxXjf7m8uJLOfTl5cVf6tEInq+cyydQTL/oB8zikp7ArCyN6KnKrVoRUqqxu5T8SS7+D/n/V2qlP15e/F09/VsuqD7+Sw19JVf/kl7QqiNBWsQE+g3Ri5C/PKB6Sz5mp6leXLbIsKqoFvZI6sYp6UaHO9BrOSi1QOeCCRXatuOwn7TqNeij6WztWjSNBSmC3mdbq5ZjaaqsHvR7a6H14lbHbkhRZVJ/HOfMX0xRQGeVoJx6Tn2muQ/DqLgwQ5dcBE0dTk6CWhw2PYtOIad6Sv1qjibUlJV5h8t5XSF9HucKm0bh0LYAOza2i/ZKCPDOWM2xMv3GaBu+GGibMV0Mvh5qV9C+Hmr7TMDcIKa+MrCDz5voO5/ZeLiNfVDdqRUggE2sgqQtyGyEz3dQ1LYYb+l5akr74Xb9AbP54J0aQdN+RP0ksZepU8+kWkZu0jFJUr68VV5LY0DWVVCT1tYDLnvbzAut2chr+KBhZ6M+VEPvKnDzqT9H7Wt1gt5vFzrFd2NvxheBXjO8uomolb0UvJI2Cbsv2HV4GlV4KZ5W72o8RlDThwxeHcWvTAdqw5W94/H6Aq5bEhHEmnhvtfCgvH2hcAAnlHMatnBCQPMHVEcxoxg6133gOkO9LpF2zIxsXZ7omKnHuoTI0as1zX74tDY3khUqxajztopLcTyDrKkIZEui3b5tmZRS799y5SZM2doWUVl/xOPqtzyphMxmXCK7+50YhvEXRT8VwoAsmSaSwkJmO4A919MnhpgQfzyohPbdsu0yMSrV2sF0/H3v84rz733vENGrTtHpkEIgx1gEMiHrbcddzeE0zbqIlT0lvWe3wen0ptE2SPBNWqcBXckxVd1NepPIE9M85GZGktftqz3rBtPYPO1cDa10EHhKedC763Adc56rlG4hj6aQzuE88XSOozd1fmWeOYK0LKwCHq7DJNPITXTppwRpnFVd3x1oBK65Sy89ptRHSDVAYRbRBqTaALU9rsohugGPRlSzrN+VMKLCFvXMet6fxvo1/jrHr+au+vc188mtE/AaJ24OGz7ZRHFudIwbT3JErkK6j95aUN8myJBn1+pths5mjKJnLwZJJEGGWhGC7mAyclGNQOWh71E/8hvhb46VsUAt2O6GrwzpV/l/M4iq8GL4fRVUabQTC1tuAqJ9WQ1zaO/imMiT54y2Kq+UJ3fpUNqjYnNbHujIeIeTxUQnqs3BlbR+dWTTnl4a2t2MDVGF2dZOFJl+amUmzijpiU588qSse5DqHGpP/mAjq+HzAWTCdcOSTEnt5UhzVViTXl9SXIuEeCYbkevCkxmN+tFHRZt6sWO+NLuZMM9OnDQiidT3YBugUhVllJQ01CYCvfu2U2kQTVzZRDU33QXuOexpkqEyHmPKkalMExl5AgTT856ro70dBRPt6+R83TfQFiGnqcHGU7nkVZzVwHUFslPJ5N4UYrOGJYINS5o0qQrOQJE7UOkMtRrisIa0TNhIhe+49FGgSTq2R1tk1MZZF/YaZElW6AJEMgsdEGvjnYD+kcg3gnfuTd9Sj29NfxBMt+TjneD+7O7U6jSp/TMchw1paZuodjXfXFhAgebu2Mqs2tuP9dJj+b/TQTLqIKn59cpIFBW4J4N+fJnAKNfcjEAXphG8tTJvmcQLOreFe7y75iMtbQi5A5HwiAQ1Q0SVPL/ffh11iKI7JldQMbqOilGXCqDpzryKACgB9CKvV1fXUsyrWX5QoGsi04vAF4EftgUrZpZxNC+9xF/Cb9omRv8E5+mRrd0psts8Ho/ubOmqFNoYoB3DbUB1C77DghIO7aoXaBPZSL5OzAxStdSg1PTTOxqm4lzfrRjuFFFFAn5Kb4UNjyg7a9jX2NKXw9bfRmvK56Efhp5f47Gt9h4JlC3FuBhbtyh5VzcuxkSRu/ZuCe4+5GPrgAsKjyUdRuJoI8CFpEDHPfZoFzepakzbjArrtuzRyzpXU6bupxjE0P2CprH98OHB3kM6JkMEp+sPYxngahXBPshHDCXbCKmQTG/8CI0MFf7h4jiMC1tdaxBjEpfLZEvXy47la40Ul0vCGJdluFAiEcPmCg50zKiGclzb4AvV9ajPrboHe8Moyaa2dUs1lJ3O6ZfZJL62Q7zuSK6ueLT+CXlnBqFrTR3sGCFFIMjy0G436DQoZFzvZHGWwNW5he/V1/uonoXlmodDFixT3gSAOCmDwv6R7QRpduphdbvbUfAa/duuj4Al1R0gA6T+3EGB21L0qXetwDKuHEK2r3Lh8F1m9XD08wam7gUuY56EdFXylOlUlE2RaSM3krUHFVCyIhkaCI3+BP1oV6KVqb6NtYntrr2IVmMCM09RfHCPxgx/pETVKPgG+1BfAkVujIiCq3B4adInrRzUaKufya1LnTJSBTPqRTIh06BrMoqmRtf5QS9pMCzkpah6CdnLUUa+RCqrgWhteoKWdb72AyULZv1Pp7Uz60gWnfcmyme9j4O023Qo1IRWteAL/JxbDdvjtcm4S1J78OghzjMP1gsYNwc7vF4hQqLbZq90B7FtlfaQoBDHbZIXWmnOrW6k2lJhCnvwPCrfcOWPGmyeR0HL8yzjjhv8Y2nLWOY4/wMJI4C2", "notebook/11_stage_8_robustness_gates.py": "eNqtWutu28oR/m/A77BY/zCZyLSUk8SBGgUwah3HaJAYstEDVBXolbiSeUyR7O7SjuCoOL/6AG3foe/RR8mTdGaXy5tEX06qH6a5l28uOzM7O0tK6e7OhWILTt6R77/9m4ySaSZVzKUkC6a43N0ZtP52d3Z3zsOUR2HMyTQLo4CkEYs7xAI6J2xFer0O+XuPXAsyS5Zpprjr4cQRZ4Ek6poTJhSfs5mSJBVJkM14QKYrgyFJ9/tv/zoiLA4IX4YK2Lm6EjxNhJKHomDV16x6y+DqityF6powknJxgK0kZVIezlkYEamYyqTGArK7O1kccBGtwnhB4mw55UJ65BL40dOiUKp8qODyOomAWeCfA4pg4eJakblIlsjO8enw8+WFIY4Trq7Oz86Hn84+D/3R8GJ4PPrjR9P53//8ZCTPYj1ZC4/CEJEkqt/HPgK/dKWuk5jEieLTJLk57PV8icrw3/kbIqcrMj44SDIFiiW5Yia7OxSXdXdHU/H9eaYywX2fhEscAFwCNlNhEkscZVvFImVC8qLhV5nExYtcyRwvZeo6CqcW7BxeEeXjcDQkA/3qAMkwAoKuB7pLolvuuB5g81jt7gCQhxBeGEsulNPtoEodnO66BTkg4KfWtJgkUUrIHqjkb6xPhq+7r5Di7s4eOcW1qiyRE4VzBQakOS1WRq/LxqrggqAtXn6Elo9fPp1cAP/3Zg1o7PM0lEnApb8MY9on7zpFD5jpQrDlsuh7XfYJvoR18xcZlyrvfWV7g0xorft3YRwkd77kM+h2fuqSF+Qt6OGt+ce14++48BUsCwck9hWGdr1X3QJss7NX8ChB1eES/vjzXjFKM9P1jgqIBajMV4nPpABzSjfgioExz8DqI38WgTP580RMwyDgCHcpMg6j1mY9Aj4n/izJgG5VD47gC3AnsQL7vZN97VvjIJypiUsOPpAwVn1DSHAw1JjIbOn0CJAhAjpJbbYZWPmFc+IID/h26DWTNbq0Qyh1CeDAw4sS0KfjIuI97WGfAubxueKSrt2KCFHCAj8RfpzE3EFj7Wu7dnM2gSQ4kHYEj38F3qRjuypSfIbJphF43+xG7/KQkNQUwFOApuJflcPjWRJAVBrQTM0P3lF0C5zIv854qshQP8CM2khaMfgtizKGysBQ4WhdqyyN+LhcgA7RjxwKwsZII4Ej4SSt8Q6ux5KB/jGAStfTwcVQNQsDbhOlWhbfNjmW5yhchNOImyH2rfAtx7VBb4+chLcQgkO1ItqC8pUufQ2dM1/oojFf4cJUCnKwRBtj3bWBtE4KgNuM1WJUWDuHzSRn2W4VpgssytrzHORXE8AcT0xf8EAfMszTKseVxeSpHwYwmKdjmlOFBjopRwQhE8h9zU5BvyfHl8f+ydnx6Owvx5dnXz6TQzKn9xpv7aHBGW+4X5dQsS9Tzm6AV1wiHjv3ckzzJk1V8yqRVaRqdCr5AoOLBPWPJ+7aLeHQNQrEft1ZUSEeS1MeBw4EI6fnQczret035CU+ehD8IPTghlBh6oC8ct0KAYhVrZIfX4yeIPEeOZuTLAUr5Qz2YPA4Ypy4QzLJ89zlaF/qFGKWxEGIzkZ+GY7+QOYsisiUzW6qcCqBfOOaZwJAwhla7NcVubvmscUiEJjifUWmHNoE7P4rrrwSAcM8hgDjI0aS09PR8PT4cngCAlEuBIgKuClb6A3ESFbTugXZFpCMr+p9dVCNO8Wcx2OP/RnD9adRMrsxsRlhjVVYpzZW0d+I1MimnVubgXbmksHAWP6WifgzJAcNhGJ9kOr92jWtuKGlUQZJEhemfTvmXcUitY86Feg7PRcM03Vbpk9BbxVLAFeeo302fKSpiT3y0dpKnzCIeWgjd4kI0NjmIeyrM26MqD4Pe9FJx7IQPR+t2Xzjlp4KTIxLBiaTOlBVagHxL0BPBEdzcN/VVFywOnRFjAemoUN6Lvx57RZRkd0ufEAChgwGTkbk6lz9jlNxZm6lEvQEbgZ6LXGCBk7QwAmaOEEdx8bpC5v29E0CaPzvjd4JzZiITdvDR83pYCSPIvAQnLwllEBilSa4HsZwYLhZFtMONnnLrfkUE2K+2DoB2tnmhCVnMeRvhWacnOTLHAo19MpqxHYCh7Z3Uz+nkPB9/8c/IU6+PB2O4KCTVvV0lG/Z6dETVdQSl2pqQhqDSi6kY8BRI2BUXWQeCqm0nx+NyxGTcbdixTpxZbMZKhOHPxQLcGz1vUiNawFjNoPcfLai9Z3mqTQ2480zaYFWCplCqbNLVJk+t1g+Ku2NiGJ0bIykgDmwE9FCrAnoTBBjSAlwX8eiOAQS+yI26wyJdhqjMLXEYeidRcLUHFScy2DgnH6A/K08ao336yes/cl6g4g5sCMz58cXFxS1VKNHALKC2DyzTYwH0J+Pzz5Vsdedx6UvssYnyF+mqM/WQP0k+QwdVGhuaqFxPv0RPdSOsw+rwqbVz9VB88T8RC0UWfyG+BtH8B9RQOPM3iq9s5kiQCKKaS4cb/PkQoz3Czg4++9P3C1HF4zqb7t9rzdfQ+mLboeFffH3wxLUyfbszB6fjL7iQ9YY94iB11Z2S7VjH8N4zki3Tb6qoI/j9Sp4WrBW09myQqU1QV7vbGelalvbCjhapPd2VxeVQbq3JXV8P3gCcG+yffLG4m4Oc6sm376GT/GAsgrVavx5LviwcbxveH69uvVEr7dZZ0N/jUrZDzn808QNnitu8HvFDTbFDf5/4lYrha0C55no8yJ7Sw3yiZLb5LcR3dsKmz+igkYZtFULmGZhooaPSo5WxsrnGUR79fVBFT0Yxpq8Yf6IbQ37eaDwuyXkaAEdo9p2BVz86ez8fHhC3R+LNnHi16rNratB4eQczkOo9TvTMIbyJEmTiAmoH7oPLkRe9X3cAmtJRUsB/DGzm9jEOy+hlhcMjUuGB1LpWkL3YMpJ8+i4NSLTPJZsjV9Uu1stFmzz+g1X0U7R4BUXrV+p4S9wpzIHDzSesdX1BCs+Rt0NcfG27hkQWvkNCHkTQoEjeAZKYb850Lp2JaGnFZXwssYOd1oYjKHxBvbtWFfNazccxZy+rrTrIjyUHu2FB9TX8ThGv5mbrG/kz2je8Ly0Jgv/X5iLy28bVvvtQP/yR/Gs/f+NVmrOpQqqlwfAhK0HNap9wNf9AkIVzICgRMyb9sDytXAu0/TiBTYa1ULLixfAd4nqlnJbkpRua9yDao5RHH1oDgp10yG3KFeuaC9UfNm4kalMndMD5PEGOOuT+9s1fTJLH/QawQWy4MXdSn7TWF5h60tlchUwxQ7z+2B7X33ltcD+grViFpdFaQuHYXYZSolX1MreTEObQeRYGyBXueG2oBeNxo7pX2Pq/ZrAsQRHVe68lgzaYBu4zY0XOJnAauoAb+o3jbs6LDnYK2PvWCwyDBzn+CagXidnItSXVANqPwYQjS8LLGepxwIoWeQIDrXX2VAzAcZYFqkBXhBDCWo0PP8yuryAG44RVALVKuUDfS2XFxLFAn0p9TRLCCi1QGXFElo8g+0tb4JQOOZOWg70Laa5BfCTG/3qVgomnUr4bl6q5aU6vK/ZGguKuflIoG+L/RV+sJq25XMGWp/j3cH2xk2hfhkAw81ivb0y02dDOtZfDbybkDuB5+R7C7NuHXif86pLExDH0aPxCexVuzA+Y5f+qgI4rwYNOLRVRuZhGAfbiFy3x66xQIjHvh+zJX6kgMHY99EefZ/mxoYfDcDygNhopq77P52f/5g=", "notebook/12_stage_9_report.py": "eNrtWs1u3Ep23gvQOxTaCMzWNNktW/KPrmVAtvvqGpFtjaTAC4/Briaru2mxSV5WUXKPr4NZBcgqQJJVNskmuECeYJDlPMA8hF8geYR851SRzW617JtBMMkigmGJ5KlTVef3O6eq0+lsb50bOVXisfjyu38UJ3IsSlXkpRG/Etd5ealneeEXslCliEs5MUk23d46/G/9bG9tb50mhUqTTIlxlaSxKFKZ9UQ9sfdCLsTubk/89T0xK7sBDThTMtbCzJSQpVETGRktijKPq0jFYrywY7UYfPndPzwSMouFmicgMde5eCXLyzi/zsQkSZU+IG4CP3Zbup/KcWj/DuYxfeCNJ5lRZSZTEWOdV6qU41SJSV7yEp6fD8Xe7u49gbGrzFgyIUvGcSNma4LTlypVJs+E93I4HJpSZn6iZ11a2LPczLBDJYyaQygGm9M5zbngt0rqhTC5yLGi6zIxkIYWmbpuCQWSjK3EKuy4zOe8YlqfKPPcHDT7LxZmhjVkuVHjPL/s794LNQkxfFyLo1iId76fV6aoTL3D99tbHTKS7S3mHYaTylSlCkORzNlMZAaO0iR5pomqfltOC1lq1bz4oPOsedAL7fgV0szSZFwzO8UjcflheDYUh/zoYUroMQy7Qal0nl4prxuAt8rM9hYYBcQiSDKtSuMNekKb0qPh3W4zHSYIi9oAIcC0EOIOBPGjPBDDvcE9mnF7K1YTEaa5jMO8DLM8Ux5xPuBFdA+sDJMJyY9XHaiPiTbaqz9Zq4BsMvEag+1LUy5ufiZRBDSR5hmwLcxp1EfjqSzKY3jYYacyE/9Rh/ZAA9XHSBVGDPkXJH3blM02YmVUZMKyysJ5HisPW58msOgDyEKbd3ESmfdd4T8laTlm0PLzVGqdTGBxs0QLDCZhnb9685dD4elFBrsySSSmORzYJKrsCrjH6cvT4cnL18OgNrMLWB/sy4CzLFrWAaFpLdiItRiNEh0So5AYHV6UlRqNBIik0HOZppbVck5VJBr7EJ5ZFEkEggVYqGIw2MWw6wQ+9ETsD0Rl4MQyi5TuOjeyjCgaYD1KzsVU0vwzeaVAMVfYaTYlD1NXMq3wLRBHzZ5YBJBEo87rmSJXZf+aQpDlQsywCRunEBuEgd0rI/KJeHooHola7PX6tWUjozKHLECyRzFtWsr5HIuiyCGzhV3SWKX5NfhKg/9g+DQPrYUElF8q0XcigveP849WWYiCte/D6ueVhqchskISaV7FkBkzwwbyolHXGdsP7xHLPoBYO6zxzmjUo4daFuvP/smbt/5rvA0a87F/ZPDbVGWNyTkTzsLWTi3FpzKAsLxO86HTA5suR90SAbkR3+fu0vvE4aEY3LR/t+gW3RPIH4xW5n0i9n7Z0N0HG+jWtl4ngtWPnZYX1u4XjmU8VR79eUAed9PzzlQWK5trxqToMfyGn6GxaY6VLzVHxjUarSQxKKERPzZA85CYlmvasJmvkLv93RzkLd/wup+KL//0L//x+78TYmdndfDODltzlJdFpclsa3NWEK1fe7P1lu9EZ43vDAGRY3VWzceq1JwKtUnSVNRh3IdKY0ITMPFZXkFkFJfHSvxY5ZRE11kikE0SSu9wpSpF2qwy8OMFuUUiOkQzSFonv1XBb7LfZC0W3RVdt8QAEfznP//8b9i/DZNnf/WaN74hWsLD0kWws7PCGOMvONZal73GMqcKmmcgwAlSbg6mrUAo25vtaAQPBJwbobMnVDANhAua3YDjdCNoGUVVKaOF6K8wezs8w5sX+H+STJH1oYkxoAjrY4fjkI9oinjHkEHvcAja2eHIg6XCCtrcoJ0ooY01ymiSstNKIAjEkFImFZSNrYB7kzvavLzROpwZ7Do4MwhlFSd5mGRXwAh5uQCucUlikVelZd3mNYqlkVoBzY1EnJRInBjUo2Vk4sY05NMQO/Hswt4Qq9QKMwRwUVTjFMqhlTs7tfrFftqW1W2FCk5LocbcGOStBYgpAuYqMkmL4Gx4+ubs4jx88ZJ01AmCfpmPIfhMaW3ZITJ0AC4AKyPlgcCG1yAF46TwOv0GXtxhYyAs4vTLeQbrUpkGFsbm/bz4DpuAteVWP5HEOsgAkVrsTl0emDIbBPgbK9y0vCYITb8BqabfQkqr0TjcMBlHiKV3LZCnyVUpc9pC5BGsstQmCFsxvOTAHC7D7bpuGiDAW2Yl1a+8Gr3VKIBJ6qewxgQ1mY3Dt2M3R0YwY4M5vDi6OAqPjo/PhsdHF8MXJHBQqjTF6gnbBIQ6OwzZPn126yrLX8gJlCBAIi3k1Cr7Jrc4ufoFZhpTbaUTs2AnpeRVqw5fWLVg0kpOxBXWSPC2ZRREe6udbZhiOdKyDG+ztrWFNKTfNDyV6vUFOh6M9uNqXmgPb7s17roFJk0R5g9bBQTWC+FbnFQbTKe9YjZZDAHVuyXF+3eD90uaaYj4DhqmfdeJ8gxyoYKt895ypiSF6PDpc9c+awTOZI7/2i/rHFFvmZHstzhLDbtJK3ihKv+kGSg28CSwAnJgEg2nGbn+dk2NVpJlXmWxZzn4dkxP7DVaQFTA4gGe7ixrfGqBnNl8TPHhmURCleJ7JKwsShDwLmR66Z/PAGfO602Iur1BfD9twn3dz+FxHXoOxCdYbViZKMuvQ2jMw1caeeeO2A3Ec8Yj9OyLoVOoCzOoPjB6Z+cTWU4dZrqfd3aYtg4zHmfLGKLk3FlW6IJ0eVjmSE+XkBhRDckXe2hxdsD8boOc7/bE3bsbgblj+KL2OcE+B5SrlBjVTZKbHjkK3H7vBY3sxFEZzRKKfUAa9Pm51BGSZ9xjFSMQ+yb38QsbBJwaxXmk+zXsDM+G58Ojs+c/EHPxh5/vd7nvMRqNtrcYDogvf/P3cEtZJr+1YqHno/Mz/n08tL9bAIseGyO1H6dTiJy/OsbbW0NgRsGwQ9iKOeO+UYKkUVDmyEzK9RcHUNEKoJRACa7O0eHBduPtLS5+J8lHpKYGM9ZSuh+IH2qodsZQyZrHiYvwS/zm0vNMpTE1crAFY9Uqx1anNSWpNOvLu7UGG1bf70K+ObRFmb+PZhMt+Ep1V9lMdsOaaMkJ5CsU9dj1uY5pf07+vyLZN8snp/VsNn5oZ8QbHrW9dYTUh3qhZCLIDUXrnNDXUksNG+gizREWATSrwhX+2jYMJdmWrsbAlaYiHMDAEDJLUCvmCdptr04EmyuiGAcA0wbKwK0U/akI4+4CVPoBFlsrag8ItkEe4piQhw0Ja/jus6PfR7uh7uTRq58cFPmpeS1+wlvf/tS/6U+iHYCOwWu/lNf9Oh4Ekb4aCUswbiiwULRFtIr7T1yWeBpcy4YwIsKv+Ksl223YtTypxbA0Zk5tgq+QUDp03O413JAlNpPcb0ig+M0ke0su1ntXttgi3F8SOk8G5U2M5KgfbKKOEx2hhNCKsBB3l9tDHm4ashk6uRGP2lLfAFkd2eM22Wrl31t+WG1F89CmK+Bw8Ty+gWxbo9ah7f9VpPn/GOl/CyP9upJY0mRBEXDIKe20ldISymB10r6Bm/xv4aadnaMKtXapD9A3CJ9kEvDkacjvJ/C1hCex35BdcQBCHCzBW3feYr+6Jiyq/0o9rYHV0Zh6KBEnzvNCyUtV+tKgEh5zFpg08K7xcVsFz5LpzKfesG02VeQ0Ph1EVGVEafv1ySklWartGR/ZbQfiLQpqC6zABAiwQvLqIes76TSdD+4W2fXAL9fwSZpnUx9c5w1SaTrcfqquVNqGKNw9jtEAszqy+UnOoVxuSG9Ik5wjr6nNQb2vtVRJ22/lye0tTpR0xmbT5BxDqBGXKllmtPuIZlIlaSYQbzJBvRaLZKhFxC0wXhIZ3ViZa4V56btPB2I6Qk/CtBeJrdTwYPkWDTLSt0MFtonVyBROnUwR5CBL25Fv4y0LtAid9bDlBNuVgHEtUEkoPcXJHfUS87GGiFSwBOUvM8OHjxb9bW8tywEDs6Ye5LXmNRd5jAVBSvAG5wLz5CNJq8whojltQ1J7SXOfCRpCI2yhE92jrwXaR06zTcuoR+Af5gNqASmZ5McKe6C55jjohKFTysFZFwsfgd6GzZaTLbORQKz3nbX1+MH5Cpn69lalFZpjgldkbHNHLqA2sms3F82SAv2jDLa2S07BZ7pkAgncq1Q/Vgm1Cumsjg9XoAlaC6BaY2FOZ67hsqwFzpQ9AyWHthC3WHBaV4FF8W0Pwb9nKpvKNOGVvDgR50WJBQKFDVxXNyAWVg3PhmcXLLXlo8+AUniRVuNKmSwtuszpFWDEHBGz72hJlMzpLXq1VHv0xURSNeZfuxcT8gRTIXm3goDwRucy+5DEs8HjQd+R+nMVJ9XcH2ejLvN0ZSGhbRtSUf+VFidabHpy8spyRQe9iTzkG3ZzR8/Oj5qVtuLYWmTA3JHxx+hzxiteZo2PIj28xeRRnupW1dGOz1zyfKuC+p8rmfjs47a6aXvLhbnE3gag0zGyNbKesob6tpgilV/B4+gzQlVqW/FKtyD7KxwD5nGe5tOFfUlvd1HUNnsF0F+1w/v4brWTFIl/PKNVIpr6DmbzLQnvj78P5woWdng/ePi4J/CIuHU4CHYf8oNWUzw83LcPxeXh42DQDZYruEfSxTn5qqnxrF83LBRwZQKHXXRp95A8n6ni9QS/xjK67CLYzKrsksRjAIznSVaZRiQ09/2v2eX2FmzSj0t8yw5AiOEJguS++B4HyLPV2cWvr1V2L9j3B8H+M/8lHcNWlIW9fDKxFVW9ptbse63oVRemuCuxdFyyY7jX0nH7Y/6I0yKDvnzjjq68wyEK/IJOa1HHYe4N9S01GfBQIRmlXdGANHBwJbWLlDCt1kL3xdGyZXDqHIhu1JDk2Dh3B+IvxKya484Hiiq8s91yHMzivgdmjStAI1e55E67f/x38eVv/1UMgkcD5If4AyzOjSIDsB7/h58xe7AXLAtJ2xjQt3UGPJuLm1xLJkCnjKkPxPMhp7hJidri5j9fx4Bm+9OaBhsADdxrrWkA0TwI4MnI51q73D1cS/rUBJvTsQffNhoDahq+dLPjmjM7Nm1yyOKAxGV/U6CINwA6DQiRBb5RiMONH3QfNJ/Aq49Anghrzf0BqLXVqxgrjl2ExjhW03EYQhVhlLoLADRB0ItuNZVApE3OfBggAuHm0/ICjg/spZojVne4WR8fQ16EBekWBqb8brMJkEpgxqksNAGpnI0yyXxHTBUiW3dz7uOjtqh9R0R0iUR4R8fD1xfndOzSt858/ObkBYbm0aXvsMT3uwh/e49x9SKJCU/YfPjMemqRIzmTIjQu8aTkS5pvQbUynEt5VqgzFU8Zgmi7/Pmcl8ZCekRN1SxCieUs4KjRV6/G5qvo8WbeWeJHizphYttbt8HYzeA10au9JHvo2SMISAVBVpAGtEOnN8G4vRxgDDq0Krqs94ZONQPTxNqttYDnfNNg1BxajrhHC5h44BoU/OIVGz9eWS/ou0PS+hEhK5nAHGzDgtvGuGsG8qCoLqs+XeFD08EBhJAxYEgvw/0HD/buDwbjgJ6oFfH1RsRcJpmH4HrlLidh3+/R8OAi1pb53JeAbFz1SjVrfb8sOCqnFUn3lJ5KD1pkFUBlh536fmHavtTYustY19BFIOM4lI6T16nvwKHuxgIllHRIt8pWj5qQQ3EGrw75epi711NO6W5LEfDSiKHmjTWlNL0JLO9gfomzZs9eZNN8AwpH9HQiFeaX/Lg873P9HrvtmoHruyx7QU7Cdn83SFe7Q5113gFfzLJHXBvOPLG0zedezHbD4JW20u2jCa57k847NvLH7xGYYLKcD9zUn79FyhN9Xjv/HVjrQl8kDKmPgNuKdMElDMnWwrC+2kK3ByFy47EJdrv/BcYp+EQ=", "notebook/lib_pipeline.py": "eNq9Wety28YV/q8ZvcMW+QM4JGzl4iYcK1NaohUmEqkhmaau4oGXwIJYC8BidgHJjMczfYg+Q9V736FPoumT9NvFlZDcJjNV+QPE7jl79pyz57qwLGt/bxlRyQIS87Wkckv+9bvfk4xnLOYp87gYEOVHLKEDktE8UgNS5DzmOWfK3d/b31tFXJFEBEXMiM/SXNKYK6ZIHjES0JySmG5FkQ/IN8v5rCIFIjQNiEpoHLfk9vcKBTbWW8KuGPhQOd0wrJA8y13yLctyErCMpQFL/e0wlIwRW6SxRgzAu0OUIDwn10JegpZICSVHIqbrEhU7SBJKkYCrLREpA/eWln5/z8x6XljkhWSeR3iSCZmDxVTkNOciVRqrmvXVVfMeURVh53r4Rom0fheqfpNsr9o2p35MlVZOBWqmBiTkLA6gFhVwP69Y0voG+Rr7HMMKkm8znm5qwDjdDsg0Z5KuY6ZZXUzO595iPl+RQ7PKFspl6RWXInU3LLf3CH7W+fR8cjqdTbzzxfybydHKrLAG5Qqog8dQhuNKpkR8xWzHzWAmaV797Tkd0N7xeDVeTlZL73i6wKYtA4+JpaVULFdG1x+ReZFnBdQiRA696oUfWmRV4MX4e4AbTEAlva6BYP5oslxOjnsomRQ+g7IDq9lkvJj+dryawgx3UQNOJf/RnHSNPF4uekhUyRp4MukDN6wBjmez+Wq8usNPZU0tP+OTk8Xk5D7MzUayTYWqlbK4X6+S6dPXan0xPekg1OhACfkGJq1RzubHk9N7ycB3WVweTigkCQhPycW+MZFa+wOyq+oB6Wu0moHaBp2lUFQNqLVSjxvhK/wO4wNSyTMgLddAezUqUQM3uQy4tEs7VIcrWbABYW+5yj1xaYaOlkYb2/B/9yMfaYKTjCsoDF69wX5y+wCbQDHT5Wrx0jta/voDDtz6bhfZGjQnVtpHyaKLkGU5zl6H8ovp5PR4CeLVQVusFMvjAYhYfgRrZbF+DQppHMNTzNdjuNVG0iRhVnVwFoKgJ1kicuZtCqZyQ0AggisvofKS5V7KrpWejYVPY09HtWaxiugnnz81QI7soZhH8xxESmfEtKJJhjgEJliHM9UQ8MG1YWzNc4PlXa4zs5uJX4r/yLz1FiR3hAljumlISPaG+WZaMqqqbQ1fHqTgiMxGKXBfZvZ9VRpXwEISCxp4tZ5thwy/QhZV+YUO4rW58pBgKekelGuMVdlOhaJ/kiJtkhdgeibyF6JIg4mUQtothv6F1rsuofck4UrpVKCTtixSvRVbC3H5+MmBZxKo98SjRcCFx9MruIuAPWRbpBuJk2ppO+XrNc+jXU4FEq6NA9TFwKEFPSD5igA7HlpFHg6/sBzkLBJ2BWFIo6nRgw3Lc4+hiwWjAZN26Dit7hS9Yq3upLhWo47yjC5nSNOj/8iZdQ2WfhZ715IjU8L0a96+NxN2WOXglCZMHfY8xemvd81fVIrVgeoIKnUENfLsnl13JcD2u8sRkcadL2Fe4FQvvtSLe7u/72gNVdmGI817lceqD1hddQoXsmWpb6wwTHnR84lX5BeHtUfA7F89eBwtC0JiJzTHmyJZTFOC/IqcrOOM8wCbr8aLk8nKW41/M5/Nz17CFCons87Gi28nugSylqiH5gv9djQ/Ox/PXupXHMt3p+Nq/mx8tJh709nx9Kiemq++nix0hDDntZifTu7bJBJljMwoApk+Nz3oB1DtsjBlVgYeRN+MMT/SoyK9TMV12mxzPj9FFl693NnKyoRCTX1VrtfFhH53CPmIrHmKIh9OI+A3hS7XiT0+mcxWSzcJKkP7VVOY7u+ZP7JkmwTRozIuVY6QLUYovmU52aaQziRCkMx17hiREPaXV6hpcGcOEtJLJs1ynlaTOXub6yC8hgCJIQvpLKsDTIVEF4EoH/TBvkhDrnsFVu0D2BP3SRVOhAx24g2Axv1tOBkt4twLqa+D5aFGqRxcCvgdYmqzU30WH9Lad7kuyjUHJYGiHveU9JPVeVcfH1ZFBaESEabhuDTREpShOUIs2raK0/aBhglmyUwZuDQd2GclelXAClmz1NF17/AAGR60MasVOxee1rZdjFrdmPilZy9Ac6CbmV4MK3siu+jmjkpjNT0cZWWhP5WaergScTonEYszFEAPQF5LbzKIp7tNWxdTI1MiDohYvxlpgfupU+PUvdtPqZybdGsWtmn2v6ZWzZEbFElmgxWkU71E6ZaaKp/zwxc0VtiO6wY+P/ykc5you4K+PEYKSDO6nyH5M0oRw5fOfihAOjZUlngiNJ3ujib9CJ5dW/IBefaMHDw1/MCmKvIRIFX775aU7A+oTq7vMqZz8hrlsMn2pvyIabIOKCKVq5VhGw6cAVmjLuiVEZFbZIg0zDYEnB3jjtyIvQ34BlnEfjgDPzMdI0loykPsRGx0ohI3QD5f66ucrfNAdi+Zj7DtmYbV1nWaCUMDEoVeIXk18EWSoBXQZ9NGqYGpi1U70fcR3ASNM323hEsjQxrXXJIPzBE7upgjuSCvX5e98uNacldb1uvXrlXHwUYlh53W1TTZ3RVWnU6u+yno4lXTMzQr7usVsBLYjV0ru8E2rqTTgX3HO5x2W5caYe13LU1LS22NiBG+M10qF4DypQvqqBrwzqiLVLZNo/IAugAaBCxAuweYV+Q+EqmHjGc7Fc772p3aYFcLOTAydFx5Z/2un9ZXbwwJk3Xu3cx4QPTzR5jBjhfVYBc07RrDxR6Oix1CnWYf0r0Wq9VZ1Sw8aAKReZ7shL1eyTGo0yystL5dvKjS7Kt7HMg0UvCfGXqXUoirA/dToqOrCXgBSiWQZTQhx6jwFZwZ5mkukTWFF0avI3jZ8nwy/hYYz/TSr8gzP0pj/JlaEv/oWPDEOUbCzG4zjaTtVuOi6vvq9euGqf9zCtRyQms6rDfK2w3fdb8F/03oWxsVqdGzW9XEZGhGTd3s7K7WfW7bS+zeDNRqe9ee43tkL+s+3Hc7m4zcT8P35F3NWjm8d92z2RhK1g+VXZZEmtLvvQH0lvX4D8v21zZyfEysH1Lr4RzpOU03MUX4neCfq4jUFXJ5AHZV3upOAX2Qj7iCsBjjukcHe30DHEga5uoh0tnz8ezkVN+gnkx1J/r8dIJThXO4CZQJZlJlW7c3f7i9ubm9+ePtzZ9ub/58e/OX25u/3t787fbm77c3/9Cd3ZODTz797POnv/ziS6sTDZs+wOQBzwhh7zQNvRgJNznlmyi/ZvrZrpd9fQhzda8alx3iAwtulXAJXesatQfHyOAfTafV2G3R45hm+gvIdQQzUBn1mb62ajGWOT73KBIjjcHPHkMVKClwv9XiN7ufg8GmczUflJ6zdAPOh6dCZOSfN5+7B+Yz1KNHUYHc8ehR28MQbr5QVZ1NgVCiOyF8FiJpkSDUxWrI02Ep1NA0i2YDfds4VIgMuKsIarNyif4GViVuRcwnKWiCIWQmPEVg4X553zbMdFNozsPtBajqkrA5pTs1bNNpwU5qLCR7XJf4zLZ+KJ58yZ6aZ2CezDzDvpHoBg8mgrIc9wEtQeUak4t1VXnXMp0WUTJXFWtbWj+ojzVtgodyEEhwaPZuKar2/g22dxQU", "notebook/run_all.py": "eNqtWNty27gZvteM3gHlXpRMJVl27MRRV+m4s06ambTxxMnsheOhIRKisKIAlgBta7Oe6VUfoNMn3Cfp9wOgjnazMy0vbBLA//3nAxRFUbdzrvK+1X2hclbJSpRSCabrbCaMrbnVdbcz/sbT7XQ7Hxtl2KXlhTBsyH7957/YKyYVgHJR99idtDNmLN7rA/zTjWUZr2xTCzAVNbZAiPNWdzs3N7WodG3NQd2otNRF+r3bfj2w9/bmhhnNOJtyWYI20BmWi0lTFHxSCscLDLqdWvSBoKQqmJ1hfaax26o4YJ+NYDc3/f601gvAHrgPq/FqNQMhuJhSZmJA6n1aVjLjJWsMGI5GtMTwVEvwUkxpKyZaz53EvCwH1ZI9/nzHpk1ZrqT4JooXj50wEg0mbVFyfafgH8EXTKty2e1E5Mpux51O02lDtk1TJhdkSsYVwLmVWhk61a7WRcVrI1YLPxmtVh+mmVS1zoQx66Xl+t3KhQj8Km5npZy0zC7wSVy+Yz9KBUENy8CXjD+V9yN2oe9EfTkTMEMuprwprSGLZ9Xh0ckRQmUmsxmiQ/3eMqEynQtC+rNQBS8lK8plNTMIKwhfE7RYSGsRCZMlAmMhWDaTZYgLM2A/1rxiPuAOfPgRGOLy86c3/VN6och4BisIZZ+xoC/E0RRmN1WNkIxpdeBB8K+WVZwkN2wiSn1HaLkWhoTNam5mTNwKxeSUgsfFZin43ODrs5KkixcbIWXr5ch7H0Zt0WsBS01lAd/FTnfE7jhq7LR/GiXkdrusxIjJQulaXHFr6z5MiDjKr7ewoOf/hiXuM1FZFp9hWU4aK84hNvL4w6V7SYLoFTcunv5y/vGcjZ3jYwQfUjNNE0gAp9+KOBl4+6JInF98wDk6vlp7/+Ft+sO7j1h2uwcsCukf+RiKgx05DN5jJoP9bY/yU21mO3t78fng7OLdn5Ju5/LT2dvzyxErpbFXtqlKcQW3gZb+IMHK62twu/IqxNEw6jEWDQ9TxygdprzJpU6lgidR/pZIQzrBPtWNSHorqgkWo+FRSzUJZLNlIQXKSyDapcoc1fOWKktzeStqI+3S0dtAt0116CU8DlSHIOK1/Nnl84rRHq8jT3USqI5SburN0+3zhpcGZBQSSojcoLYaK+o+EtFQbT4g27agzz3oiwD6PC3EbwPlir1//1cGF7G5WLZwxx7uZYA7TteFagt1W7MTT3UaqKAhPIVypOy2KHtCWOprUAiNTZmprhcwPYtRTKYotxOezZOWxQvP4lVg8SLlRVGL4hsmf+moDoeB6mUqbnnZfEudU0/VBuBpWutJY6xCJUrBEXXssaB45anaAHyV+rTZc0ZLdU3p1O0gx1lKDUYr4VNr5BPDZ5b7SFj/tUuUNs1dGZxGX9RVaE1IoPHYt3v21aE8jPDiIB5oLwqWtEOkGvlmQH/ipAXUGdbXHWYA3Njv0XNFdUzci6yxlOQudWNXZA6CnMl1b308u8vHdIIKCNwdJosUZaFq7JgM0GNW3IfXNd1uWewxQdXNjKkElTwTUTgcxM7ttjKsD/38FsYU7LW17IBNo3Z48e4JRqIRJlpRhBo4WMxzWcf+wwR5xT2qV6rn7jNZk9zV0oqUtNmw1zTacweLW3ck7Nd//Jt9ze1ocDh9MO4L6JZ9dY2tFjCW67POcV9UtMb9A4v6/X5ooBhB+tjF4kZDpCNPkMCW+yS06En+mx+2rP50E/b7aLY7irDfjdlwtMZv43cdvW/O3r0//wE22jWBN5YRsCOs/dCGMT3+lC8q23mxxtVzmilWtm7JAym5cp2FCy5VjBHsNnQqaHXNfmF/Q2IilOify0OwaNMQy+3INjiri4Yq3gV91XEuvLdRacbRxWNjfCtLNeB5nvJAHkd+wkTsA8KOI/rwMeuW3Hw2Ro98ktrqFa3V+5SvnqY0c1n1i6rBcZ55yQ0ERXTDTpvxsfFgbKzG0SUow4yHAY5bV94xZKFRIfdCj2ExpsnnPXaStCKAuYENq4EzIclinAOSdpxHS9gdNFzjSDO9QCmhGwoL08VaOhr2gDRYW44meToaRwiKqJeg+eUelyJz9zBtftlXljBiT/T6cSK/+X3YbI2fjLaxMAVaqRqxLzDZP4X9HRix29aVVl2w7wD6oW8DbJ9sC36XfC9p6GBFtzSv2lenFxWwLeDxQ7KZjU+qRoqsGtyWM3ctE7LysPU+bhlaW0qXisb3pcLdwMqMFbrEDVliFFpPJ8TJ19tjqj55kwmyYQu0R4JdUUmDEoPLi2K5NHMW091jA9KrnzU19YFy2WLV4u+NxCzNZs0CU5RUsMYfqT0HUaXxlw1c4xFzuLYQrFO8BP8ZNy0Q3Y/szF2HNY1AdD1XySDkxkrOsStHcdyO4zm3/GC1HSWDotSTOHo2oNti1FZh8ElJa5BvVEhKKLodUjasIDbcsL4FraNrWWpOMAQ/oHcTEwSqNM9949u7zmzERbi5nLt/sOpvyYbAc1AI1CXp9UjJe9FuyGxo6Yv55uYEEs5XPYnisD2+14s2ov+RKHo84tBV258zVs7fzIhV0EfH/m4yTFfnUlENccPZnq6TdeB/dD+VbP6qEKqr0ne+wt4JaHMrthJjDbdTPn3xozHdDdJuLnZjLrWDDWP4vIQxFbnVeACka4+l61JLxiwxmnjo5P+U7MEPmyPt6tcvqjilsGKnfQ997wbbNFV8QT+vQKooTamTp2m0cZ+n+Sp2DT5J/gOuat06"}
PIPELINE_CODE_ROOT = (
    Path("/content/bangla_financial_runtime/pipeline_code") if IN_COLAB else PROJECT_ROOT
)
os.environ["PIPELINE_PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["PIPELINE_CODE_ROOT"] = str(PIPELINE_CODE_ROOT)
sys.path.insert(0, str(PIPELINE_CODE_ROOT / "notebook"))
def materialize_pipeline_files() -> int:
    for relative_path, blob in EMBEDDED_STAGE_FILES.items():
        target = PIPELINE_CODE_ROOT / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        source = zlib.decompress(base64.b64decode(blob))
        # Avoid unnecessary Drive writes while still repairing a missing or
        # partial notebook directory after a Colab restart.
        if not target.exists() or target.read_bytes() != source:
            target.write_bytes(source)
    return len(EMBEDDED_STAGE_FILES)

print("Materialized", materialize_pipeline_files(), "embedded pipeline files under", PIPELINE_CODE_ROOT)


Materialized 15 embedded pipeline files under /content/bangla_financial_runtime/pipeline_code


In [3]:
# Install the runtime used by this notebook. Colab already provides torch.
import importlib.metadata as importlib_metadata
import subprocess, sys
from packaging.requirements import Requirement

PACKAGES = [
    # Install this compiled scientific stack together.  Installing an
    # unconstrained NumPy beside Colab's preloaded scipy/sklearn can leave
    # extension modules compiled against different NumPy ABIs.
    "numpy==2.2.6",
    "scipy==1.15.3",
    "scikit-learn==1.6.1",
    "pandas==2.2.3",
    "matplotlib==3.10.1",
    "seaborn==0.13.2",
    "pyannote.audio>=4,<5",
    "faster-whisper>=1.2,<2",
    # CTranslate2 4.8 targets Transformers 5 and passes its new `dtype`
    # loader keyword.  This pinned 4.6 release is compatible with the stable
    # Transformers 4.48 Whisper loader used for this fine-tuned checkpoint.
    "ctranslate2==4.6.1",
    # Pin to the stable 4.x API used by this Whisper fine-tune and the
    # CTranslate2 converter.  An unconstrained future Transformers major
    # release can break converter imports even after a successful download.
    "transformers==4.48.3",
    "datasets>=3,<5",
    "accelerate>=1,<2",
    "huggingface_hub>=0.28",
    "google-genai>=1",
    "jiwer>=3,<5",
    "soundfile>=0.12",
    "pyloudnorm>=0.1",
    "mutagen>=1.47",
]
if IN_COLAB:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
    # Colab can expose an incompatible preinstalled torchvision next to its
    # Torch build. Transformers then imports torchvision's optional image
    # helpers while loading a Whisper *audio* model and crashes on a missing
    # torchvision::nms operator. This pipeline never uses torchvision; remove
    # it rather than replacing Torch or downloading another multi-GB wheel.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"], check=False)

    def requirement_satisfied(spec: str) -> bool:
        req = Requirement(spec)
        try:
            installed = importlib_metadata.version(req.name)
        except importlib_metadata.PackageNotFoundError:
            return False
        return installed in req.specifier

    needs_install = [spec for spec in PACKAGES if not requirement_satisfied(spec)]
    environment_changed = bool(needs_install)
    compiled_modules = {"numpy", "scipy", "sklearn", "pandas", "matplotlib", "transformers", "ctranslate2"}
    loaded_before_install = sorted(compiled_modules.intersection(sys.modules))
    if needs_install:
        print("Installing/updating", len(needs_install), "runtime requirements...")
        subprocess.run([
            sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--no-cache-dir", *PACKAGES,
        ], check=True)

    # Validate compiled imports in a clean subprocess; this detects the exact
    # partial NumPy/SciPy/sklearn ABI failures that an in-kernel import can hide.
    probe = subprocess.run([
        sys.executable, "-c",
        "import numpy, scipy, sklearn, pandas, transformers, ctranslate2; print('scientific stack import OK')",
    ], text=True, capture_output=True)
    if probe.returncode:
        print(probe.stderr[-8000:])
        print("Repairing the compiled scientific stack...", flush=True)
        subprocess.run([
            sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-cache-dir",
            "numpy==2.2.6", "scipy==1.15.3", "scikit-learn==1.6.1", "pandas==2.2.3",
        ], check=True)
        loaded_before_install = ["compiled_stack_repair"]
        environment_changed = True
    else:
        print(probe.stdout.strip())

    if environment_changed and loaded_before_install:
        # A binary package already imported by Colab cannot be safely replaced
        # in place. Restart once; Drive outputs remain intact and the next Run
        # all skips installation because the requested versions are installed.
        print("Dependencies changed after compiled modules were loaded:", loaded_before_install, flush=True)
        print("Restarting the runtime once. After reconnect, choose Runtime > Run all.", flush=True)
        os.kill(os.getpid(), 9)
print("Runtime dependencies ready.")


scientific stack import OK
Runtime dependencies ready.


In [4]:
# Secrets are read without printing their values.
if IN_COLAB:
    from google.colab import userdata
    for key in ("HF_TOKEN", "GOOGLE_API_KEY"):
        try:
            value = userdata.get(key)
        except Exception:
            value = None
        if value:
            os.environ[key] = value

print({
    "HF_TOKEN": bool(os.environ.get("HF_TOKEN")),
    "GOOGLE_API_KEY": bool(os.environ.get("GOOGLE_API_KEY")),
})


{'HF_TOKEN': True, 'GOOGLE_API_KEY': False}


## 1. One configuration cell

Run this cell once, edit only the paths and switches, then run downward.
Model revisions are pinned to the repository state verified when this notebook
was assembled. Updating a revision is an explicit experiment, not an invisible
upgrade.


In [5]:
from dataclasses import dataclass, asdict
import torch

@dataclass(frozen=True)
class Config:
    seed: int = 17
    # This is the project's persistent Drive location.  Stage outputs remain
    # here across runtime restarts and are reused by the resume checks below.
    audio_source_dir: str = "/content/drive/MyDrive/ML_Project/datasets"
    import_audio: bool = True
    # Explicit allow-list: only these filenames (as they sit in
    # audio_source_dir) are ingested for this run's 5 episodes. Anything
    # else found in the same Drive folder (e.g. from syncing a whole local
    # project folder) is quarantined into a sibling datasets_excluded/
    # folder rather than silently entering the pipeline. Set to () to
    # ingest every recognized audio file found in audio_source_dir instead.
    included_audio_filenames: tuple[str, ...] = (
        "EP 30 Minutes Special 226_ Budget 2026-2027 - Green Transition and Energy Security.mp3",
        "Talk show on current trade, investment and economic condition of Bangladesh..mp3",
        "অনক করখন বনধ হয়ছ, সমন আরও কঠন পরসথত আসছ_ বটএমএ সভপত Star Biz Dialogue Star News.mp3",
        "অরথনতক সকট ও বজট  The Business Review  Bangladesh Economy  Talk Show  Kaler Kantho.mp3",
        "কয়কট বযক বদ বক সব বযকই সকট  ড. তফক আহমদ চধর Stream Talk  Dhaka Stream.mp3",
    )
    resume_completed_stages: bool = True
    force_rerun_stages: tuple[str, ...] = ()  # e.g. ("diarization", "asr")
    force_redownload_asr_model: bool = False  # clears local /content ASR cache only
    download_final_bundle: bool = True  # browser download at the final cell
    include_trained_model_in_bundle: bool = True
    persist_final_bundle_to_drive: bool = False  # avoids a second large Drive copy
    # False: these 5 audios are distinct programmes/channels, not repeat
    # episodes of one show, so the per-episode programme/channel metadata
    # inferred from each filename is kept instead of being overwritten.
    single_program_case_study: bool = False
    programme_name: str = "RTV Business Talk"  # only applied when single_program_case_study=True
    channel_name: str = "RTV Business"  # only applied when single_program_case_study=True
    self_attest_licenses: bool = False
    download_financial_corpus: bool = False
    run_diarization: bool = True
    run_asr: bool = True
    run_ger_ablation: bool = False
    run_weak_labelling: bool = False
    run_audio_weak_labelling: bool = True
    run_sentiment_training: bool = False
    allow_synthetic_smoke_test: bool = False

    diarization_model: str = "pyannote/speaker-diarization-community-1"
    diarization_revision: str = "3533c8cf8e369892e6b79ff1bf80f7b0286a54ee"
    asr_model: str = "bengaliAI/tugstugi_bengaliai-asr_whisper-medium"
    asr_revision: str = "da605cc1bd2f60a18d8e440e977ddfa921a88e63"
    sentiment_model: str = "csebuetnlp/banglabert"
    sentiment_revision: str = "9ce791f330578f50da6bc52b54205166fb5d1c8c"
    financial_dataset: str = "ashtrayAI/Bangla_Financial_news_articles_Dataset"
    financial_dataset_revision: str = "3e4d0c59dfc04f1951daf1d228f8a46879de474c"
    weak_label_model: str = "gemini-3.6-flash"  # configurable current API model

    max_chunk_sec: float = 28.0
    beam_size: int = 5
    repetition_penalty: float = 0.8
    confidence_review_threshold: float = 0.85
    multilabel_threshold: float = 0.50
    der_collar_default: float = 0.25
    min_segment_sec: float = 0.75
    min_speaker_airtime_sec: float = 9.0
    stitch_gap_sec: float = 2.0

CFG = Config()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"
print(asdict(CFG))
print("device =", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")


{'seed': 17, 'audio_source_dir': '/content/drive/MyDrive/ML_Project/datasets', 'import_audio': True, 'included_audio_filenames': ('EP 30 Minutes Special 226_ Budget 2026-2027 - Green Transition and Energy Security.mp3', 'Talk show on current trade, investment and economic condition of Bangladesh..mp3', 'অনক করখন বনধ হয়ছ, সমন আরও কঠন পরসথত আসছ_ বটএমএ সভপত Star Biz Dialogue Star News.mp3', 'অরথনতক সকট ও বজট  The Business Review  Bangladesh Economy  Talk Show  Kaler Kantho.mp3', 'কয়কট বযক বদ বক সব বযকই সকট  ড. তফক আহমদ চধর Stream Talk  Dhaka Stream.mp3'), 'resume_completed_stages': True, 'force_rerun_stages': (), 'force_redownload_asr_model': False, 'download_final_bundle': True, 'include_trained_model_in_bundle': True, 'persist_final_bundle_to_drive': False, 'single_program_case_study': False, 'programme_name': 'RTV Business Talk', 'channel_name': 'RTV Business', 'self_attest_licenses': False, 'download_financial_corpus': False, 'run_diarization': True, 'run_asr': True, 'run_ger_ablation

In [6]:
# Determinism, directories, atomic JSON, hashes, and a stage runner.
import hashlib, json, random, subprocess, time
from datetime import datetime, timezone
import numpy as np

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.seed)
os.environ["PYTHONHASHSEED"] = str(CFG.seed)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

DIRS = {
    name: PROJECT_ROOT / rel for name, rel in {
        "datasets": "datasets", "raw": "data/raw", "processed": "data/processed",
        "diar": "data/diarization", "asr": "data/asr", "fused": "data/fused",
        "ger": "data/ger", "drafts": "data/annotation_drafts",
        "annotated": "data/annotated", "weak": "data/weak_labels",
        "weak_audio": "data/weak_audio",
        "aggregated": "data/aggregated", "models": "models", "reports": "reports",
        "figures": "reports/figures", "artifacts": "artifacts",
    }.items()
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

# Keep heavyweight, reproducible downloads off Drive.  Colab's local disk is
# ephemeral but large; Drive stores only audio, stage outputs, reports and the
# final small artifact bundle. This avoids temporary checkpoint + conversion
# copies exhausting a Drive quota.
RUNTIME_CACHE_ROOT = Path("/content/bangla_financial_runtime") if IN_COLAB else PROJECT_ROOT / "runtime_cache"
RUNTIME_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(RUNTIME_CACHE_ROOT / "hf_home")
os.environ["HF_HUB_CACHE"] = str(RUNTIME_CACHE_ROOT / "hf_hub")
os.environ["TRANSFORMERS_CACHE"] = str(RUNTIME_CACHE_ROOT / "transformers")
print("Persistent Drive workspace:", PROJECT_ROOT)
print("Ephemeral model/cache workspace:", RUNTIME_CACHE_ROOT)

def atomic_json(path: Path, obj) -> None:
    def json_default(value):
        # NumPy scalar types (np.bool_, np.int64, np.float64) are not handled
        # by Python's standard JSON encoder. `.item()` converts them to their
        # native bool/int/float equivalents without changing their value.
        item = getattr(value, "item", None)
        if callable(item):
            return item()
        if isinstance(value, Path):
            return str(value)
        if isinstance(value, (set, frozenset)):
            return sorted(value)
        raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")

    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(obj, ensure_ascii=False, indent=2, default=json_default),
        encoding="utf-8",
    )
    os.replace(tmp, path)

def sha256(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def repo_run(*args: str, check: bool = True) -> subprocess.CompletedProcess:
    script = PIPELINE_CODE_ROOT / args[0]
    if not script.exists() and "materialize_pipeline_files" in globals():
        materialize_pipeline_files()
    if not script.exists():
        raise FileNotFoundError(
            f"Missing standalone stage script: {script}. Run the earlier "
            "'Standalone payload' cell, or use the latest standalone notebook."
        )
    cmd = [sys.executable, str(script), *args[1:]]
    print("$", " ".join(cmd))
    child_env = os.environ.copy()
    child_env["PIPELINE_PROJECT_ROOT"] = str(PROJECT_ROOT)
    proc = subprocess.run(cmd, cwd=PROJECT_ROOT, env=child_env, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout[-12000:])
    if proc.stderr:
        print(proc.stderr[-5000:])
    if check and proc.returncode:
        raise RuntimeError(f"Command failed ({proc.returncode}): {' '.join(cmd)}")
    return proc

def eligible_episode_ids() -> list[str]:
    # Read the durable registry without regenerating any expensive output.
    runtime_registry = Path(os.environ.get("PIPELINE_REGISTRY_CSV", ""))
    registry_path = runtime_registry if runtime_registry.is_file() else DIRS["raw"] / "registry.csv"
    if not registry_path.exists():
        return []
    import csv
    with registry_path.open(encoding="utf-8", newline="") as f:
        return [r["episode_id"] for r in csv.DictReader(f)
                if r.get("duration_flag") in {"ok", "short", "long"}]

def apply_corpus_metadata() -> None:
    '''Apply durable, human-editable episode metadata after inventory refresh.'''
    import csv
    registry_path = DIRS["raw"] / "registry.csv"
    if not registry_path.exists():
        return
    with registry_path.open(encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
        fields = list(rows[0]) if rows else []
    metadata_path = DIRS["raw"] / "episode_metadata.csv"
    metadata_fields = ["episode_id", "local_path", "has_remote_guest", "covers_market_news", "notes"]
    if metadata_path.exists():
        with metadata_path.open(encoding="utf-8", newline="") as f:
            overrides = {r.get("local_path") or r.get("episode_id"): r for r in csv.DictReader(f)}
    else:
        overrides = {}
        with metadata_path.open("w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=metadata_fields)
            writer.writeheader()
            for row in rows:
                writer.writerow({
                    "episode_id": row["episode_id"], "local_path": row["local_path"],
                    "has_remote_guest": "", "covers_market_news": "", "notes": "human review required",
                })
    changed = False
    for row in rows:
        if CFG.single_program_case_study and (
                row.get("programme") != CFG.programme_name or row.get("channel") != CFG.channel_name):
            row["programme"], row["channel"], changed = CFG.programme_name, CFG.channel_name, True
        override = overrides.get(row.get("local_path")) or overrides.get(row.get("episode_id")) or {}
        for key in ("has_remote_guest", "covers_market_news"):
            value = str(override.get(key, "")).strip().lower()
            if value in {"true", "false"} and row.get(key, "").lower() != value:
                row[key], changed = value, True
    def write_registry_copy(path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fields)
            writer.writeheader(); writer.writerows(rows)
            f.flush()
            try: os.fsync(f.fileno())
            except OSError: pass

    # Google Drive shortcut/FUSE mounts can briefly hide a file after an
    # atomic rename. Write the small authoritative CSV in place, then give all
    # subprocess stages a local /content mirror through an environment path.
    if changed and fields:
        write_registry_copy(registry_path)
    runtime_registry = RUNTIME_CACHE_ROOT / "registry.csv"
    if fields:
        write_registry_copy(runtime_registry)
        os.environ["PIPELINE_REGISTRY_CSV"] = str(runtime_registry)
    print(f"Corpus metadata: {CFG.channel_name} / {CFG.programme_name}")
    print("Review episode flags once in:", metadata_path)

def stage_is_complete(stage: str, output_dir: Path, suffix: str = ".json") -> bool:
    # Conservative persistent checkpoint: every eligible episode exists.
    episode_ids = eligible_episode_ids()
    if not CFG.resume_completed_stages or stage in CFG.force_rerun_stages or not episode_ids:
        return False
    return all((output_dir / f"{episode_id}{suffix}").exists() for episode_id in episode_ids)

def run_or_resume(
    stage: str, output_dir: Path, command: list[str], *, suffix: str = ".json",
    episode_flag: str | None = "--episodes",
) -> bool:
    if stage_is_complete(stage, output_dir, suffix):
        print(f"[resume] {stage}: all durable outputs already exist; skipping compute.")
        return False
    episode_ids = eligible_episode_ids()
    resumable = CFG.resume_completed_stages and stage not in CFG.force_rerun_stages and episode_ids
    missing = [e for e in episode_ids if not (output_dir / f"{e}{suffix}").exists()] if resumable else []
    if resumable and episode_flag and 0 < len(missing) < len(episode_ids):
        # A Colab disconnect can leave some episodes already durable on Drive.
        # Recompute only the missing ones instead of redoing the whole stage.
        print(f"[resume] {stage}: {len(missing)}/{len(episode_ids)} episode(s) missing; "
              f"resuming only: {', '.join(missing)}")
        repo_run(*command, episode_flag, ",".join(missing))
    else:
        print(f"[resume] {stage}: missing/incomplete output; running all eligible episodes.")
        repo_run(*command)
    return True

RUN_MANIFEST = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "config": asdict(CFG), "device": DEVICE, "files": {},
}


Persistent Drive workspace: /content/drive/MyDrive/ML_Project
Ephemeral model/cache workspace: /content/bangla_financial_runtime


### `.puku` semantic-index audit

The `.puku/puku-embeddings.db` file is an index of project text/code chunks,
not an additional labelled financial dataset. This read-only audit records
what it covered; model training never treats embeddings as ground truth.


In [7]:
import sqlite3

puku_db = PROJECT_ROOT / ".puku" / "puku-embeddings.db"
if puku_db.exists():
    con = sqlite3.connect(f"file:{puku_db.as_posix()}?mode=ro", uri=True)
    integrity = con.execute("PRAGMA integrity_check").fetchone()[0]
    counts = {
        table: con.execute(f"SELECT count(*) FROM {table}").fetchone()[0]
        for table in ("Files", "Chunks", "SummaryStats")
    }
    model_meta = con.execute("SELECT version, model FROM CacheMeta").fetchall()
    indexed = [r[0] for r in con.execute("SELECT uri FROM Files ORDER BY uri")]
    con.close()
    PUKU_AUDIT = {"integrity": integrity, "counts": counts, "model": model_meta,
                  "indexed_files": indexed}
    print({k: v for k, v in PUKU_AUDIT.items() if k != "indexed_files"})
else:
    PUKU_AUDIT = {"status": "not present"}
    print(PUKU_AUDIT)


{'status': 'not present'}


## 2. Data acquisition, governance, and hygiene

Audio import is rights-first. Set `import_audio=True` only after the source
folder contains approved research audio. The inventory stage hashes every
file and creates the registry; review programme, remote-guest, market-news,
and license fields before annotation.


In [8]:
import shutil

source_dir = Path(CFG.audio_source_dir)
audio_exts = {".mp3", ".wav", ".m4a", ".flac", ".ogg", ".aac"}
allowed_names = set(CFG.included_audio_filenames)  # empty set = no filtering

if CFG.import_audio:
    if source_dir.exists():
        for src in sorted(p for p in source_dir.iterdir() if p.suffix.lower() in audio_exts):
            dst = DIRS["datasets"] / src.name
            # With the recommended layout source and destination are the same
            # folder: never copy/hash an audio file onto itself.
            if src.resolve() == dst.resolve():
                continue
            if not dst.exists() or sha256(src) != sha256(dst):
                shutil.copy2(src, dst)
                print("imported", src.name)
    if IN_COLAB and not any(p.suffix.lower() in audio_exts for p in DIRS["datasets"].iterdir()):
        from google.colab import files
        n = len(allowed_names) or "the authorized"
        print(f"No Drive audio found. Choose the {n} audio file(s) for this run to upload.")
        uploaded = files.upload()
        for name, content in uploaded.items():
            if Path(name).suffix.lower() in audio_exts:
                (DIRS["datasets"] / Path(name).name).write_bytes(content)
                print("uploaded", name)

# Keep this run scoped to exactly the allow-listed episodes, even if the same
# Drive folder later accumulates other audio (e.g. syncing a whole local
# project folder). Anything else is moved -- not deleted -- to a sibling
# folder outside datasets/ so stage 0's recursive scan never sees it.
if allowed_names:
    EXCLUDED_AUDIO_DIR = DIRS["datasets"].parent / "datasets_excluded"
    for p in sorted(DIRS["datasets"].iterdir()):
        if p.is_file() and p.suffix.lower() in audio_exts and p.name not in allowed_names:
            EXCLUDED_AUDIO_DIR.mkdir(parents=True, exist_ok=True)
            dst = EXCLUDED_AUDIO_DIR / p.name
            if dst.exists():
                p.unlink()
            else:
                shutil.move(str(p), str(dst))
            print("excluded (not in included_audio_filenames):", p.name)

print("audio files:", [p.name for p in sorted(DIRS["datasets"].iterdir())])

excluded (not in included_audio_filenames): অনক করখন বনধ হয়ছ, সমন আরও কঠন পরসথত আসছ_ বটএমএ সভপত Star Biz Dialogue Star News.mp3
excluded (not in included_audio_filenames): কয়কট বযক বদ বক সব বযকই সকট  ড. তফক আহমদ চধর Stream Talk  Dhaka Stream.mp3
excluded (not in included_audio_filenames): যকতরষটরর ভমক থকত পর তব চনক বদ দয় রহঙগ সমসযর সমধন হব ন.mp3
audio files: ['Bangla_Financial_Pipeline_Standalone_Colab.ipynb', 'EP 30 Minutes Special 226_ Budget 2026-2027 - Green Transition and Energy Security.mp3', 'Talk show on current trade, investment and economic condition of Bangladesh..mp3', 'অরথনতক সকট ও বজট  The Business Review  Bangladesh Economy  Talk Show  Kaler Kantho.mp3']


In [9]:
# Optional: download the unlabelled 7,695-article financial corpus at a pinned revision.
# It is an experiment input, so cache it locally rather than consuming Drive.
from huggingface_hub import snapshot_download

FINANCIAL_RAW = RUNTIME_CACHE_ROOT / "financial_news_raw"
if CFG.download_financial_corpus:
    snapshot_download(
        repo_id=CFG.financial_dataset,
        repo_type="dataset",
        revision=CFG.financial_dataset_revision,
        local_dir=FINANCIAL_RAW,
        allow_patterns=["Bangla_fin_news_articles/*.csv", "README.md"],
    )
    csv_count = len(list(FINANCIAL_RAW.glob("Bangla_fin_news_articles/*.csv")))
    print("financial article CSVs =", csv_count)


In [10]:
# Standalone preflight: safe to rerun after a Colab restart or Drive remount.
if "materialize_pipeline_files" not in globals():
    raise RuntimeError("Run the earlier 'Standalone payload' cell before starting Stage 0.")
materialize_pipeline_files()
required_stage = PIPELINE_CODE_ROOT / "notebook" / "01_stage_0_audio_inventory.py"
assert required_stage.exists(), f"Standalone payload repair failed: {required_stage}"
print("Standalone preflight OK:", required_stage)


Standalone preflight OK: /content/bangla_financial_runtime/pipeline_code/notebook/01_stage_0_audio_inventory.py


In [11]:
# Stage 0a/0b/0c: inventory is cheap; hygiene is kept when every WAV exists.
os.environ["SINGLE_PROGRAM_CASE_STUDY"] = str(CFG.single_program_case_study).lower()
inventory_args = ["notebook/01_stage_0_audio_inventory.py"]
if CFG.self_attest_licenses:
    inventory_args.append("--self-attest")
repo_run(*inventory_args)
apply_corpus_metadata()
if not stage_is_complete("hygiene", DIRS["processed"], ".wav"):
    repo_run("notebook/02_stage_0b_audio_hygiene.py")
else:
    print("[resume] hygiene: processed WAVs already exist; skipping.")
repo_run("notebook/03_stage_0c_diversity_audit.py")


$ /usr/bin/python3 /content/bangla_financial_runtime/pipeline_code/notebook/01_stage_0_audio_inventory.py
[stage0] repo root: /content/drive/MyDrive/ML_Project
[stage0] scanning: /content/drive/MyDrive/ML_Project/datasets
[stage0] found 3 audio file(s)
[stage0] registry written: data/raw/registry.csv
[stage0] summary: {'total': 3, 'ok': 3, 'short_flag': 0, 'long_flag': 0, 'rejected': 0, 'rejection_reasons': []}
  - ep001  dur= 2729.5s  sr=44100  ch=2  codec=MP3             flag=ok        programme='EP 30 Minutes Special 226_ Budget 2026-2027 - Green Transiti'
  - ep002  dur= 2077.1s  sr=44100  ch=2  codec=MP3             flag=ok        programme='Talk show on current trade, investment and economic conditio'
  - ep003  dur= 3459.8s  sr=44100  ch=2  codec=MP3             flag=ok        programme='অরথনতক সকট ও বজট  The Business Review  Bangladesh Economy  T'

Corpus metadata: RTV Business / RTV Business Talk
Review episode flags once in: /content/drive/MyDrive/ML_Project/data/raw/episode_

CompletedProcess(args=['/usr/bin/python3', '/content/bangla_financial_runtime/pipeline_code/notebook/03_stage_0c_diversity_audit.py'], returncode=0, stdout='[stage0c] episodes=3 programmes=3 remote=0 market=1 in_window=3\n[stage0c] GAPS:\n  - need at least 8 episodes, have 3 (add 5 more)\n  - need >= 4 programmes for the configured study scope, have 3\n  - need >= 2 episodes with remote guests, have 0\n[stage0c] report: reports/diversity_audit.md\n', stderr='')

In [12]:
import pandas as pd
from IPython.display import display, Markdown

registry_path = DIRS["raw"] / "registry.csv"
registry = pd.read_csv(registry_path) if registry_path.exists() else pd.DataFrame()
display(registry)
audit_path = DIRS["reports"] / "diversity_audit.md"
if audit_path.exists():
    display(Markdown(audit_path.read_text(encoding="utf-8")))


,episode_id,channel,duration_sec,programme,has_remote_guest,covers_market_news,local_path,sha256,license_attestation,sample_rate,channels,codec,bit_rate_kbps,file_size_bytes,duration_flag,rejection_reason,sha256_verified,notes
0,ep001,unknown,2729.46,EP 30 Minutes Special 226_ Budget 2026-2027 - ...,False,False,datasets/EP 30 Minutes Special 226_ Budget 202...,33385930614491c9001f5ce0c7ac213e8b1acf4bbebea9...,UNVERIFIED — must be completed before Stage 4,44100,2,MP3,67,23130812,ok,NaN,True,NaN
1,ep002,unknown,2077.07,"Talk show on current trade, investment and eco...",False,True,"datasets/Talk show on current trade, investmen...",210fad2e7915ef8f1c33232dd893ad4bb646ae78335fb2...,UNVERIFIED — must be completed before Stage 4,44100,2,MP3,68,17744184,ok,NaN,True,NaN
2,ep003,unknown,3459.79,অরথনতক সকট ও বজট The Business Review Banglad...,False,False,datasets/অরথনতক সকট ও বজট The Business Review...,a266b4857a0e771a7777f5291324ee2019a4cb7f723526...,UNVERIFIED — must be completed before Stage 4,44100,2,MP3,81,35140180,ok,NaN,True,NaN


# Stage 0c — Diversity Audit

Verification against the build-plan §4.1 selection criteria. Run ``python notebook/03_stage_0c_diversity_audit.py`` to refresh.

## Headline numbers

- Eligible episodes: **3**
- Distinct programmes: **3**
- Episodes with a remote guest: **0**
- Episodes covering market news: **1**
- Episodes inside 30--60 min window: **3**

## Per-programme breakdown

- 'EP 30 Minutes Special 226_ Budget 2026-2027 - Green Transition and Energy Security': 1
- 'Talk show on current trade, investment and economic condition of Bangladesh.': 1
- 'অরথনতক সকট ও বজট  The Business Review  Bangladesh Economy  Talk Show  Kaler Kantho': 1

## Duration summary

- Mean duration: **45.92 min**
- Min duration: **34.62 min**
- Max duration: **57.66 min**

## Gaps vs. selection criteria

- ⚠️  need at least 8 episodes, have 3 (add 5 more)
- ⚠️  need >= 4 programmes for the configured study scope, have 3
- ⚠️  need >= 2 episodes with remote guests, have 0


## 3. Speaker diarization

Primary inference uses pyannote community-1 on GPU and consumes its exclusive
single-speaker timeline for later fusion. Post-processing applies chronological
IDs, first-speaker-wins overlap handling, a 0.17 s inter-speaker gap, ≤2 s
same-speaker stitching, `<0.75 s` micro-segment removal, and `<9 s total`
under-used-speaker pruning. Report DER with both 0.25 s and 0 s collars.

Fine-tuning the segmentation head is the preferred final experiment once
enough in-domain RTTM exists; the notebook accepts a custom checkpoint through
`DIARIZATION_BACKBONE` without pretending a fallback VAD run is research DER.


In [13]:
os.environ.update({
    "DIARIZATION_BACKBONE": CFG.diarization_model,
    "DIARIZATION_DEVICE": DEVICE,
    "DIARIZATION_MIN_DURATION_OFF": "0.1",
    "DIARIZATION_MIN_CLUSTER_SIZE": "20",
})
if CFG.run_diarization:
    if not os.environ.get("HF_TOKEN"):
        print("WARNING: no HF_TOKEN; stage runner will create a clearly marked VAD fallback only.")
    run_or_resume("diarization", DIRS["diar"], [
        "notebook/04_stage_1_diarization.py", "--device", DEVICE,
    ])


[resume] diarization: missing/incomplete output; running all eligible episodes.
$ /usr/bin/python3 /content/bangla_financial_runtime/pipeline_code/notebook/04_stage_1_diarization.py --device cuda
[stage1] 3 episode(s) to diarize  device=cuda

Could not download Pipeline from pyannote/speaker-diarization-community-1.
It might be because the repository is private or gated:

* visit https://hf.co/pyannote/speaker-diarization-community-1 to accept user conditions
* visit https://hf.co/settings/tokens to create an authentication token
* load the Pipeline with the `token` argument:
    >>> Pipeline.from_pretrained('pyannote/speaker-diarization-community-1', token='hf_....')

  - ep001  provider=fallback  raw= 406 -> cleaned=  91  speakers=1  -> data/diarization/ep001.rttm
      note: pyannote skipped: 403 Client Error. (Request ID: Root=1-6a7a1499-7c5f906b25ee50545003b2ea;486f26c4-4393-4334-92d2-0e1290ca0aec)

Cannot access gated repo for url https://huggingface.co/pyannote/speaker-diarizati

In [14]:
# Diarization sanity audit. Fallback output is never accepted as measured DER.
diar_audit = []
for path in sorted(DIRS["diar"].glob("*.json")):
    obj = json.loads(path.read_text(encoding="utf-8"))
    segs = obj.get("segments", [])
    per_spk = {}
    for s in segs:
        per_spk[s["speaker_id"]] = per_spk.get(s["speaker_id"], 0.0) + s["end_sec"] - s["start_sec"]
    diar_audit.append({
        "episode": path.stem, "provider": obj.get("provider"), "segments": len(segs),
        "speakers": len(per_spk), "min_turn": min((s["end_sec"]-s["start_sec"] for s in segs), default=0),
        "min_speaker_airtime": min(per_spk.values(), default=0),
    })
display(pd.DataFrame(diar_audit))


,episode,provider,segments,speakers,min_turn,min_speaker_airtime
0,ep001,fallback,91,1,0.76,584.42
1,ep002,fallback,91,1,0.75,494.56
2,ep003,fallback,138,1,0.75,600.09


## 4. Long-form Bangla ASR

The BengaliAI checkpoint is a Transformers Whisper repository, so it is first
converted to CTranslate2 before `faster-whisper` loads it. Without this step,
passing the Hub ID directly may silently fall back to a generic Whisper size.
The stage emits word timestamps needed by fusion and uses resumable episode
JSON outputs.


In [15]:
# Convert the pinned BengaliAI Transformers checkpoint once on Colab's local
# disk. Keeping both the original checkpoint and CTranslate2 conversion in
# Drive can consume 6+ GB during one run. `--force` makes this cell safely
# retry a previously interrupted conversion without re-downloading the HF
# checkpoint; the full converter error is retained if it still fails.
CT2_DIR = RUNTIME_CACHE_ROOT / f"tugstugi-medium-ct2-{CFG.asr_revision[:8]}"
HF_ASR_DIR = RUNTIME_CACHE_ROOT / f"tugstugi-medium-hf-{CFG.asr_revision[:8]}"
if CFG.force_redownload_asr_model:
    # Explicit user-requested clean retry.  These are ephemeral /content
    # model artifacts only; no Drive audio, stage JSON, report or annotation
    # file is touched.
    import shutil
    for cache_dir in (HF_ASR_DIR, CT2_DIR):
        if cache_dir.exists():
            shutil.rmtree(cache_dir)
            print("Removed local ASR cache:", cache_dir)
if CFG.run_asr and not (CT2_DIR / "model.bin").exists():
    snapshot_download(
        repo_id=CFG.asr_model, revision=CFG.asr_revision, local_dir=HF_ASR_DIR,
    )
    cmd = [
        "ct2-transformers-converter", "--model", str(HF_ASR_DIR),
        "--output_dir", str(CT2_DIR),
        "--force",
        "--copy_files", "tokenizer.json", "tokenizer_config.json",
        "special_tokens_map.json", "preprocessor_config.json",
        "--quantization", "float16" if DEVICE == "cuda" else "int8",
    ]
    print("$", " ".join(cmd))
    conversion = subprocess.run(cmd, text=True, capture_output=True)
    if conversion.stdout:
        print(conversion.stdout[-8000:])
    if conversion.returncode:
        print(conversion.stderr[-12000:])
        raise RuntimeError(
            "CTranslate2 conversion failed. The checkpoint remains cached at "
            f"{HF_ASR_DIR}; fix the printed converter error and rerun this cell."
        )
ASR_RUNTIME_MODEL = str(CT2_DIR) if (CT2_DIR / "model.bin").exists() else CFG.asr_model
print("ASR runtime model =", ASR_RUNTIME_MODEL)


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

$ ct2-transformers-converter --model /content/bangla_financial_runtime/tugstugi-medium-hf-da605cc1 --output_dir /content/bangla_financial_runtime/tugstugi-medium-ct2-da605cc1 --force --copy_files tokenizer.json tokenizer_config.json special_tokens_map.json preprocessor_config.json --quantization float16
ASR runtime model = /content/bangla_financial_runtime/tugstugi-medium-ct2-da605cc1


In [16]:
os.environ.update({
    "ASR_DEVICE": DEVICE,
    "ASR_COMPUTE": COMPUTE_TYPE,
    "ASR_BEAM_SIZE": str(CFG.beam_size),
    "ASR_REPETITION_PENALTY": str(CFG.repetition_penalty),
    "ASR_CONDITION_ON_PREVIOUS_TEXT": "false",
    "ASR_USE_DEMUCS": "auto",
})
# Recovery guard: a runtime restart can leave durable audio/processed WAVs
# while the small registry.csv is absent. Rebuild that metadata first, then
# run hygiene only if an expected 16 kHz WAV is genuinely missing.
if not (DIRS["raw"] / "registry.csv").exists():
    print("[recovery] registry.csv missing; rebuilding inventory from datasets/.")
    inventory_args = ["notebook/01_stage_0_audio_inventory.py"]
    if CFG.self_attest_licenses:
        inventory_args.append("--self-attest")
    repo_run(*inventory_args)
apply_corpus_metadata()
episode_ids = eligible_episode_ids()
missing_wavs = [ep for ep in episode_ids if not (DIRS["processed"] / f"{ep}.wav").exists()]
if missing_wavs:
    print("[recovery] missing processed WAVs:", missing_wavs)
    repo_run("notebook/02_stage_0b_audio_hygiene.py")
if CFG.run_asr:
    run_or_resume("asr", DIRS["asr"], [
        "notebook/05_stage_2_asr.py",
        "--model", ASR_RUNTIME_MODEL,
        "--device", DEVICE,
        "--compute-type", COMPUTE_TYPE,
        "--chunk-sec", str(CFG.max_chunk_sec),
        "--beam-size", str(CFG.beam_size),
        "--condition-on-previous-text", "false",
        "--demucs-mode", "auto",
    ])


Corpus metadata: RTV Business / RTV Business Talk
Review episode flags once in: /content/drive/MyDrive/ML_Project/data/raw/episode_metadata.csv
[resume] asr: missing/incomplete output; running all eligible episodes.
$ /usr/bin/python3 /content/bangla_financial_runtime/pipeline_code/notebook/05_stage_2_asr.py --model /content/bangla_financial_runtime/tugstugi-medium-ct2-da605cc1 --device cuda --compute-type float16 --chunk-sec 28.0 --beam-size 5 --condition-on-previous-text false --demucs-mode auto
[stage2] ASR over 3 episode(s); model=/content/bangla_financial_runtime/tugstugi-medium-ct2-da605cc1 device=cuda compute=float16 chunk=28.0s
  - ep001  model=/content/bangla_financial_runtime/tugstugi-medium-ct2-da605cc1  segments=163  chunks=98  elapsed=3708.0s  -> data/asr/ep001.json
      warnings: demucs: chunk 0 flux=0.026 demucs=False; chunk 1 flux=0.022 demucs=False; chunk 2 flux=0.026 demucs=False; chunk 3 flux=0.027 demucs=False; chunk 4 flux=0.025 demucs=False; chunk 5 flux=0.027 de

In [17]:
# ASR sanity report: real words, timestamps, confidence, and provider/model identity.
asr_audit = []
for path in sorted(DIRS["asr"].glob("*.json")):
    obj = json.loads(path.read_text(encoding="utf-8"))
    segs = obj.get("segments", [])
    words = [w for s in segs for w in (s.get("words") or [])]
    asr_audit.append({
        "episode": path.stem, "model": obj.get("model"), "segments": len(segs),
        "words": len(words), "timestamped_words": sum("start" in w and "end" in w for w in words),
        "warnings": obj.get("warnings", ""),
    })
display(pd.DataFrame(asr_audit))


,episode,model,segments,words,timestamped_words,warnings
0,ep001,/content/bangla_financial_runtime/tugstugi-med...,163,5377,5377,demucs: chunk 0 flux=0.026 demucs=False; chunk...
1,ep002,/content/bangla_financial_runtime/tugstugi-med...,125,4411,4411,demucs: chunk 0 flux=0.029 demucs=False; chunk...
2,ep003,/content/bangla_financial_runtime/tugstugi-med...,242,7546,7546,demucs: chunk 0 flux=0.039 demucs=False; chunk...


## 5. Timestamp fusion: words → episode-local speakers

This is the under-engineered join that the proposal treats as a contribution.
Each ASR word is assigned by midpoint to the exclusive diarization interval.
If numerical boundary drift leaves it outside every interval, the interval
with maximum temporal overlap is used. Unassigned words are retained with
`speaker_id=-1` and later fail the submission gate—they are never dropped.


In [18]:
import re, unicodedata

ZERO_WIDTH_BIDI = re.compile(r"[\u200b-\u200f\u202a-\u202e\u2060-\u206f\ufeff]")
SENTENCE_END = re.compile(r"[।!?]+$")

def normalize_transcript_draft(text: str) -> str:
    text = unicodedata.normalize("NFC", text or "")
    text = ZERO_WIDTH_BIDI.sub("", text)
    return re.sub(r"\s+", " ", text).strip()

def overlap_seconds(a0, a1, b0, b1) -> float:
    return max(0.0, min(a1, b1) - max(a0, b0))

def assign_word_to_speaker(word: dict, diar_segments: list[dict]) -> tuple[int, str]:
    ws, we = float(word["start"]), float(word["end"])
    mid = (ws + we) / 2.0
    containing = [s for s in diar_segments if float(s["start_sec"]) <= mid < float(s["end_sec"])]
    if containing:
        # Exclusive timelines normally produce one match. Chronological order
        # implements the project's first-speaker-wins reference policy.
        chosen = min(containing, key=lambda s: (float(s["start_sec"]), int(s["speaker_id"])))
        return int(chosen["speaker_id"]), "midpoint"
    scored = [(overlap_seconds(ws, we, float(s["start_sec"]), float(s["end_sec"])), s)
              for s in diar_segments]
    best_overlap, chosen = max(scored, key=lambda x: x[0], default=(0.0, None))
    if chosen is not None and best_overlap > 0:
        return int(chosen["speaker_id"]), "max_overlap"
    return -1, "unassigned"

def fuse_episode(episode_id: str) -> dict:
    diar_path = DIRS["diar"] / f"{episode_id}.json"
    asr_path = DIRS["asr"] / f"{episode_id}.json"
    diar = json.loads(diar_path.read_text(encoding="utf-8"))
    asr = json.loads(asr_path.read_text(encoding="utf-8"))
    dsegs = sorted(diar.get("segments", []), key=lambda s: (s["start_sec"], s["end_sec"]))

    words = []
    for seg in asr.get("segments", []):
        seg_words = seg.get("words") or []
        if not seg_words and seg.get("text_verbatim"):
            # Keep a coarse timed token so missing word timestamps are visible.
            seg_words = [{"start": seg["start"], "end": seg["end"],
                          "word": seg["text_verbatim"], "prob": None, "coarse": True}]
        for word in seg_words:
            speaker_id, method = assign_word_to_speaker(word, dsegs)
            words.append({**word, "speaker_id": speaker_id, "assignment_method": method})
    words.sort(key=lambda w: (w["start"], w["end"]))

    # Utterances split on speaker change, >2 s pause, sentence-final punctuation,
    # or 28 s duration. This keeps turns semantically useful for sentiment.
    utterances, current = [], []
    def flush():
        nonlocal current
        if not current:
            return
        i = len(utterances)
        text = normalize_transcript_draft("".join(str(w.get("word", "")) for w in current))
        utterances.append({
            "utterance_id": f"{episode_id}_utt{i:05d}", "episode_id": episode_id,
            "speaker_id": int(current[0]["speaker_id"]),
            "start_sec": round(float(current[0]["start"]), 3),
            "end_sec": round(float(current[-1]["end"]), 3),
            "text_verbatim": text, "text_normalized": text,
            "words": current,
        })
        current = []

    for word in words:
        if current:
            speaker_changed = word["speaker_id"] != current[-1]["speaker_id"]
            long_pause = float(word["start"]) - float(current[-1]["end"]) > 2.0
            too_long = float(word["end"]) - float(current[0]["start"]) > CFG.max_chunk_sec
            ended = bool(SENTENCE_END.search(str(current[-1].get("word", "")).strip()))
            if speaker_changed or long_pause or too_long or ended:
                flush()
        current.append(word)
    flush()

    payload = {
        "schema_version": "2.0", "episode_id": episode_id,
        "diarization_provider": diar.get("provider"), "asr_model": asr.get("model"),
        "overlap_policy": "first-speaker-wins", "assignment": "midpoint_then_max_overlap",
        "words": words, "utterances": utterances,
        "stats": {
            "n_words": len(words), "n_utterances": len(utterances),
            "unassigned_words": sum(w["speaker_id"] == -1 for w in words),
            "coarse_tokens": sum(bool(w.get("coarse")) for w in words),
        },
    }
    atomic_json(DIRS["fused"] / f"{episode_id}.json", payload)
    return payload

fused_reports = []
for asr_path in sorted(DIRS["asr"].glob("*.json")):
    if (DIRS["diar"] / asr_path.name).exists():
        fused_reports.append(fuse_episode(asr_path.stem)["stats"] | {"episode": asr_path.stem})
display(pd.DataFrame(fused_reports))


,n_words,n_utterances,unassigned_words,coarse_tokens,episode
0,5377,231,4082,0,ep001
1,4411,193,3241,0,ep002
2,7546,313,6126,0,ep003


### Financial entities, ticker restoration, and transcript contract

Populate `data/raw/financial_entities.csv` with canonical names and aliases.
Restoration is deterministic and auditable. It does not hallucinate entities:
only listed aliases can be replaced. Digit-to-word conversion remains a human
annotation responsibility unless a verified converter is installed; the
validator below blocks remaining digits.


In [19]:
ENTITY_CSV = DIRS["raw"] / "financial_entities.csv"
starter_entities = pd.DataFrame([
    {"canonical": "DSEX", "aliases": "ডিএসইএক্স|DSE X|শেয়ার সূচক", "target_type": "MARKET"},
    {"canonical": "Bangladesh Bank", "aliases": "বাংলাদেশ ব্যাংক|কেন্দ্রীয় ব্যাংক", "target_type": "REGULATOR"},
    {"canonical": "BSEC", "aliases": "বিএসইসি|বি এস ই সি|সিকিউরিটিজ কমিশন", "target_type": "REGULATOR"},
    {"canonical": "NBR", "aliases": "এনবিআর|এন বি আর|জাতীয় রাজস্ব বোর্ড", "target_type": "REGULATOR"},
    {"canonical": "BBS", "aliases": "বিবিএস|বি বি এস|বাংলাদেশ পরিসংখ্যান ব্যুরো", "target_type": "REGULATOR"},
    {"canonical": "Ministry of Finance", "aliases": "অর্থ মন্ত্রণালয়|অর্থ মন্ত্রণালয়ের|অর্থ মন্ত্রণালয়ের", "target_type": "REGULATOR"},
    {"canonical": "Planning Commission", "aliases": "পরিকল্পনা কমিশন", "target_type": "REGULATOR"},
    {"canonical": "Government of Bangladesh", "aliases": "বাংলাদেশ সরকার|সরকারের অর্থনৈতিক নীতি|সরকারি অর্থনীতি", "target_type": "REGULATOR"},
    {"canonical": "IMF", "aliases": "আইএমএফ|আই এম এফ|আন্তর্জাতিক মুদ্রা তহবিল", "target_type": "REGULATOR"},
    {"canonical": "World Bank", "aliases": "বিশ্বব্যাংক|বিশ্ব ব্যাংক", "target_type": "REGULATOR"},
    {"canonical": "ADB", "aliases": "এডিবি|এ ডি বি|এশীয় উন্নয়ন ব্যাংক", "target_type": "REGULATOR"},
    {"canonical": "GDP", "aliases": "জিডিপি|জি ডি পি|মোট দেশজ উৎপাদন", "target_type": "MACRO_INDICATOR"},
    {"canonical": "CPI", "aliases": "সিপিআই|সি পি আই|ভোক্তা মূল্যসূচক", "target_type": "MACRO_INDICATOR"},
    {"canonical": "inflation", "aliases": "মূল্যস্ফীতি|মুদ্রাস্ফীতি|দ্রব্যমূল্য", "target_type": "MACRO_INDICATOR"},
    {"canonical": "remittance", "aliases": "রেমিট্যান্স|প্রবাসী আয়|প্রবাসী আয়|রেমিটেন্স", "target_type": "MACRO_INDICATOR"},
    {"canonical": "foreign exchange reserves", "aliases": "বৈদেশিক মুদ্রার রিজার্ভ|রিজার্ভ|বৈদেশিক মুদ্রার মজুত", "target_type": "MACRO_INDICATOR"},
    {"canonical": "exchange rate", "aliases": "বিনিময় হার|বিনিময় হার|ডলার রেট|টাকার দর", "target_type": "MACRO_INDICATOR"},
    {"canonical": "policy rate", "aliases": "নীতি সুদহার|পলিসি রেট|রেপো রেট", "target_type": "MACRO_INDICATOR"},
    {"canonical": "imports", "aliases": "আমদানি|আমদানি ব্যয়|আমদানি ব্যয়", "target_type": "MACRO_INDICATOR"},
    {"canonical": "exports", "aliases": "রপ্তানি|রফতানি|রপ্তানি আয়|রপ্তানি আয়", "target_type": "MACRO_INDICATOR"},
    {"canonical": "trade balance", "aliases": "বাণিজ্য ভারসাম্য|বাণিজ্য ঘাটতি|ট্রেড ব্যালেন্স", "target_type": "MACRO_INDICATOR"},
    {"canonical": "letter of credit", "aliases": "এলসি|এল সি|ঋণপত্র", "target_type": "MACRO_INDICATOR"},
    {"canonical": "employment", "aliases": "কর্মসংস্থান|বেকারত্ব|চাকরির বাজার", "target_type": "MACRO_INDICATOR"},
    {"canonical": "purchasing power", "aliases": "ক্রয়ক্ষমতা|ক্রয়ক্ষমতা|জীবনযাত্রার ব্যয়|জীবনযাত্রার ব্যয়", "target_type": "MACRO_INDICATOR"},
    {"canonical": "RMG sector", "aliases": "আরএমজি|তৈরি পোশাক খাত|গার্মেন্টস খাত", "target_type": "SECTOR"},
    {"canonical": "banking sector", "aliases": "ব্যাংকিং খাত|ব্যাংক খাত", "target_type": "SECTOR"},
    {"canonical": "energy sector", "aliases": "জ্বালানি খাত|বিদ্যুৎ ও জ্বালানি খাত", "target_type": "SECTOR"},
    {"canonical": "agriculture sector", "aliases": "কৃষি খাত|কৃষিখাত", "target_type": "SECTOR"},
])
if ENTITY_CSV.exists():
    existing_entities = pd.read_csv(ENTITY_CSV).fillna("")
    entities = pd.concat([existing_entities, starter_entities], ignore_index=True)
    entities = entities.drop_duplicates("canonical", keep="first")
else:
    entities = starter_entities
entities.to_csv(ENTITY_CSV, index=False)
print("Bangladesh financial/economic entity lexicon rows =", len(entities))
alias_rows = []
for row in entities.to_dict("records"):
    for alias in [row["canonical"], *str(row["aliases"]).split("|")]:
        alias = alias.strip()
        if alias:
            alias_rows.append((alias, row["canonical"], row["target_type"]))
alias_rows.sort(key=lambda x: len(x[0]), reverse=True)

def restore_financial_entities(text: str) -> tuple[str, list[dict]]:
    restored, hits = normalize_transcript_draft(text), []
    for alias, canonical, target_type in alias_rows:
        flags = re.IGNORECASE if alias.isascii() else 0
        pattern = re.compile(rf"(?<!\w){re.escape(alias)}(?!\w)", flags)
        if pattern.search(restored):
            restored = pattern.sub(canonical, restored)
            hits.append({"alias": alias, "canonical": canonical, "target_type": target_type})
    return restored, hits

for path in sorted(DIRS["fused"].glob("*.json")):
    obj = json.loads(path.read_text(encoding="utf-8"))
    for u in obj.get("utterances", []):
        u["text_normalized"], u["entity_restorations"] = restore_financial_entities(u["text_verbatim"])
    atomic_json(path, obj)
print("Entity restoration applied to fused drafts.")


Bangladesh financial/economic entity lexicon rows = 28
Entity restoration applied to fused drafts.


## 6. Optional GER ablation

GER may improve spelling but may also change facts, numbers, named entities,
or sentiment. It is cached with edit-rate diagnostics and is evaluated as a
separate condition. Do not copy GER text into the gold field without human
verification.


In [20]:
if CFG.run_ger_ablation:
    repo_run(
        "notebook/06_stage_3_ger.py", "--prefer", "gemini",
        "--batch-size", "32", "--rpm", "14",
        "--confidence-threshold", str(CFG.confidence_review_threshold),
    )
else:
    print("GER ablation disabled.")


GER ablation disabled.


## 7. Three-layer annotation scaffold

The generated drafts are not gold. Human annotators correct verbatim text,
speaker attribution/role, explicit target span/type, and binary polarity.
Multi-target utterances use one label row per target. Implicit targets are
queued for review rather than guessed.


In [21]:
TARGETS = {"MARKET", "SECTOR", "COMPANY", "REGULATOR", "MACRO_INDICATOR"}
POLARITIES = {"positive", "negative"}
ROLES = {"host", "panellist", "remote_guest", "audience", "non_speech"}
ANNOTATION_STATUSES = {"labelled", "non_evaluative", "out_of_scope", "review_required"}
TARGET_GUIDANCE = {
    "MARKET": "stock/capital/commodity market or market-wide index",
    "SECTOR": "industry such as RMG, banking, agriculture, energy or imports/exports of a named sector",
    "COMPANY": "a named company, bank or business organization",
    "REGULATOR": "Bangladesh government economic policy, ministry, central bank, tax/statistics authority or international policy institution",
    "MACRO_INDICATOR": "GDP, inflation, remittance, reserves, exchange rate, policy rate, employment, purchasing power, aggregate imports/exports or trade balance",
}
atomic_json(DIRS["raw"] / "target_annotation_policy.json", {
    "schema_version": "1.0",
    "programme": CFG.programme_name,
    "target_types": TARGET_GUIDANCE,
    "annotation_statuses": sorted(ANNOTATION_STATUSES),
    "polarity_classes": sorted(POLARITIES),
    "exclusion_policy": {
        "non_evaluative": "financially relevant but no evaluative polarity",
        "out_of_scope": "not financial/economic discourse",
        "review_required": "ambiguous status, target, span, or polarity; human decision required",
    },
})

def make_annotation_draft(fused_path: Path) -> Path:
    fused = json.loads(fused_path.read_text(encoding="utf-8"))
    rows = []
    for u in fused.get("utterances", []):
        rows.append({
            "utterance_id": u["utterance_id"], "episode_id": u["episode_id"],
            "speaker_id": u["speaker_id"], "role_tag": "",
            "start_sec": u["start_sec"], "end_sec": u["end_sec"],
            "text_verbatim": u["text_verbatim"], "text_normalized": u["text_normalized"],
            "labels": [], "annotation_status": "review_required",
            "transcript_source": "asr_fused_draft", "speaker_source": "diarization_draft",
            "human_verified": False, "annotator_id": "", "notes": "",
        })
    draft = {
        "schema_version": "2.0", "episode_id": fused["episode_id"],
        "is_gold_tier": False, "overlap_policy": "first-speaker-wins",
        "utterances": rows,
    }
    out = DIRS["drafts"] / fused_path.name
    atomic_json(out, draft)
    return out

drafts = [make_annotation_draft(p) for p in sorted(DIRS["fused"].glob("*.json"))]
print("annotation drafts:", [p.name for p in drafts])
print("Copy a corrected file to data/annotated only after every row passes validation.")


annotation drafts: ['ep001.json', 'ep002.json', 'ep003.json']
Copy a corrected file to data/annotated only after every row passes validation.


In [22]:
ASCII_OR_BANGLA_DIGIT = re.compile(r"[0-9০-৯]")

def validate_annotated_episode(obj: dict) -> pd.DataFrame:
    issues = []
    for u in obj.get("utterances", []):
        uid = u.get("utterance_id", "<missing>")
        text = u.get("text_verbatim") or ""
        def add(code, detail): issues.append({"utterance_id": uid, "code": code, "detail": detail})
        if not text.strip(): add("EMPTY_TRANSCRIPT", "verbatim text required")
        if unicodedata.normalize("NFC", text) != text: add("NOT_NFC", "normalize Unicode to NFC")
        if ZERO_WIDTH_BIDI.search(text): add("BIDI_OR_ZERO_WIDTH", "remove hidden control characters")
        if ASCII_OR_BANGLA_DIGIT.search(text): add("DIGIT_NOT_WORD", "write numeral as Bangla words")
        if not isinstance(u.get("speaker_id"), int) or u.get("speaker_id", -1) < 0:
            add("BAD_SPEAKER", "episode-local non-negative integer required")
        if u.get("role_tag") not in ROLES: add("BAD_ROLE", str(u.get("role_tag")))
        if u.get("role_review_required"):
            add("ROLE_REVIEW_UNRESOLVED", "confirm or correct the heuristic speaker role")
        if float(u.get("end_sec", 0)) <= float(u.get("start_sec", 0)):
            add("BAD_TIME", "end must exceed start")
        labels = u.get("labels") or []
        status = u.get("annotation_status", "labelled" if labels else "review_required")
        if status not in ANNOTATION_STATUSES: add("BAD_ANNOTATION_STATUS", str(status))
        if status == "labelled" and not labels:
            add("MISSING_LABEL", "labelled utterance requires explicit target + binary polarity")
        if status in {"non_evaluative", "out_of_scope"} and labels:
            add("LABEL_ON_EXCLUDED_UTTERANCE", "excluded utterances must not carry sentiment labels")
        if status == "review_required":
            add("UNRESOLVED_REVIEW", "resolve as labelled, non_evaluative, or out_of_scope")
        seen_span_labels = {}
        for label in labels:
            if label.get("target_type") not in TARGETS: add("BAD_TARGET", str(label.get("target_type")))
            if label.get("polarity") not in POLARITIES: add("BAD_POLARITY", str(label.get("polarity")))
            if not label.get("target_text"): add("MISSING_TARGET_SPAN", "explicit target span required")
            elif str(label["target_text"]).casefold() not in (
                    (u.get("text_normalized") or "") + "\n" + text).casefold():
                add("TARGET_SPAN_NOT_IN_TEXT", str(label["target_text"]))
            span_key = str(label.get("target_text", "")).strip().casefold()
            current = (label.get("target_type"), label.get("polarity"))
            previous = seen_span_labels.setdefault(span_key, current)
            if previous[0] != current[0]:
                add("CONFLICTING_TARGET_TYPE", "one explicit span must have one target type")
            elif previous[1] != current[1]:
                add("CONTRADICTORY_POLARITY", "one explicit target cannot be both positive and negative in one utterance")
        if not u.get("human_verified"): add("NOT_HUMAN_VERIFIED", "draft cannot be gold")
    return pd.DataFrame(issues, columns=["utterance_id", "code", "detail"])

annotation_audit = {}
for path in sorted(DIRS["annotated"].glob("*.json")):
    obj = json.loads(path.read_text(encoding="utf-8"))
    issues = validate_annotated_episode(obj)
    annotation_audit[path.stem] = issues.to_dict("records")
    print(path.stem, "PASS" if issues.empty else f"{len(issues)} issue(s)")
    if not issues.empty:
        display(issues.head(30))


### Inter-annotator agreement

Export each annotator's JSON independently. κ is computed on aligned
utterance/target rows before adjudication. A target and polarity κ of at least
0.80 is the acceptance goal; disagreement rationales remain an artifact.


In [23]:
from collections import Counter

def cohen_kappa_score(labels_a, labels_b) -> float:
    # Cohen's kappa for two aligned categorical label sequences. Kept
    # dependency-free so agreement remains available while Colab's optional
    # compiled sklearn stack is being repaired.
    a, b = list(labels_a), list(labels_b)
    if len(a) != len(b):
        raise ValueError("Label sequences must have equal length")
    if not a:
        raise ValueError("Cohen's kappa requires at least one aligned label")
    n = len(a)
    observed = sum(x == y for x, y in zip(a, b)) / n
    count_a, count_b = Counter(a), Counter(b)
    expected = sum(count_a[label] * count_b[label] for label in count_a.keys() | count_b.keys()) / (n * n)
    if expected == 1.0:
        return 1.0 if observed == 1.0 else 0.0
    return float((observed - expected) / (1.0 - expected))

def annotation_rows(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    obj = json.loads(path.read_text(encoding="utf-8"))
    rows, statuses = [], []
    for u in obj.get("utterances", []):
        statuses.append({"utterance_id": u["utterance_id"], "annotation_status": u.get("annotation_status")})
        for label in u.get("labels") or []:
            # Anchor the comparison on the annotator-selected explicit span.
            # Outer alignment below turns a missing/different span into __NONE__.
            span = " ".join(str(label.get("target_text", "")).casefold().split())
            rows.append({"key": f"{u['utterance_id']}::{span}", **label})
    label_columns = ["key", "target_text", "target_type", "polarity", "confidence"]
    return (pd.DataFrame(rows, columns=label_columns).drop_duplicates("key"),
            pd.DataFrame(statuses).drop_duplicates("utterance_id"))

def agreement_report(annotator_a: Path, annotator_b: Path) -> dict:
    a, status_a = annotation_rows(annotator_a)
    b, status_b = annotation_rows(annotator_b)
    aligned = a.merge(b, on="key", suffixes=("_a", "_b"), how="outer")
    if aligned.empty:
        raise ValueError("No annotation label rows to compare")
    for column in ("target_type_a", "target_type_b", "polarity_a", "polarity_b"):
        aligned[column] = aligned[column].fillna("__NONE__")
    status = status_a.merge(status_b, on="utterance_id", suffixes=("_a", "_b"), how="outer").fillna("__MISSING__")
    return {
        "n_aligned": len(aligned),
        "n_utterances": len(status),
        "kappa_status": cohen_kappa_score(status.annotation_status_a, status.annotation_status_b),
        "kappa_target": cohen_kappa_score(aligned.target_type_a, aligned.target_type_b),
        "kappa_polarity": cohen_kappa_score(aligned.polarity_a, aligned.polarity_b),
        "kappa_joint": cohen_kappa_score(
            aligned.target_type_a + "::" + aligned.polarity_a,
            aligned.target_type_b + "::" + aligned.polarity_b,
        ),
    }

# Example after dual annotation:
# agreement = agreement_report(Path('annotations/annotator_a.json'), Path('annotations/annotator_b.json'))
# atomic_json(DIRS['aggregated'] / 'annotation_agreement.json', agreement)


## 8. Financial corpus and optional weak labelling

The Hugging Face corpus is unlabelled. It becomes training data only after an
LLM assigns an explicit target span/type and binary polarity, confidence and
provenance are stored, and a stratified 10% sample is human checked. Provide
about 20 genuine seed examples at `data/raw/weak_label_seeds.jsonl`; the
notebook will not fabricate them.


In [24]:
def load_financial_articles(root: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(root.glob("Bangla_fin_news_articles/*.csv")):
        try:
            frame = pd.read_csv(path)
        except UnicodeDecodeError:
            frame = pd.read_csv(path, encoding="utf-8-sig")
        row = frame.iloc[0].to_dict() if len(frame) else {}
        lowered = {str(k).strip().lower(): v for k, v in row.items()}
        title = next((lowered[k] for k in lowered if "title" in k), "")
        body = next((lowered[k] for k in lowered if any(x in k for x in ("news", "body", "article"))), "")
        rows.append({"article_id": path.stem, "title": str(title), "text": str(body), "source_file": str(path)})
    return pd.DataFrame(rows)

financial_articles = load_financial_articles(FINANCIAL_RAW) if FINANCIAL_RAW.exists() else pd.DataFrame()
print("loaded articles =", len(financial_articles))
display(financial_articles.head(3))


loaded articles = 0


""


In [25]:
# SEED_PATH = DIRS["raw"] / "weak_label_seeds.jsonl"

# def read_jsonl(path: Path) -> list[dict]:
#     if not path.exists(): return []
#     return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

# def entity_reference_prompt() -> str:
#     return "\n".join(
#         f"- {row.canonical} => {row.target_type}; aliases: {row.aliases}"
#         for row in entities.itertuples(index=False)
#     )

# def weak_label_prompt(text: str, seeds: list[dict]) -> str:
#     examples = "\n".join(json.dumps(x, ensure_ascii=False) for x in seeds[:8])
#     definitions = "\n".join(f"- {k}: {v}" for k, v in TARGET_GUIDANCE.items())
#     known_entities = entity_reference_prompt()
#     return f'''You are a Bangla-fluent financial discourse annotator.
# Return JSON only. Find explicit financial targets; do not infer implicit targets.
# Allowed target_type: MARKET, SECTOR, COMPANY, REGULATOR, MACRO_INDICATOR.
# Allowed polarity: positive, negative. Neutral is forbidden.
# Target definitions:
# {definitions}
# Known Bangladesh-domain entity/indicator references (guidance, not automatic labels):
# {known_entities}
# One sentence may return multiple labels. Use annotation_status=non_evaluative for
# questions, greetings, advertisements or purely factual statements with no evaluation.
# Use annotation_status=out_of_scope for non-economic daily-life/political discussion.
# Do not force either category into positive/negative. If the status or target is
# ambiguous, use review_required. Preserve company/ticker spelling.

# Schema:
# {{"labels":[{{"target_text":"...","target_type":"COMPANY","polarity":"positive","confidence":0.91}}],
#  "annotation_status":"labelled","review_required":false}}

# Human examples:
# {examples}

# Text:
# {text}
# '''

# def validate_weak_result(obj: dict, current_text: str | None = None) -> dict:
#     labels = obj.get("labels") or []
#     valid_by_span = {}
#     conflicting_spans = set()
#     for x in labels:
#         span_ok = not current_text or str(x.get("target_text", "")) in current_text
#         if (x.get("target_type") in TARGETS and x.get("polarity") in POLARITIES
#                 and x.get("target_text") and span_ok):
#             x["confidence"] = float(x.get("confidence", 0.0))
#             key = str(x["target_text"]).strip().casefold()
#             prior = valid_by_span.get(key)
#             if prior and (prior["target_type"], prior["polarity"]) != (x["target_type"], x["polarity"]):
#                 conflicting_spans.add(key)
#             if prior is None or x["confidence"] > prior["confidence"]:
#                 valid_by_span[key] = x
#     valid = list(valid_by_span.values())
#     declared_status = obj.get("annotation_status")
#     status_label_conflict = bool(valid) and declared_status in {"non_evaluative", "out_of_scope"}
#     status = declared_status
#     if valid:
#         status = "labelled"
#     elif status not in {"non_evaluative", "out_of_scope"}:
#         status = "review_required"
#     obj["labels"] = valid
#     obj["annotation_status"] = status
#     obj["review_required"] = (bool(obj.get("review_required")) or status == "review_required"
#                               or status_label_conflict or bool(conflicting_spans) or any(
#         x["confidence"] < CFG.confidence_review_threshold for x in valid
#     ))
#     obj["validation_flags"] = (
#         (["status_label_conflict"] if status_label_conflict else []) +
#         (["conflicting_label_resolved_by_confidence"] if conflicting_spans else [])
#     )
#     return obj

# def weak_label_articles(frame: pd.DataFrame, limit: int | None = None) -> None:
#     if not os.environ.get("GOOGLE_API_KEY"):
#         raise RuntimeError("GOOGLE_API_KEY is required")
#     seeds = read_jsonl(SEED_PATH)
#     if len(seeds) < 8:
#         raise RuntimeError(f"Need at least 8 genuine seed examples at {SEED_PATH}")
#     from google import genai
#     from google.genai import types
#     client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
#     work = frame.head(limit) if limit else frame
#     for row in work.to_dict("records"):
#         out = DIRS["weak"] / f"{row['article_id']}.json"
#         if out.exists():
#             continue
#         text = normalize_transcript_draft((row.get("title", "") + "\n" + row.get("text", ""))[:12000])
#         response = client.models.generate_content(
#             model=CFG.weak_label_model,
#             contents=weak_label_prompt(text, seeds),
#             config=types.GenerateContentConfig(response_mime_type="application/json"),
#         )
#         result = validate_weak_result(json.loads(response.text), text)
#         result.update({
#             "article_id": row["article_id"], "text": text,
#             "label_source": "llm_weak", "labeller_model": CFG.weak_label_model,
#             "prompt_seed_sha256": sha256(SEED_PATH),
#         })
#         atomic_json(out, result)
#         time.sleep(4.5)  # conservative free-tier pacing

# if CFG.run_weak_labelling:
#     weak_label_articles(financial_articles)
# else:
#     print("Weak labelling disabled; no API calls made.")


Weak labelling disabled; no API calls made.


In [27]:
# # End-to-end automatic route: weak-label the fused talk-show utterances in
# # batches. These labels produce an exploratory discourse profile but never
# # masquerade as human gold or enter headline evaluation.
# def audio_batch_prompt(items: list[dict], seeds: list[dict]) -> str:
#     examples = "\n".join(json.dumps(x, ensure_ascii=False) for x in seeds[:8]) or "(no human seeds supplied)"
#     payload = json.dumps(items, ensure_ascii=False)
#     definitions = "\n".join(f"- {k}: {v}" for k, v in TARGET_GUIDANCE.items())
#     known_entities = entity_reference_prompt()
#     return f'''You are a Bangla-fluent financial discourse annotator.
# For every input item, label only current_text and return the same utterance_id.
# previous_text and next_text are context for pronouns and references only; target_text
# must be an exact span from current_text.
# Allowed target_type: MARKET, SECTOR, COMPANY, REGULATOR, MACRO_INDICATOR.
# Allowed polarity: positive, negative. Never emit neutral.
# Target definitions:
# {definitions}
# Known Bangladesh-domain entity/indicator references (guidance, not automatic labels):
# {known_entities}
# Government economic policy and ministries are REGULATOR. Aggregate remittance,
# reserves, inflation, exchange rate, employment, purchasing power, imports, exports
# and trade balance are MACRO_INDICATOR; a named industry's trade is SECTOR.
# Use annotation_status=non_evaluative for questions, greetings, advertisements and
# pure factual narration without evaluation. Use out_of_scope for non-economic
# daily-life or political discussion. Do not force excluded text into sentiment.
# Confidence must be between 0 and 1. Use review_required for ambiguity, unresolved
# implicit targets, or confidence below 0.85. Return JSON only.

# Output schema:
# {{"items":[{{"utterance_id":"...","labels":[{{"target_text":"...","target_type":"MARKET","polarity":"positive","confidence":0.9}}],"annotation_status":"labelled","review_required":false}}]}}

# Optional human examples:
# {examples}

# Input items:
# {payload}
# '''

# def weak_label_audio_episode(fused_path: Path, batch_size: int = 16) -> Path:
#     if not os.environ.get("GOOGLE_API_KEY"):
#         raise RuntimeError("Add GOOGLE_API_KEY to Colab Secrets for automatic audio weak labelling")
#     from google import genai
#     from google.genai import types
#     client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
#     seeds = read_jsonl(SEED_PATH)
#     fused = json.loads(fused_path.read_text(encoding="utf-8"))
#     out_path = DIRS["weak_audio"] / fused_path.name
#     cached = json.loads(out_path.read_text(encoding="utf-8")) if out_path.exists() else {"utterances": []}
#     if cached.get("schema_version") != "2.1":
#         if cached.get("utterances"):
#             print(fused["episode_id"], "re-labelling legacy weak output with context-aware schema 2.1")
#         cached = {"utterances": []}
#     done = {u["utterance_id"]: u for u in cached.get("utterances", [])}
#     for cached_u in done.values():
#         cached_u.setdefault("annotation_status", "labelled" if cached_u.get("labels") else "review_required")
#         cached_u.setdefault("role_review_required", cached_u.get("role_tag") == "unassigned")
#     episode_utterances = fused.get("utterances", [])
#     speaker_stats = {}
#     for utterance in episode_utterances:
#         sid = int(utterance["speaker_id"])
#         stat = speaker_stats.setdefault(sid, {"turns": 0, "airtime": 0.0, "first": float("inf")})
#         stat["turns"] += 1
#         stat["airtime"] += max(0.0, float(utterance["end_sec"]) - float(utterance["start_sec"]))
#         stat["first"] = min(stat["first"], float(utterance["start_sec"]))
#     host_candidate = (max(speaker_stats, key=lambda sid: (
#         speaker_stats[sid]["turns"] + (3 if speaker_stats[sid]["first"] <= 60 else 0),
#         speaker_stats[sid]["airtime"], -speaker_stats[sid]["first"],
#     )) if speaker_stats else None)
#     draft_roles = {sid: ("host" if sid == host_candidate else "panellist") for sid in speaker_stats}
#     for cached_u in done.values():
#         if cached_u.get("role_tag") == "unassigned":
#             cached_u["role_tag"] = draft_roles.get(int(cached_u["speaker_id"]), "panellist")
#             cached_u["role_source"] = "turn_pattern_heuristic"
#             cached_u["role_review_required"] = True
#     context_by_id = {}
#     for i, utterance in enumerate(episode_utterances):
#         context_by_id[utterance["utterance_id"]] = {
#             "episode_id": fused["episode_id"], "speaker_id": utterance["speaker_id"],
#             "previous_text": ((episode_utterances[i-1].get("text_normalized") or
#                                episode_utterances[i-1].get("text_verbatim", "")) if i else ""),
#             "current_text": utterance.get("text_normalized") or utterance.get("text_verbatim", ""),
#             "next_text": ((episode_utterances[i+1].get("text_normalized") or
#                            episode_utterances[i+1].get("text_verbatim", ""))
#                           if i + 1 < len(episode_utterances) else ""),
#         }
#     pending = [u for u in episode_utterances if u["utterance_id"] not in done]
#     for start in range(0, len(pending), batch_size):
#         batch = pending[start:start+batch_size]
#         inputs = [{"utterance_id": u["utterance_id"], **context_by_id[u["utterance_id"]]} for u in batch]
#         last_error = None
#         for attempt in range(4):
#             try:
#                 response = client.models.generate_content(
#                     model=CFG.weak_label_model,
#                     contents=audio_batch_prompt(inputs, seeds),
#                     config=types.GenerateContentConfig(response_mime_type="application/json"),
#                 )
#                 returned = json.loads(response.text).get("items", [])
#                 break
#             except Exception as exc:
#                 last_error = exc
#                 if attempt == 3: raise
#                 time.sleep(2 ** attempt * 5)
#         by_id = {x.get("utterance_id"): validate_weak_result(
#             x, context_by_id.get(x.get("utterance_id"), {}).get("current_text", "")
#         ) for x in returned}
#         for u in batch:
#             label_obj = by_id.get(u["utterance_id"], {
#                 "labels": [], "annotation_status": "review_required", "review_required": True,
#             })
#             done[u["utterance_id"]] = {
#                 **{k: u[k] for k in ("utterance_id", "episode_id", "speaker_id", "start_sec", "end_sec",
#                                       "text_verbatim", "text_normalized")},
#                 "role_tag": draft_roles.get(int(u["speaker_id"]), "panellist"),
#                 "role_source": "turn_pattern_heuristic", "role_review_required": True,
#                 "labels": label_obj["labels"], "annotation_status": label_obj["annotation_status"],
#                 "review_required": label_obj["review_required"],
#                 "validation_flags": label_obj.get("validation_flags", []),
#                 "human_verified": False, "label_source": "llm_weak",
#                 "labeller_model": CFG.weak_label_model,
#             }
#         partial = {
#             "schema_version": "2.1", "episode_id": fused["episode_id"],
#             "is_gold_tier": False, "label_tier": "llm_weak_exploratory",
#             "utterances": sorted(done.values(), key=lambda x: x["start_sec"]),
#         }
#         atomic_json(out_path, partial)
#         print(fused["episode_id"], min(start+batch_size, len(pending)), "/", len(pending))
#         time.sleep(1.0)
#     final_payload = {
#         "schema_version": "2.1", "episode_id": fused["episode_id"],
#         "is_gold_tier": False, "label_tier": "llm_weak_exploratory",
#         "context_window": "previous/current/next utterance",
#         "utterances": sorted(done.values(), key=lambda x: x["start_sec"]),
#     }
#     atomic_json(out_path, final_payload)
#     return out_path

# if CFG.run_audio_weak_labelling:
#     weak_audio_paths = [weak_label_audio_episode(p) for p in sorted(DIRS["fused"].glob("*.json"))]
#     print("weak-labelled audio:", [p.name for p in weak_audio_paths])
# else:
#     print("Automatic audio weak labelling disabled.")


In [28]:
# Create the stratified human-review queue (10%, never fewer than one per stratum).
weak_rows = []
for path in sorted(DIRS["weak"].glob("*.json")):
    obj = json.loads(path.read_text(encoding="utf-8"))
    for i, label in enumerate(obj.get("labels") or []):
        weak_rows.append({"row_id": f"{path.stem}::{i}", "example_id": path.stem,
                          "article_id": path.stem,
                          "text": obj.get("text", ""), **label,
                          "review_required": obj.get("review_required", False)})
weak_df = pd.DataFrame(weak_rows)
weak_train_df = pd.DataFrame()
weak_label_audit = {"status": "NOT_APPLICABLE", "n_review": 0, "accuracy": None}
if not weak_df.empty:
    forced = weak_df[weak_df.review_required].copy()
    forced["review_kind"] = "forced"
    sampled_parts = [
        sub.sample(max(1, int(round(len(sub) * 0.10))), random_state=CFG.seed)
        for _, sub in weak_df.groupby(["target_type", "polarity"], sort=True)
    ]
    sampled = pd.concat(sampled_parts, ignore_index=True) if sampled_parts else weak_df.iloc[0:0].copy()
    sampled["review_kind"] = "stratified_audit"
    review_queue = pd.concat([forced, sampled], ignore_index=True).drop_duplicates("row_id", keep="first")
    review_path = DIRS["weak"] / "human_review_queue.csv"
    previous_review = pd.read_csv(review_path) if review_path.exists() else pd.DataFrame()
    previous_by_id = (previous_review.set_index("row_id").to_dict("index")
                      if not previous_review.empty and "row_id" in previous_review else {})
    for column, default in (("human_verified", False), ("approved", False), ("review_notes", "")):
        review_queue[column] = [previous_by_id.get(row_id, {}).get(column, default)
                                for row_id in review_queue.row_id]
    review_queue.to_csv(review_path, index=False)

    def csv_bool(value) -> bool:
        return str(value).strip().lower() in {"1", "true", "yes"}

    audit_rows = review_queue[review_queue.review_kind == "stratified_audit"].copy()
    audit_verified = audit_rows.human_verified.map(csv_bool) if not audit_rows.empty else pd.Series(dtype=bool)
    audit_complete = bool(len(audit_rows)) and bool(audit_verified.all())
    audit_accuracy = (float(audit_rows.loc[audit_verified, "approved"].map(csv_bool).mean())
                      if audit_complete else None)
    audit_pass = audit_complete and audit_accuracy >= 0.85
    weak_label_audit = {
        "status": "PASS" if audit_pass else "PENDING",
        "n_review": len(audit_rows), "accuracy": audit_accuracy,
        "required_accuracy": 0.85, "review_file": str(review_path),
    }
    atomic_json(DIRS["aggregated"] / "weak_label_audit.json", weak_label_audit)
    if audit_pass:
        rejected_ids = set(review_queue.loc[
            review_queue.human_verified.map(csv_bool) & ~review_queue.approved.map(csv_bool), "row_id"
        ])
        weak_train_df = weak_df[
            (~weak_df.review_required) &
            (weak_df.confidence.astype(float) >= CFG.confidence_review_threshold) &
            (~weak_df.row_id.isin(rejected_ids))
        ].copy()
        weak_train_df["group_id"] = "weak::" + weak_train_df.article_id.astype(str)
        weak_train_df["example_id"] = "weak::" + weak_train_df.article_id.astype(str)
        weak_train_df["label_source"] = "llm_weak_audited"
    else:
        print("Weak corpus is quarantined until the stratified audit is verified at >=85% accuracy:", review_path)
    display(review_queue.groupby(["target_type", "polarity"]).size().rename("n"))
else:
    print("No weak labels yet.")


No weak labels yet.


## 9. Target-aware multi-label BanglaBERT training

The model predicts ten independent `target_type::polarity` probabilities, so
one utterance can retain several simultaneous targets. Splits are grouped by
episode; random utterance leakage is forbidden even within the single RTV
programme. Weak labels keep their
provenance and are never placed in the held-out human test set.

The research digest proposes domain-adaptive pretraining. BanglaBERT is an
ELECTRA discriminator, so a naive `AutoModelForMaskedLM` pass would be the
wrong objective. This notebook therefore uses the released discriminator for
the reproducible baseline and leaves full generator+discriminator DAPT as a
separate, explicitly named ablation rather than silently faking MLM.


In [29]:
def collect_human_labels() -> pd.DataFrame:
    rows = []
    for path in sorted(DIRS["annotated"].glob("*.json")):
        obj = json.loads(path.read_text(encoding="utf-8"))
        if not obj.get("is_gold_tier"):
            continue
        if not validate_annotated_episode(obj).empty:
            continue
        for u in obj.get("utterances", []):
            for label_index, label in enumerate(u.get("labels") or []):
                rows.append({
                    "row_id": f"{u['utterance_id']}::{label_index}",
                    "example_id": u["utterance_id"],
                    "episode_id": obj["episode_id"], "group_id": obj["episode_id"],
                    "speaker_id": u["speaker_id"], "role_tag": u["role_tag"],
                    "text": u.get("text_normalized") or u["text_verbatim"],
                    "target_text": label["target_text"], "target_type": label["target_type"],
                    "polarity": label["polarity"], "label_source": "human",
                })
    return pd.DataFrame(rows)

human_df = collect_human_labels()
print("verified human label rows =", len(human_df))
if not human_df.empty:
    display(human_df.groupby(["target_type", "polarity"]).size().rename("n"))


verified human label rows = 0


In [30]:
def accuracy_score(y_true, y_pred) -> float:
    truth, pred = list(y_true), list(y_pred)
    if len(truth) != len(pred):
        raise ValueError("y_true and y_pred must have equal length")
    return float(sum(a == b for a, b in zip(truth, pred)) / len(truth)) if truth else 0.0

def _label_scores(y_true, y_pred, labels) -> dict:
    truth, pred = list(y_true), list(y_pred)
    scores = {}
    for label in labels:
        tp = sum(a == label and b == label for a, b in zip(truth, pred))
        fp = sum(a != label and b == label for a, b in zip(truth, pred))
        fn = sum(a == label and b != label for a, b in zip(truth, pred))
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        scores[label] = {"precision": precision, "recall": recall, "f1-score": f1,
                         "support": sum(a == label for a in truth)}
    return scores

def f1_score(y_true, y_pred, labels=None, average="macro", zero_division=0) -> float:
    truth, pred = list(y_true), list(y_pred)
    labels = list(labels) if labels is not None else sorted(set(truth) | set(pred))
    scores = _label_scores(truth, pred, labels)
    if average != "macro":
        raise ValueError("Only average='macro' is supported here")
    return float(sum(v["f1-score"] for v in scores.values()) / len(labels)) if labels else 0.0

def classification_report(y_true, y_pred, labels=None, target_names=None,
                          output_dict=True, zero_division=0):
    truth, pred = list(y_true), list(y_pred)
    labels = list(labels) if labels is not None else sorted(set(truth) | set(pred))
    names = list(target_names) if target_names is not None else [str(x) for x in labels]
    raw = _label_scores(truth, pred, labels)
    report = {name: raw[label] for label, name in zip(labels, names)}
    report["accuracy"] = accuracy_score(truth, pred)
    report["macro avg"] = {
        "precision": sum(v["precision"] for v in raw.values()) / max(len(raw), 1),
        "recall": sum(v["recall"] for v in raw.values()) / max(len(raw), 1),
        "f1-score": sum(v["f1-score"] for v in raw.values()) / max(len(raw), 1),
        "support": len(truth),
    }
    return report if output_dict else json.dumps(report, ensure_ascii=False, indent=2)

def _group_holdout_indices(frame: pd.DataFrame, test_size: float, seed: int):
    groups = list(dict.fromkeys(frame.group_id.tolist()))
    if len(groups) < 2:
        raise ValueError("A grouped split requires at least two distinct groups")
    rng = random.Random(seed)
    rng.shuffle(groups)
    n_test = max(1, min(len(groups) - 1, round(len(groups) * test_size)))
    test_groups = set(groups[:n_test])
    train_idx = [i for i, group in enumerate(frame.group_id.tolist()) if group not in test_groups]
    test_idx = [i for i, group in enumerate(frame.group_id.tolist()) if group in test_groups]
    return train_idx, test_idx

JOINT_LABELS = [f"{t}::{p}" for t in sorted(TARGETS) for p in sorted(POLARITIES)]
LABEL2ID = {x: i for i, x in enumerate(JOINT_LABELS)}
ID2LABEL = {i: x for x, i in LABEL2ID.items()}

def split_human(frame: pd.DataFrame):
    if frame.empty or frame.group_id.nunique() < 3:
        raise ValueError("Need at least three distinct episodes for train/public/private splits")
    train_idx, hold_idx = _group_holdout_indices(frame, test_size=0.40, seed=CFG.seed)
    train, hold = frame.iloc[train_idx].copy(), frame.iloc[hold_idx].copy()
    public_idx, private_idx = _group_holdout_indices(hold, test_size=0.50, seed=CFG.seed + 1)
    return train, hold.iloc[public_idx].copy(), hold.iloc[private_idx].copy()

def to_multilabel_examples(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame(columns=["example_id", "text", "group_id", "labels", "row_ids"])
    work = frame.copy()
    if "example_id" not in work:
        work["example_id"] = work.get("article_id", work.row_id).astype(str)
    rows = []
    for example_id, sub in work.groupby("example_id", sort=False):
        vector = [0.0] * len(JOINT_LABELS)
        for _, label_row in sub.iterrows():
            vector[LABEL2ID[f"{label_row.target_type}::{label_row.polarity}"]] = 1.0
        first = sub.iloc[0]
        rows.append({
            "example_id": str(example_id), "text": first.text, "group_id": first.group_id,
            "labels": vector, "row_ids": sub.row_id.astype(str).tolist(),
        })
    return pd.DataFrame(rows)

def decode_multilabel_probabilities(probabilities, threshold: float) -> np.ndarray:
    probabilities = np.asarray(probabilities)
    pred = np.zeros_like(probabilities, dtype=int)
    # Each target may occur once with one polarity. Multiple different targets
    # remain allowed, but contradictory polarities for one target do not.
    for row_index, vector in enumerate(probabilities):
        for target in sorted(TARGETS):
            candidate_ids = [LABEL2ID[f"{target}::{p}"] for p in sorted(POLARITIES)]
            best = max(candidate_ids, key=lambda i: vector[i])
            if vector[best] >= threshold:
                pred[row_index, best] = 1
        if not pred[row_index].any():
            pred[row_index, int(vector.argmax())] = 1
    return pred

def multilabel_metrics_arrays(label_ids, logits, threshold: float) -> dict:
    if isinstance(logits, tuple):
        logits = logits[0]
    probabilities = 1.0 / (1.0 + np.exp(-np.asarray(logits)))
    gold = np.asarray(label_ids).astype(int)
    pred = decode_multilabel_probabilities(probabilities, threshold)
    per_label_f1 = []
    for j in range(gold.shape[1]):
        tp = int(((gold[:, j] == 1) & (pred[:, j] == 1)).sum())
        fp = int(((gold[:, j] == 0) & (pred[:, j] == 1)).sum())
        fn = int(((gold[:, j] == 1) & (pred[:, j] == 0)).sum())
        per_label_f1.append(2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0)
    return {
        "macro_f1": float(np.mean(per_label_f1)),
        "subset_accuracy": float(np.mean(np.all(gold == pred, axis=1))),
        "per_label_f1": {label: float(per_label_f1[i]) for i, label in enumerate(JOINT_LABELS)},
    }

def train_joint_classifier(human_frame: pd.DataFrame, weak_frame: pd.DataFrame, output_dir: Path):
    from datasets import Dataset
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              DataCollatorWithPadding, Trainer, TrainingArguments)
    human_train, public_df, private_df = split_human(human_frame)
    # Weak labels are training-only. The held-out test set is always human and
    # episode/programme-disjoint from the human portion of training.
    train_df = pd.concat([human_train, weak_frame], ignore_index=True, sort=False)
    train_examples = to_multilabel_examples(train_df)
    public_examples = to_multilabel_examples(public_df)
    private_examples = to_multilabel_examples(private_df)
    tokenizer = AutoTokenizer.from_pretrained(CFG.sentiment_model, revision=CFG.sentiment_revision)
    model = AutoModelForSequenceClassification.from_pretrained(
        CFG.sentiment_model, revision=CFG.sentiment_revision,
        num_labels=len(JOINT_LABELS), label2id=LABEL2ID, id2label=ID2LABEL,
        problem_type="multi_label_classification",
    )
    def prepare(df):
        return Dataset.from_pandas(df[["text", "labels"]], preserve_index=False)
    def tokenize(batch): return tokenizer(batch["text"], truncation=True, max_length=256)
    train_ds = prepare(train_examples).map(tokenize, batched=True)
    public_ds = prepare(public_examples).map(tokenize, batched=True)
    private_ds = prepare(private_examples).map(tokenize, batched=True)
    def metrics(pred):
        result = multilabel_metrics_arrays(pred.label_ids, pred.predictions, CFG.multilabel_threshold)
        return {"subset_accuracy": result["subset_accuracy"], "macro_f1": result["macro_f1"]}
    common = dict(
        output_dir=str(output_dir), learning_rate=2e-5, num_train_epochs=5,
        per_device_train_batch_size=16, per_device_eval_batch_size=32,
        weight_decay=0.01, warmup_ratio=0.1, save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="macro_f1",
        greater_is_better=True, seed=CFG.seed, report_to=[], fp16=(DEVICE == "cuda"),
    )
    try:
        args = TrainingArguments(eval_strategy="epoch", **common)
    except TypeError:
        args = TrainingArguments(evaluation_strategy="epoch", **common)
    trainer = Trainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=public_ds,
        processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=metrics,
    )
    trainer.train()
    public_pred = trainer.predict(public_ds)
    threshold_candidates = [round(x, 2) for x in np.arange(0.30, 0.71, 0.05)]
    public_candidates = {
        threshold: multilabel_metrics_arrays(public_pred.label_ids, public_pred.predictions, threshold)
        for threshold in threshold_candidates
    }
    calibrated_threshold = max(
        threshold_candidates,
        key=lambda threshold: (public_candidates[threshold]["macro_f1"],
                               public_candidates[threshold]["subset_accuracy"],
                               -abs(threshold - CFG.multilabel_threshold)),
    )
    public_scores = public_candidates[calibrated_threshold]
    # The private split is evaluated exactly once after all checkpoint and
    # threshold choices have been made on training/public data.
    private_pred = trainer.predict(private_ds)
    private_scores = multilabel_metrics_arrays(private_pred.label_ids, private_pred.predictions,
                                                calibrated_threshold)
    private_f1, public_f1 = private_scores["macro_f1"], public_scores["macro_f1"]
    report = {
        "metrics": private_pred.metrics,
        "public_macro_f1": public_f1, "private_macro_f1": private_f1,
        "public_private_gap": abs(public_f1 - private_f1),
        "multilabel_threshold": calibrated_threshold,
        "threshold_selection_split": "public",
        "checkpoint_selection_split": "public",
        "private_split_role": "final_evaluation_only",
        "private_subset_accuracy": private_scores["subset_accuracy"],
        "public_subset_accuracy": public_scores["subset_accuracy"],
        "per_label_f1": private_scores["per_label_f1"],
        "public_example_ids": public_examples.example_id.tolist(),
        "private_example_ids": private_examples.example_id.tolist(),
        "n_human_train_rows": len(human_train), "n_weak_train_rows": len(weak_frame),
        "n_train_examples": len(train_examples),
        "public_groups": sorted(public_df.group_id.unique().tolist()),
        "private_groups": sorted(private_df.group_id.unique().tolist()),
        "train_groups": sorted(train_df.group_id.unique().tolist()),
    }
    trainer.save_model(output_dir); tokenizer.save_pretrained(output_dir)
    atomic_json(output_dir / "multilabel_threshold.json", {"threshold": calibrated_threshold})
    atomic_json(DIRS["aggregated"] / "joint_labeller_eval.json", report)
    return report

# A fine-tuned checkpoint is large; retain it in the Colab runtime by default
# and explicitly export it only when the Drive quota permits.
JOINT_MODEL_DIR = RUNTIME_CACHE_ROOT / "banglabert_financial_joint"
if CFG.run_sentiment_training:
    if len(to_multilabel_examples(human_df)) < 20:
        raise RuntimeError("Need at least 20 fully verified human utterances; synthetic labels are not accepted")
    joint_report = train_joint_classifier(human_df, weak_train_df, JOINT_MODEL_DIR)
    print(joint_report["metrics"])
else:
    print("Sentiment training disabled.")


Sentiment training disabled.


In [31]:
# Reusable multi-label inference. If no trained checkpoint exists, scientific
# metrics are SKIPPED, not filled with a rule-based stand-in.
JOINT_PIPELINE = None
JOINT_TOKENIZER = None
if (JOINT_MODEL_DIR / "config.json").exists():
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    JOINT_TOKENIZER = AutoTokenizer.from_pretrained(str(JOINT_MODEL_DIR))
    JOINT_PIPELINE = AutoModelForSequenceClassification.from_pretrained(str(JOINT_MODEL_DIR)).to(DEVICE)
    JOINT_PIPELINE.eval()
threshold_path = JOINT_MODEL_DIR / "multilabel_threshold.json"
ACTIVE_MULTILABEL_THRESHOLD = (
    float(json.loads(threshold_path.read_text(encoding="utf-8"))["threshold"])
    if threshold_path.exists() else CFG.multilabel_threshold
)

def predict_joint(texts: list[str]) -> list[dict] | None:
    if JOINT_PIPELINE is None:
        return None
    rows = []
    for start in range(0, len(texts), 32):
        batch = texts[start:start+32]
        encoded = JOINT_TOKENIZER(batch, truncation=True, max_length=256,
                                  padding=True, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            probabilities = torch.sigmoid(JOINT_PIPELINE(**encoded).logits).cpu().numpy()
        decoded = decode_multilabel_probabilities(probabilities, ACTIVE_MULTILABEL_THRESHOLD)
        for vector, mask in zip(probabilities, decoded):
            selected = np.flatnonzero(mask).tolist()
            labels = []
            for i in selected:
                target_type, polarity = ID2LABEL[i].split("::")
                labels.append({"target_type": target_type, "polarity": polarity,
                               "confidence": float(vector[i])})
            labels.sort(key=lambda x: x["confidence"], reverse=True)
            rows.append({**labels[0], "labels": labels, "multilabel": True})
    return rows


## 10. Stage metrics

No proxy DER or confidence-derived WER is reported. Metrics are computed only
when their reference artifacts exist. Missing references produce `SKIPPED`.


In [32]:
from jiwer import wer as jiwer_wer, cer as jiwer_cer

def proper_noun_recovery(reference: str, hypothesis: str, entity_frame: pd.DataFrame) -> dict:
    ref, hyp = reference.casefold(), hypothesis.casefold()
    rows = []
    for e in entity_frame.to_dict("records"):
        canonical = str(e["canonical"])
        if canonical.casefold() in ref:
            rows.append({"canonical": canonical, "target_type": e["target_type"],
                         "recovered": canonical.casefold() in hyp})
    if not rows:
        return {"support": 0, "recall": None, "by_target": {}}
    frame = pd.DataFrame(rows)
    return {
        "support": len(frame), "recall": float(frame.recovered.mean()),
        "by_target": frame.groupby("target_type").recovered.mean().to_dict(),
    }

def rttm_to_annotation(path: Path):
    from pyannote.core import Annotation, Segment
    ann = Annotation(uri=path.stem)
    for line in path.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) >= 8 and parts[0] == "SPEAKER":
            start, dur, speaker = float(parts[3]), float(parts[4]), parts[7]
            ann[Segment(start, start + dur)] = speaker
    return ann

def diarization_metrics(reference_rttm: Path, hypothesis_rttm: Path) -> dict:
    from pyannote.metrics.diarization import DiarizationErrorRate
    ref, hyp = rttm_to_annotation(reference_rttm), rttm_to_annotation(hypothesis_rttm)
    return {
        "der_collar_0.25": float(DiarizationErrorRate(collar=0.25, skip_overlap=False)(ref, hyp)),
        "der_collar_0.00": float(DiarizationErrorRate(collar=0.00, skip_overlap=False)(ref, hyp)),
    }

def text_metrics(reference: str, hypothesis: str) -> dict:
    ref, hyp = normalize_transcript_draft(reference), normalize_transcript_draft(hypothesis)
    return {"wer": float(jiwer_wer(ref, hyp)), "cer": float(jiwer_cer(ref, hyp)),
            "proper_noun": proper_noun_recovery(ref, hyp, entities)}

def attribution_accuracy(gold_rows: list[dict], fused_rows: list[dict]) -> dict:
    matched = []
    for g in gold_rows:
        candidates = [(overlap_seconds(g["start_sec"], g["end_sec"], p["start_sec"], p["end_sec"]), p)
                      for p in fused_rows]
        ov, pred = max(candidates, key=lambda x: x[0], default=(0, None))
        if pred is not None and ov > 0:
            matched.append(int(g["speaker_id"]) == int(pred["speaker_id"]))
    return {"n": len(matched), "accuracy": float(np.mean(matched)) if matched else None}


In [33]:
# Compute available per-episode reference metrics. Gold RTTM convention:
# data/annotated/<episode_id>.rttm
stage_metrics = {"episodes": {}, "skipped": []}
for gold_path in sorted(DIRS["annotated"].glob("*.json")):
    ep = gold_path.stem
    gold = json.loads(gold_path.read_text(encoding="utf-8"))
    gold_rows = gold.get("utterances", [])
    fused_path = DIRS["fused"] / f"{ep}.json"
    metrics = {}
    if fused_path.exists():
        fused = json.loads(fused_path.read_text(encoding="utf-8"))
        ref_text = " ".join(u.get("text_normalized") or u.get("text_verbatim", "") for u in gold_rows)
        hyp_text = " ".join(u.get("text_normalized") or u.get("text_verbatim", "") for u in fused.get("utterances", []))
        metrics.update(text_metrics(ref_text, hyp_text))
        metrics["attribution"] = attribution_accuracy(gold_rows, fused.get("utterances", []))
    else:
        stage_metrics["skipped"].append({"episode": ep, "metric": "ASR/fusion", "reason": "missing fused file"})
    gold_rttm, hyp_rttm = DIRS["annotated"] / f"{ep}.rttm", DIRS["diar"] / f"{ep}.rttm"
    if gold_rttm.exists() and hyp_rttm.exists():
        metrics.update(diarization_metrics(gold_rttm, hyp_rttm))
    else:
        stage_metrics["skipped"].append({"episode": ep, "metric": "DER", "reason": "missing gold RTTM"})
    stage_metrics["episodes"][ep] = metrics
atomic_json(DIRS["aggregated"] / "stage_metrics.json", stage_metrics)
stage_metrics


{'episodes': {}, 'skipped': []}

## 11. Factorial error propagation

For each gold utterance, the closest overlapping predicted utterance creates
four controlled inputs:

1. `oracle`: gold text + gold speaker
2. `diarization_corruption`: gold text + predicted speaker
3. `asr_corruption`: ASR text + gold speaker
4. `full_pipeline`: ASR text + predicted speaker

This isolates transcription and attribution errors and reports breakdowns by
speaker role and target. GER, when enabled, is an additional ablation rather
than a replacement for this factorial design.


In [34]:
def best_time_match(row: dict, candidates: list[dict]) -> dict | None:
    scored = [(overlap_seconds(row["start_sec"], row["end_sec"], x["start_sec"], x["end_sec"]), x)
              for x in candidates]
    score, item = max(scored, key=lambda x: x[0], default=(0.0, None))
    return item if item is not None and score > 0 else None

def score_condition(rows: list[dict]) -> dict:
    valid = [r for r in rows if r.get("pred")]
    if not valid:
        return {"status": "SKIPPED", "reason": "trained classifier unavailable or no aligned rows"}
    gold_joint = [{x["target_type"] + "::" + x["polarity"] for x in r["gold_labels"]} for r in valid]
    pred_joint = [{x["target_type"] + "::" + x["polarity"]
                   for x in (r["pred"].get("labels") or [r["pred"]])} for r in valid]
    def macro_set_f1(gold_sets, pred_sets, universe):
        scores = []
        for label in universe:
            tp = sum(label in g and label in p for g, p in zip(gold_sets, pred_sets))
            fp = sum(label not in g and label in p for g, p in zip(gold_sets, pred_sets))
            fn = sum(label in g and label not in p for g, p in zip(gold_sets, pred_sets))
            scores.append(2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0)
        return float(sum(scores) / len(scores)) if scores else 0.0
    gold_targets = [{x.split("::")[0] for x in labels} for labels in gold_joint]
    pred_targets = [{x.split("::")[0] for x in labels} for labels in pred_joint]
    gold_polarities = [{x.split("::")[1] for x in labels} for labels in gold_joint]
    pred_polarities = [{x.split("::")[1] for x in labels} for labels in pred_joint]
    return {
        "status": "OK", "n": len(valid),
        "target_macro_f1": macro_set_f1(gold_targets, pred_targets, sorted(TARGETS)),
        "polarity_macro_f1": macro_set_f1(gold_polarities, pred_polarities, sorted(POLARITIES)),
        "joint_macro_f1": macro_set_f1(gold_joint, pred_joint, JOINT_LABELS),
        "joint_exact_match": float(sum(g == p for g, p in zip(gold_joint, pred_joint)) / len(valid)),
        "speaker_attribution_accuracy": float(np.mean([r["gold_speaker"] == r["used_speaker"] for r in valid])),
    }

def factorial_episode(ep: str) -> dict:
    gold = json.loads((DIRS["annotated"] / f"{ep}.json").read_text(encoding="utf-8"))
    fused = json.loads((DIRS["fused"] / f"{ep}.json").read_text(encoding="utf-8"))
    aligned = []
    for g in gold.get("utterances", []):
        pred_u = best_time_match(g, fused.get("utterances", []))
        if pred_u is None: continue
        labels = g.get("labels") or []
        if not labels: continue
        aligned.append({
            "utterance_id": g["utterance_id"], "role": g.get("role_tag"),
            "gold_text": g.get("text_normalized") or g["text_verbatim"],
            "asr_text": pred_u.get("text_normalized") or pred_u["text_verbatim"],
            "gold_speaker": int(g["speaker_id"]), "pred_speaker": int(pred_u["speaker_id"]),
            "gold_labels": labels,
        })
    conditions = {
        "oracle": ("gold_text", "gold_speaker"),
        "diarization_corruption": ("gold_text", "pred_speaker"),
        "asr_corruption": ("asr_text", "gold_speaker"),
        "full_pipeline": ("asr_text", "pred_speaker"),
    }
    report = {"episode_id": ep, "n_aligned": len(aligned), "conditions": {}, "per_target": {}, "per_role": {}}
    for name, (text_key, speaker_key) in conditions.items():
        preds = predict_joint([r[text_key] for r in aligned]) if aligned else None
        rows = [{**r, "used_speaker": r[speaker_key], "pred": p}
                for r, p in zip(aligned, preds or [None]*len(aligned))]
        report["conditions"][name] = score_condition(rows)
        for target in sorted(TARGETS):
            report["per_target"].setdefault(target, {})[name] = score_condition([
                r for r in rows if any(x["target_type"] == target for x in r["gold_labels"])
            ])
        for role in sorted(ROLES):
            report["per_role"].setdefault(role, {})[name] = score_condition([r for r in rows if r["role"] == role])
    return report

factorial = {"episodes": []}
for path in sorted(DIRS["annotated"].glob("*.json")):
    if (DIRS["fused"] / path.name).exists():
        factorial["episodes"].append(factorial_episode(path.stem))
atomic_json(DIRS["aggregated"] / "factorial_error_propagation.json", factorial)
factorial


{'episodes': []}

## 12. Noise-Toleration Point (NTP)

Gold text is synthetically degraded at seven noise levels using seeded
character edits. The achieved WER—not the requested probability—is plotted
against joint F1. The ASR target is the downstream elbow, not an arbitrary
quest for ever-lower headline WER.


In [35]:
BANGLA_CHARS = list("অআইঈউঊএঐওঔকখগঘঙচছজঝঞটঠডঢণতথদধনপফবভমযরলশষসহড়ঢ়য়")

def corrupt_token(token: str, rng: random.Random) -> str:
    if not token: return token
    op = rng.choice(["drop", "swap", "repeat", "replace"])
    i = rng.randrange(len(token))
    if op == "drop": return token[:i] + token[i+1:]
    if op == "repeat": return token[:i] + token[i] + token[i:]
    if op == "replace": return token[:i] + rng.choice(BANGLA_CHARS) + token[i+1:]
    if len(token) > 1:
        j = min(i+1, len(token)-1)
        chars = list(token); chars[i], chars[j] = chars[j], chars[i]
        return "".join(chars)
    return token

def inject_noise(text: str, target_pct: int, seed: int) -> str:
    rng = random.Random(seed)
    words = text.split()
    for i in range(len(words)):
        if rng.random() < target_pct / 100:
            words[i] = corrupt_token(words[i], rng)
    return " ".join(words)

def ntp_experiment(frame: pd.DataFrame) -> pd.DataFrame:
    if JOINT_PIPELINE is None or frame.empty:
        return pd.DataFrame()
    examples = to_multilabel_examples(frame)
    rows = []
    gold_sets = [{JOINT_LABELS[i] for i, value in enumerate(vector) if value}
                 for vector in examples.labels]
    for level in [5, 15, 25, 35, 45, 55, 65]:
        noisy = [inject_noise(t, level, CFG.seed+i) for i, t in enumerate(examples.text.tolist())]
        preds = predict_joint(noisy)
        pred_sets = [{x["target_type"] + "::" + x["polarity"]
                      for x in (p.get("labels") or [p])} for p in preds]
        f1s = []
        for label in JOINT_LABELS:
            tp = sum(label in g and label in p for g, p in zip(gold_sets, pred_sets))
            fp = sum(label not in g and label in p for g, p in zip(gold_sets, pred_sets))
            fn = sum(label in g and label not in p for g, p in zip(gold_sets, pred_sets))
            f1s.append(2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0)
        rows.append({
            "requested_noise_pct": level,
            "achieved_wer": float(np.mean([jiwer_wer(a, b) for a,b in zip(examples.text, noisy)])),
            "macro_f1": float(np.mean(f1s)),
        })
    return pd.DataFrame(rows)

ntp = ntp_experiment(human_df)
if not ntp.empty:
    ntp.to_csv(DIRS["aggregated"] / "ntp_curve.csv", index=False)
    ax = ntp.plot(x="achieved_wer", y="macro_f1", marker="o", title="Noise-Toleration Curve")
    ax.figure.savefig(DIRS["figures"] / "ntp_curve.png", dpi=160, bbox_inches="tight")
display(ntp)


""


## 13. Speaker and episode discourse profiles

Aggregation never merges identities across episodes. Counts and airtime are
reported separately, target remains separate from speaker, and confidence or
review status remains visible.


In [36]:
def aggregate_profiles() -> dict:
    episodes = []
    selected: dict[str, tuple[dict, str]] = {}
    # Prefer valid human gold. Fall back to LLM weak labels so a fresh
    # standalone run still produces an explicitly exploratory profile.
    for path in sorted(DIRS["weak_audio"].glob("*.json")):
        obj = json.loads(path.read_text(encoding="utf-8"))
        selected[obj["episode_id"]] = (obj, "llm_weak_exploratory")
    for path in sorted(DIRS["annotated"].glob("*.json")):
        obj = json.loads(path.read_text(encoding="utf-8"))
        if obj.get("is_gold_tier") and validate_annotated_episode(obj).empty:
            selected[obj["episode_id"]] = (obj, "human_gold")
    for _, (obj, tier) in sorted(selected.items()):
        label_rows = []
        for u in obj.get("utterances", []):
            duration = max(0.0, float(u["end_sec"]) - float(u["start_sec"]))
            for label in u.get("labels") or []:
                label_rows.append({
                    "speaker_id": int(u["speaker_id"]), "role_tag": u["role_tag"],
                    "duration_sec": duration, "target_type": label["target_type"],
                    "target_text": label["target_text"], "polarity": label["polarity"],
                })
        frame = pd.DataFrame(label_rows)
        speakers = []
        if not frame.empty:
            for (speaker, role), sub in frame.groupby(["speaker_id", "role_tag"]):
                total = len(sub); pos = int((sub.polarity == "positive").sum())
                speakers.append({
                    "speaker_id": int(speaker), "role_tag": role,
                    "n_labelled_targets": total, "positive": pos, "negative": total-pos,
                    "positive_rate": pos/total, "labelled_airtime_sec": float(sub.duration_sec.sum()),
                    "target_counts": sub.target_type.value_counts().to_dict(),
                    "target_polarity": (sub.groupby(["target_type", "polarity"]).size()
                                         .rename("n").reset_index().to_dict("records")),
                })
        status_counts = {}
        for utterance in obj.get("utterances", []):
            status = utterance.get("annotation_status", "labelled" if utterance.get("labels") else "review_required")
            status_counts[status] = status_counts.get(status, 0) + 1
        episodes.append({"episode_id": obj["episode_id"], "label_tier": tier, "speakers": speakers,
                         "n_utterances": len(obj.get("utterances", [])),
                         "annotation_status_counts": status_counts})
    return {"schema_version": "2.0", "speaker_ids_episode_local": True, "episodes": episodes}

profiles = aggregate_profiles()
atomic_json(DIRS["aggregated"] / "discourse_profiles_v2.json", profiles)
profiles


{'schema_version': '2.0', 'speaker_ids_episode_local': True, 'episodes': []}

In [37]:
# Compact profile visualization.
import matplotlib.pyplot as plt
import seaborn as sns

plot_rows = []
for ep in profiles["episodes"]:
    for spk in ep["speakers"]:
        plot_rows.append({"episode": ep["episode_id"], "speaker": spk["speaker_id"],
                          "role": spk["role_tag"], "positive_rate": spk["positive_rate"],
                          "n": spk["n_labelled_targets"]})
plot_df = pd.DataFrame(plot_rows)
if not plot_df.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=plot_df, x="speaker", y="positive_rate", hue="episode", ax=ax)
    ax.set_ylim(0, 1); ax.set_title("Per-speaker target-aware positive rate")
    fig.savefig(DIRS["figures"] / "speaker_sentiment_profile.png", dpi=160, bbox_inches="tight")
    plt.show()


## Per-audio runtime estimate (Colab T4)

The estimate separates one-time environment/model preparation from the warm
automatic pipeline. Human gold correction is not GPU processing and is shown
separately. Actual time varies with audio duration, overlap, music, Drive I/O,
API throttling and whether a T4 or faster GPU is assigned.


In [38]:
def estimate_runtime_minutes(duration_min: float) -> pd.DataFrame:
    # Calibrated to the project's recorded 34-minute Colab plan and widened
    # for 30–60 minute audio, API retries, selective Demucs, and Drive I/O.
    rows = [
        ("One-time install + model download/conversion", 15, 30),
        ("Inventory + 16 kHz/LUFS preprocessing", max(1, .025*duration_min), max(3, .06*duration_min)),
        ("Pyannote diarization", max(2, .07*duration_min), max(8, .20*duration_min)),
        ("Bengali Whisper-medium ASR", .45*duration_min, .85*duration_min),
        ("Word-speaker fusion + entity restoration", .2, 1.0),
        ("Gemini target/polarity weak labelling", max(3, .10*duration_min), max(15, .35*duration_min)),
        ("Profiles + automatic metrics + bundle", 1, 4),
    ]
    frame = pd.DataFrame(rows, columns=["stage", "low_min", "high_min"])
    warm = frame.iloc[1:]
    frame.loc[len(frame)] = ["TOTAL — first run", frame.low_min.sum(), frame.high_min.sum()]
    frame.loc[len(frame)] = ["TOTAL — later warm run", warm.low_min.sum(), warm.high_min.sum()]
    return frame.round(1)

duration_values = registry.duration_sec.astype(float) / 60 if not registry.empty else pd.Series([45.0])
runtime_estimate = estimate_runtime_minutes(float(duration_values.iloc[0]))
display(runtime_estimate)
print("Human gold correction/target annotation, when required for publishable evaluation: add roughly 3–8 hours per 30–60 minute episode.")


,stage,low_min,high_min
0,One-time install + model download/conversion,15.0,30.0
1,Inventory + 16 kHz/LUFS preprocessing,1.1,3.0
2,Pyannote diarization,3.2,9.1
3,Bengali Whisper-medium ASR,20.5,38.7
4,Word-speaker fusion + entity restoration,0.2,1.0
5,Gemini target/polarity weak labelling,4.5,15.9
6,Profiles + automatic metrics + bundle,1.0,4.0
7,TOTAL — first run,45.5,101.7
8,TOTAL — later warm run,30.5,71.7


Human gold correction/target annotation, when required for publishable evaluation: add roughly 3–8 hours per 30–60 minute episode.


## 14. Submission gates and artifact provenance

`PASS` requires evidence. Missing measurements are `SKIPPED`, never zero or a
proxy. A single `FAIL` or `SKIPPED` means the pipeline is not yet submittable.


In [39]:
def flatten_gold() -> list[dict]:
    rows = []
    for path in sorted(DIRS["annotated"].glob("*.json")):
        obj = json.loads(path.read_text(encoding="utf-8"))
        if obj.get("is_gold_tier"):
            rows.extend(obj.get("utterances", []))
    return rows

gold_rows = flatten_gold()
gates = []
def gate(name, passed=None, value=None, threshold=None, reason=""):
    status = "SKIPPED" if passed is None else ("PASS" if passed else "FAIL")
    gates.append({"gate": name, "status": status, "value": value, "threshold": threshold, "reason": reason})

eligible = registry[~registry.duration_flag.eq("rejected")] if not registry.empty and "duration_flag" in registry else registry
gate("episode_count", len(eligible) >= 8, len(eligible), ">=8")
programme_n = int(eligible.programme.nunique()) if "programme" in eligible else 0
if CFG.single_program_case_study:
    programme_values = sorted(eligible.programme.dropna().astype(str).unique().tolist()) if "programme" in eligible else []
    gate("single_programme_scope", programme_values == [CFG.programme_name],
         programme_values, f"exactly [{CFG.programme_name}]")
else:
    gate("programme_count", programme_n >= 4, programme_n, ">=4")
if "has_remote_guest" in eligible:
    remote_n = eligible.has_remote_guest.astype(str).str.lower().isin(["1","true","yes"]).sum()
else: remote_n = 0
gate("remote_guest_episodes", remote_n >= 2, int(remote_n), ">=2")
gate("no_synthetic_gold", all(not u.get("synthetic", False) for u in gold_rows),
     sum(bool(u.get("synthetic")) for u in gold_rows), "0")
gate("layer1_no_empty_transcript", bool(gold_rows) and all((u.get("text_verbatim") or "").strip() for u in gold_rows),
     sum(not (u.get("text_verbatim") or "").strip() for u in gold_rows), "0 missing")
gate("layer2_no_missing_speaker", bool(gold_rows) and all(isinstance(u.get("speaker_id"), int) and u["speaker_id"] >= 0 for u in gold_rows),
     sum(not isinstance(u.get("speaker_id"), int) or u.get("speaker_id", -1) < 0 for u in gold_rows), "0 missing")
unresolved_roles = [u for u in gold_rows if u.get("role_tag") not in ROLES or u.get("role_review_required")]
gate("layer2_roles_resolved", bool(gold_rows) and not unresolved_roles,
     len(unresolved_roles), "0 unresolved")
resolved_statuses = {"labelled", "non_evaluative", "out_of_scope"}
unresolved_layer3 = [u for u in gold_rows if u.get("annotation_status") not in resolved_statuses]
bad_labelled = [u for u in gold_rows if u.get("annotation_status") == "labelled" and not u.get("labels")]
gate("layer3_resolved_or_excluded", bool(gold_rows) and not unresolved_layer3 and not bad_labelled,
     len(unresolved_layer3) + len(bad_labelled), "0 unresolved")
all_labels = [x for u in gold_rows for x in (u.get("labels") or [])]
covered_targets = sorted({x.get("target_type") for x in all_labels if x.get("target_type") in TARGETS})
gate("economic_topic_coverage", len(covered_targets) >= 4 if gold_rows else None,
     covered_targets, ">=4 of 5 target types represented in gold")
gate("no_neutral_class", all(x.get("polarity") in POLARITIES for x in all_labels),
     sorted({x.get("polarity") for x in all_labels}), "positive|negative only")
if not weak_df.empty:
    gate("weak_label_stratified_audit", weak_label_audit.get("status") == "PASS",
         weak_label_audit.get("accuracy"), ">=0.85 on completed 10% stratified review")
unassigned = 0
for path in DIRS["fused"].glob("*.json"):
    unassigned += json.loads(path.read_text(encoding="utf-8")).get("stats", {}).get("unassigned_words", 0)
gate("fusion_no_unassigned_words", unassigned == 0, unassigned, "0")

measured_wers = [m.get("wer") for m in stage_metrics.get("episodes", {}).values() if m.get("wer") is not None]
mean_wer = float(np.mean(measured_wers)) if measured_wers else None
gate("wer_measured", bool(measured_wers) if gold_rows else None,
     mean_wer, "requires gold transcript",
     "add verified gold transcript" if not measured_wers else "")
gate("wer_target", mean_wer <= 0.30 if mean_wer is not None else None, mean_wer, "<=0.30")
measured_ders = [m.get("der_collar_0.25") for m in stage_metrics.get("episodes", {}).values()
                 if m.get("der_collar_0.25") is not None]
gate("der_measured_0.25_collar", bool(measured_ders) if gold_rows else None,
     float(np.mean(measured_ders)) if measured_ders else None, "requires gold RTTM")
mean_der = float(np.mean(measured_ders)) if measured_ders else None
gate("der_target", mean_der <= 0.20 if mean_der is not None else None, mean_der, "<=0.20")

agreement_path = DIRS["aggregated"] / "annotation_agreement.json"
agreement = json.loads(agreement_path.read_text(encoding="utf-8")) if agreement_path.exists() else {}
kappa = agreement.get("kappa_joint")
gate("annotation_kappa", kappa >= 0.80 if kappa is not None else None, kappa, ">=0.80",
     "run agreement_report on two independent annotation files" if kappa is None else "")

joint_eval_path = DIRS["aggregated"] / "joint_labeller_eval.json"
joint_eval = json.loads(joint_eval_path.read_text(encoding="utf-8")) if joint_eval_path.exists() else {}
f1_gap = joint_eval.get("public_private_gap")
gate("public_private_f1_gap", f1_gap <= 0.03 if f1_gap is not None else None,
     f1_gap, "<=0.03", "requires train/public/private group-disjoint evaluation" if f1_gap is None else "")
private_f1 = joint_eval.get("private_macro_f1")
gate("sentiment_private_macro_f1", private_f1 >= 0.70 if private_f1 is not None else None,
     private_f1, ">=0.70")
gate("ntp_curve", not ntp.empty if JOINT_PIPELINE is not None else None, len(ntp), "7 noise levels")
if not ntp.empty and mean_wer is not None:
    ntp_limit = float(ntp.loc[ntp.macro_f1 >= ntp.macro_f1.max() - 0.03, "achieved_wer"].max())
    gate("wer_at_or_below_ntp", mean_wer <= ntp_limit, mean_wer, f"<={ntp_limit:.4f}")
else:
    gate("wer_at_or_below_ntp", None, mean_wer, "requires measured WER + NTP curve")

ne_recalls = [m.get("proper_noun", {}).get("recall") for m in stage_metrics.get("episodes", {}).values()
              if m.get("proper_noun", {}).get("recall") is not None]
gate("proper_noun_recovery_measured", bool(ne_recalls) if gold_rows else None,
     float(np.mean(ne_recalls)) if ne_recalls else None, "requires whitelist support in gold")

gate_df = pd.DataFrame(gates)
SUBMITTABLE = bool(len(gate_df)) and (gate_df.status == "PASS").all()
gate_report = {"submittable": SUBMITTABLE, "gates": gates,
               "summary": gate_df.status.value_counts().to_dict()}
atomic_json(DIRS["aggregated"] / "submission_gates.json", gate_report)
display(gate_df)
print("SUBMITTABLE =", SUBMITTABLE)


,gate,status,value,threshold,reason
0,episode_count,FAIL,3,>=8,
1,programme_count,FAIL,3,>=4,
2,remote_guest_episodes,FAIL,0,>=2,
3,no_synthetic_gold,PASS,0,0,
4,layer1_no_empty_transcript,FAIL,0,0 missing,
5,layer2_no_missing_speaker,FAIL,0,0 missing,
6,layer2_roles_resolved,FAIL,0,0 unresolved,
7,layer3_resolved_or_excluded,FAIL,0,0 unresolved,
8,economic_topic_coverage,SKIPPED,[],>=4 of 5 target types represented in gold,
9,no_neutral_class,PASS,[],positive|negative only,


SUBMITTABLE = False


In [40]:
# Content-hash every material output and save a reproducibility manifest.
material_roots = [DIRS[k] for k in ("raw", "processed", "diar", "asr", "fused", "ger",
                                            "annotated", "weak", "aggregated", "reports", "models")]
for root in material_roots:
    for path in sorted(p for p in root.rglob("*") if p.is_file()):
        rel = path.relative_to(PROJECT_ROOT).as_posix()
        RUN_MANIFEST["files"][rel] = {"sha256": sha256(path), "bytes": path.stat().st_size}
RUN_MANIFEST["puku_audit"] = {k:v for k,v in PUKU_AUDIT.items() if k != "indexed_files"}
RUN_MANIFEST["submittable"] = SUBMITTABLE
atomic_json(DIRS["artifacts"] / "run_manifest.json", RUN_MANIFEST)
print("hashed files =", len(RUN_MANIFEST["files"]))


hashed files = 32


## 15. Save a portable result bundle

The bundle contains all non-audio derived data, annotation files, metrics,
reports, figures, manifests, and (when trained) the BanglaBERT checkpoint.
In Colab the ZIP is assembled under `/content` and downloaded directly, so it
does not consume another full copy of your Drive quota. Individual stage
outputs remain persistent under `MyDrive/ML_Project`.


In [41]:
import zipfile

bundle_root = Path("/content") if IN_COLAB else DIRS["artifacts"]
bundle = bundle_root / f"bangla_financial_pipeline_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"
include_roots = [DIRS[k] for k in (
    "raw", "diar", "asr", "fused", "ger", "drafts", "annotated",
    "weak", "weak_audio", "aggregated", "reports",
)]
include_files = [DIRS["artifacts"] / "run_manifest.json", PROJECT_ROOT / "models" / "manifest.json"]
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root in include_roots:
        for path in root.rglob("*"):
            if path.is_file(): zf.write(path, path.relative_to(PROJECT_ROOT))
    for path in include_files:
        if path.exists(): zf.write(path, path.relative_to(PROJECT_ROOT))
    if CFG.include_trained_model_in_bundle and JOINT_MODEL_DIR.exists():
        for path in JOINT_MODEL_DIR.rglob("*"):
            if path.is_file():
                zf.write(path, Path("models") / "banglabert_financial_joint" / path.relative_to(JOINT_MODEL_DIR))
print(bundle, bundle.stat().st_size, "bytes", sha256(bundle))
if IN_COLAB and CFG.persist_final_bundle_to_drive:
    import shutil
    persistent_bundle = DIRS["artifacts"] / bundle.name
    shutil.copy2(bundle, persistent_bundle)
    print("Persistent ZIP copy:", persistent_bundle)
if IN_COLAB and CFG.download_final_bundle:
    from google.colab import files
    print("Starting browser download.")
    files.download(str(bundle))
elif not IN_COLAB:
    print("Bundle is ready for local download:", bundle)


/content/bangla_financial_pipeline_20260810_212153.zip 1194616 bytes fc066f16d0e11579dad87f904148ead2c7ef4a09b9060c5d1fbc4355467385fb
Starting browser download.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Interpretation checklist

- A fallback VAD output is a plumbing smoke test, not diarization evidence.
- A generic Whisper fallback is not the BengaliAI ASR experiment.
- Synthetic gold is not gold and cannot enter headline results.
- Report both DER collars and say which one is primary.
- Report proper-noun recovery beside WER.
- Report gold → ASR sentiment loss per target and per speaker role.
- Investigate remote-guest outliers instead of averaging them away.
- Stop ASR optimization at the measured downstream NTP, not at the lowest WER.
- Never compare speakers across episodes by their numeric IDs.
- If any submission gate is failed or skipped, label the run exploratory.


## Research provenance

This standalone file was compiled from `docs/PIPELINE_RESEARCH.md`, the
proposal DOCX, all papers under `docs/`, the GPT and Qwen research reports,
`docs/RESEARCH_VERIFICATION.md`, and the `.puku` index. Those source documents
were used at build time and are not required in the Colab runtime.

Core evidence map:

- Bengali-Loop: long-form Bangla protocol, normalization, first-speaker-wins,
  0.25 s DER collar.
- DL Sprint / Lipi-Ghor / WhisperAlign / ShobdoSetu / Bangla Diarizz:
  community diarization, post-processing, word timestamps, full ASR fine-tune,
  augmentation and chunking.
- Whisper: 30 s constraint and cross-chunk error propagation.
- ENDow and Jung & Choi: downstream NTP, entity sensitivity, factorial cascade
  degradation.
- Motamot, SentiGOLD, SentNoB, BABSA, BanglaASTE: few-shot weak labels,
  financial-domain difficulty, binary labels, annotation agreement and the
  future ABSA path.
